# High-Frequency Digital Price Measurement and Heterogeneous Inflation Dynamics in Bangladesh
### Evidence from a large-scale online retail price panel (Chaldal.com, 2024–2026)

**Modular, checkpointed research pipeline (Google Colab).**

| # | Module | Main output |
|---|---|---|
| 0 | Setup, configuration, Drive checkpoints, figure/table style | `config.json` |
| 1 | Data ingestion, reshaping, calendar validation, cross-check with the 2024 workbook | `01_*` |
| 2 | Data-quality audit: unit parsing, category taxonomy, anomaly flag → classify | `02_*` |
| 3 | Missingness audit and mechanism tests | `03_*` |
| 4 | Missing-price reconstruction library (8 methods incl. learned bridge) | `04_*` |
| 5 | Masked-gap validation (7/14/30/45/137-day system-wide gaps) | `05_*` |
| 6 | Analytical datasets A (observed) · B (preferred) · C (alternative) · D (conservative) | `06_*` |
| 7 | Digital Price Index: Jevons / Dutot / Carli / Laspeyres / Paasche / Fisher, category indices | `07_*` |
| 8 | Price dynamics: stickiness, hazard, volatility, synchronisation, seasonality, structural breaks | `08_*` |
| 9 | Heterogeneous behaviour: product regime clustering | `09_*` |
| 10 | Inflation dynamics: MoM/YoY, contributions, persistence, asymmetry, shock propagation | `10_*` |
| 11 | Official CPI/FPI linkage, leading-indicator and nowcasting tests | `11_*` |
| 12 | Forecasting: rolling-origin, statistical / ML / DL, aggregate + product level | `12_*` |
| 13 | Hierarchical reconciliation and uncertainty quantification | `13_*` |
| 14 | Robustness scenarios and ablation | `14_*` |
| 15 | Export of journal-ready tables and figures | `tables/`, `figures/` |

**How checkpointing works.** Every module is wrapped in `run_module(...)`. If all of its outputs already exist in
`MyDrive/DPI_Project/checkpoints/`, the module is skipped and later modules load the saved results. After a Colab
disconnect simply *Run all* again – work resumes from the last completed module. Set `FORCE_RERUN = {'m05'}` (or
`CFG['force_all']=True`) to recompute specific modules.

**Inputs** – put these files in `MyDrive/DPI_Project/data/` (the notebook will offer an upload dialog otherwise):
* `ChalDal_final.csv` – wide panel `Name | Amount | YYYY-MM-DD ...` (observed scrape days only)
* `Dataset-2024.xlsx` – optional, earlier workbook used for cross-validation
* `bbs_cpi_monthly.csv` – optional, official CPI/FPI (a template is written by Module 11 if it is missing)
* `hies_weights.csv` – optional, category expenditure weights (`category,weight`)

## Module 0 · Environment, configuration and helpers

In [ ]:
import os, sys, json, time, pickle, re, math, warnings, subprocess, hashlib, datetime as dt
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ruptures', 'lightgbm', 'xgboost', 'statsmodels'], check=False)
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/DPI_Project')
else:
    ROOT = Path(os.environ.get('DPI_ROOT', './DPI_Project')).resolve()

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter, MaxNLocator
from scipy import stats
warnings.filterwarnings('ignore')
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 30)

DIRS = {k: ROOT / k for k in ['data', 'checkpoints', 'figures', 'tables', 'models', 'logs']}
for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

In [ ]:
# ----------------------------- CONFIGURATION ---------------------------------
CFG = dict(
    raw_csv=str(DIRS['data'] / 'ChalDal_final.csv'),
    raw_xlsx=str(DIRS['data'] / 'Dataset-2024.xlsx'),
    cpi_csv=str(DIRS['data'] / 'bbs_cpi_monthly.csv'),
    hies_weights_csv=str(DIRS['data'] / 'hies_weights.csv'),
    seed=42,
    # stale snapshots: days on which ≤ this share of SKUs differ from the previous scraped day are treated as missing
    drop_stale_days=True, stale_threshold=0.0005,
    # anomaly screening (|log return| thresholds)
    anomaly_logret=np.log(1.5),          # flag moves larger than +50% / -33%
    reversal_window=3,                   # spike that reverts within k observed days
    # gap classes (calendar days)
    gap_short=7, gap_long=30,
    # masked-gap validation
    mask_lengths=[7, 14, 30, 45, 137],
    mask_windows_per_length=8,
    bridge_train_windows=60,             # masked windows used to train the learned bridge
    bridge_max_rows=1_500_000,
    mi_draws=5,                          # multiple-imputation draws of the stochastic sticky bridge
    selection_weights={'WAPE (%)': 1, 'Index RMSE (pts)': 2, 'Change-freq. error (pp)': 1, 'Wasserstein ×100': 1},
    # index construction
    base_date=None,                      # None -> first observed day
    upper_weights='sku',                 # 'equal' | 'sku' | 'hies' (needs hies_weights_csv)
    # dynamics
    pelt_penalty=None,                   # None -> BIC-type penalty
    # clustering
    k_range=list(range(3, 9)),
    # forecasting
    horizons=[1, 7, 14, 30, 60],
    n_origins=10, origin_step=14,
    n_sample_products=300,
    lstm_epochs=60,
    # Bangladesh calendar events (verify against the official BD calendar before publication)
    events={
        'Ramadan 2024': ('2024-03-12', '2024-04-10'), 'Eid al-Adha 2024': ('2024-06-17', '2024-06-19'),
        'Ramadan 2025': ('2025-03-02', '2025-03-30'), 'Eid al-Adha 2025': ('2025-06-07', '2025-06-09'),
        'Ramadan 2026': ('2026-02-19', '2026-03-20'),
    },
    force_all=False,
)
FORCE_RERUN = set()            # e.g. {'m05', 'm07'}
QUICK = os.environ.get('DPI_QUICK', '0') == '1'   # smaller settings for smoke tests
if QUICK:
    CFG.update(mask_windows_per_length=3, bridge_train_windows=20, n_origins=4, n_sample_products=80, lstm_epochs=15, mi_draws=2)
RNG = np.random.default_rng(CFG['seed'])
with open(ROOT / 'config.json', 'w') as f:
    json.dump({k: (v if not isinstance(v, (np.floating,)) else float(v)) for k, v in CFG.items()}, f, indent=2, default=str)
print('Project root:', ROOT)

In [ ]:
# ----------------------------- CHECKPOINT HELPERS ----------------------------
CK = DIRS['checkpoints']

def _ck_file(name):
    for ext in ('.parquet', '.pkl', '.json'):
        p = CK / f'{name}{ext}'
        if p.exists():
            return p
    return None

def ck_exists(name):
    return _ck_file(name) is not None

def _json_default(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (pd.Timestamp, dt.date)): return str(o)
    raise TypeError(type(o))

def save_ck(obj, name):
    """DataFrame -> parquet, dict of JSON-able -> json, anything else -> pickle."""
    if isinstance(obj, pd.DataFrame):
        df = obj.copy()
        df.columns = [str(c) for c in df.columns]
        if not isinstance(df.index, pd.RangeIndex):
            df = df.reset_index()
        p = CK / f'{name}.parquet'
        df.to_parquet(p)
        json.dump({'index_col': obj.index.name if not isinstance(obj.index, pd.RangeIndex) else None,
                   'index_names': [n for n in obj.index.names] if not isinstance(obj.index, pd.RangeIndex) else None},
                  open(CK / f'{name}.meta', 'w'), default=str)
    else:
        try:
            p = CK / f'{name}.json'
            with open(p, 'w') as f:
                json.dump(obj, f, indent=1, default=_json_default)
        except TypeError:
            if p.exists(): p.unlink()
            p = CK / f'{name}.pkl'
            pickle.dump(obj, open(p, 'wb'), protocol=4)
    return p

def load_ck(name):
    p = _ck_file(name)
    if p is None:
        raise FileNotFoundError(f'Checkpoint {name} not found – run the module that produces it.')
    if p.suffix == '.parquet':
        df = pd.read_parquet(p)
        mp = CK / f'{name}.meta'
        if mp.exists():
            m = json.load(open(mp))
            if m.get('index_names'):
                cols = [c if c is not None else 'index' for c in m['index_names']]
                df = df.set_index(cols)
                df.index.names = m['index_names']
        return df
    if p.suffix == '.json':
        return json.load(open(p))
    return pickle.load(open(p, 'rb'))

def run_module(key, fn, outputs):
    """Run `fn` unless all `outputs` checkpoints exist (resume-safe)."""
    force = CFG['force_all'] or key in FORCE_RERUN
    if not force and all(ck_exists(o) for o in outputs):
        print(f'[{key}] ✓ checkpoints found – skipped ({", ".join(outputs)})')
        return
    t0 = time.time()
    print(f'[{key}] running ...')
    fn()
    missing = [o for o in outputs if not ck_exists(o)]
    el = time.time() - t0
    with open(DIRS['logs'] / 'run_log.csv', 'a') as f:
        f.write(f'{dt.datetime.now().isoformat()},{key},{el:.1f},{"OK" if not missing else "MISSING:" + "|".join(missing)}\n')
    print(f'[{key}] done in {el/60:.1f} min' + (f' – WARNING missing {missing}' if missing else ''))

# wide matrices (products × dates) are stored with string date columns
def wide_to_ck(W, name):
    D = W.copy(); D.columns = [c.strftime('%Y-%m-%d') if hasattr(c, 'strftime') else str(c) for c in D.columns]
    D.index.name = 'product_id'
    save_ck(D.astype('float32'), name)

def wide_from_ck(name):
    D = load_ck(name)
    if 'product_id' in D.columns:
        D = D.set_index('product_id')
    D.columns = pd.to_datetime(D.columns)
    return D.astype('float64')

In [ ]:
# ----------------------------- JOURNAL STYLE ---------------------------------
# Okabe–Ito colour-blind-safe palette; serif text; vector PDF + 600-dpi PNG; Elsevier/MDPI column widths.
OKABE = ['#0072B2', '#D55E00', '#009E73', '#CC79A7', '#E69F00', '#56B4E9', '#F0E442', '#000000']
GREY, LIGHT = '#6b6b6b', '#d9d9d9'
W1, W2, W15 = 3.54, 7.25, 5.5     # single, double, 1.5 column (inches)

def set_style():
    plt.rcParams.update({
        'font.family': 'serif', 'font.serif': ['DejaVu Serif', 'Times New Roman', 'Times'],
        'mathtext.fontset': 'dejavuserif', 'font.size': 8, 'axes.titlesize': 8.5, 'axes.labelsize': 8,
        'xtick.labelsize': 7, 'ytick.labelsize': 7, 'legend.fontsize': 7, 'legend.frameon': False,
        'axes.linewidth': 0.6, 'axes.spines.top': False, 'axes.spines.right': False,
        'xtick.major.width': 0.6, 'ytick.major.width': 0.6, 'xtick.major.size': 3, 'ytick.major.size': 3,
        'xtick.direction': 'out', 'ytick.direction': 'out', 'lines.linewidth': 1.0,
        'axes.grid': True, 'grid.color': '#ececec', 'grid.linewidth': 0.5, 'axes.axisbelow': True,
        'axes.prop_cycle': matplotlib.cycler(color=OKABE), 'figure.dpi': 110,
        'savefig.dpi': 600, 'savefig.bbox': 'tight', 'savefig.pad_inches': 0.02,
        'pdf.fonttype': 42, 'ps.fonttype': 42, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    })
set_style()

FIG_REGISTRY = []
def savefig(fig, name, caption=''):
    for ext in ('pdf', 'png'):
        fig.savefig(DIRS['figures'] / f'{name}.{ext}')
    FIG_REGISTRY.append((name, caption))
    with open(DIRS['figures'] / '_captions.json', 'a') as f:
        f.write(json.dumps({'name': name, 'caption': caption}) + '\n')
    plt.show() if IN_COLAB else None
    plt.close(fig)

def panel_label(ax, s, x=-0.12, y=1.04):
    ax.text(x, y, f'({s})', transform=ax.transAxes, fontsize=9, fontweight='bold', va='bottom', ha='left')

def date_axis(ax, fmt='%b\n%Y', interval=3):
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=interval))
    ax.xaxis.set_major_formatter(mdates.DateFormatter(fmt))

def shade_gaps(ax, gaps=None, min_len=1, label=True):
    """Grey bands for scrape gaps (not observed days)."""
    if gaps is None:
        gaps = load_ck('03_gap_inventory') if ck_exists('03_gap_inventory') else None
    if gaps is None: return
    first = True
    for _, g in gaps.iterrows():
        if g['length'] >= min_len:
            ax.axvspan(pd.Timestamp(g['start']), pd.Timestamp(g['end']) + pd.Timedelta(days=1), color=LIGHT, alpha=0.7, lw=0,
                       label='Scrape gap (imputed)' if (first and label) else None, zorder=0)
            first = False

def shade_events(ax, ymax_frac=1.0):
    for i, (k, (a, b)) in enumerate(CFG['events'].items()):
        ax.axvspan(pd.Timestamp(a), pd.Timestamp(b), ymax=ymax_frac, color='#E69F00', alpha=0.12, lw=0, zorder=0)

# ----------------------------- TABLE EXPORT ----------------------------------
UNI2TEX = {'×': r'$\times$', 'Δ': r'$\Delta$', 'ρ': r'$\rho$', '≤': r'$\leq$', '≥': r'$\geq$', '–': '--', '—': '---',
           'σ': r'$\sigma$', 'χ²': r'$\chi^2$', '²': r'$^2$', '÷': r'$\div$', '−': r'$-$', 'β': r'$\beta$', 'τ': r'$\tau$',
           '±': r'$\pm$', '→': r'$\rightarrow$', '·': r'$\cdot$', 'é': r"\'e", 'ä': r'\"a', 'ö': r'\"o', 'ü': r'\"u',
           '≈': r'$\approx$', 'α': r'$\alpha$', '₁₀': r'$_{10}$', '‰': r'\textperthousand{}', '’': "'", '“': '``', '”': "''",
           '∝': r'$\propto$', '•': r'$\bullet$', 'ρ̂': r'$\hat\rho$', '≠': r'$\neq$', 'π': r'$\pi$', 'μ': r'$\mu$'}

def tex_escape(s):
    s = (str(s).replace('\\', r'\textbackslash{}').replace('&', r'\&').replace('%', r'\%').replace('_', r'\_')
         .replace('#', r'\#').replace('$', r'\$').replace('~', r'\textasciitilde{}').replace('^', r'\^{}')
         .replace('<', r'\textless{}').replace('>', r'\textgreater{}').replace('|', r'\textbar{}'))
    for u, l in UNI2TEX.items():
        s = s.replace(u, l)
    return s

def save_table(df, name, caption, label=None, index=False, float_fmt='{:.3f}', notes=None, col_fmt=None):
    """CSV + booktabs LaTeX (threeparttable with notes)."""
    df.to_csv(DIRS['tables'] / f'{name}.csv', index=index)
    d = df.copy()
    if not index: d = d.reset_index(drop=True)
    def fmt(v):
        if isinstance(v, (float, np.floating)):
            if np.isnan(v): return '--'
            return float_fmt.format(v)
        if isinstance(v, (int, np.integer)): return f'{v:,}'
        return str(v)
    body = d.map(fmt) if hasattr(d, 'map') else d.applymap(fmt)
    esc = tex_escape
    cols = ([esc(index if isinstance(index, str) else (d.index.name or ''))] if index else []) + [esc(c) for c in d.columns]
    ncol = len(cols); wide = ncol > 7
    if col_fmt is None:
        # journal layout: wrap long headers in fixed-width columns when the table has many columns (needs \usepackage{array})
        if ncol > 4:
            lens = [max([len(str(body.columns[0]))] + [len(str(x)) for x in body.iloc[:, 0]])] if len(body.columns) else [10]
            first_cm = float(np.clip(0.19 * lens[0], 1.6, 4.2))
            avail = (15.5 if ncol <= 7 else 16.5 if ncol <= 11 else 22.5) - first_cm
            w = float(np.clip(avail / max(ncol - 1, 1) - 0.25, 0.9, 2.6))
            col_fmt = ('l' if index else '') + f'>{{\\raggedright\\arraybackslash}}p{{{first_cm:.1f}cm}}' + \
                      ''.join(f'>{{\\raggedleft\\arraybackslash}}p{{{w:.2f}cm}}' for _ in range(len(d.columns) - 1))
        else:
            col_fmt = ('l' if index else '') + 'l' + 'r' * (len(d.columns) - 1)
    env = 'sidewaystable' if ncol > 11 else 'table'          # very wide tables: landscape (\usepackage{rotating})
    lines = [rf'\begin{{{env}}}' + ('' if env == 'sidewaystable' else '[!htbp]'), r'\centering', r'\small' if not wide else r'\footnotesize', r'\begin{threeparttable}',
             rf'\caption{{{esc(caption)}}}', rf'\label{{{label or "tab:" + name}}}', (r'\scriptsize\setlength{\tabcolsep}{3pt}' if wide else '') + rf'\begin{{tabular}}{{{col_fmt}}}', r'\toprule',
             ' & '.join(cols) + r' \\', r'\midrule']
    for i in range(len(body)):
        row = ([esc(d.index[i])] if index else []) + [esc(x) if not re.match(r'^[-\d.,eE+%]+$', str(x)) else str(x).replace('%', r'\%') for x in body.iloc[i]]
        lines.append(' & '.join(row) + r' \\')
    lines += [r'\bottomrule', r'\end{tabular}']
    if notes:
        lines += [r'\begin{tablenotes}\footnotesize', rf'\item \textit{{Notes:}} {esc(notes)}', r'\end{tablenotes}']
    lines += [r'\end{threeparttable}', rf'\end{{{env}}}']
    (DIRS['tables'] / f'{name}.tex').write_text('\n'.join(lines))
    return df

def show(df, n=20):
    try:
        from IPython.display import display
        display(df.head(n))
    except Exception:
        print(df.head(n).to_string())

# ----------------------------- METRICS ---------------------------------------
def m_mae(y, p): return float(np.nanmean(np.abs(y - p)))
def m_rmse(y, p): return float(np.sqrt(np.nanmean((y - p) ** 2)))
def m_mape(y, p): return float(100 * np.nanmean(np.abs((y - p) / y)))
def m_smape(y, p): return float(100 * np.nanmean(2 * np.abs(y - p) / (np.abs(y) + np.abs(p))))
def m_wape(y, p): return float(100 * np.nansum(np.abs(y - p)) / np.nansum(np.abs(y)))

def dm_test(e1, e2, h=1, power=2):
    """Diebold–Mariano with Harvey–Leybourne–Newbold small-sample correction. H1: model 2 more accurate if stat>0."""
    e1, e2 = np.asarray(e1), np.asarray(e2)
    d = np.abs(e1) ** power - np.abs(e2) ** power
    d = d[~np.isnan(d)]; T = len(d)
    if T < 5 or np.allclose(d, 0): return np.nan, np.nan
    dbar = d.mean()
    gam = [np.sum((d[k:] - dbar) * (d[:T - k] - dbar)) / T for k in range(h)]
    var = (gam[0] + 2 * sum(gam[1:])) / T
    if var <= 0: return np.nan, np.nan
    stat = dbar / np.sqrt(var) * np.sqrt((T + 1 - 2 * h + h * (h - 1) / T) / T)
    p = 2 * (1 - stats.t.cdf(abs(stat), df=T - 1))
    return float(stat), float(p)

def bh_fdr(p):
    p = np.asarray(p, float); n = np.sum(~np.isnan(p)); q = np.full_like(p, np.nan)
    idx = np.where(~np.isnan(p))[0]; order = idx[np.argsort(p[idx])]
    ranked = p[order] * n / np.arange(1, n + 1)
    q[order] = np.minimum.accumulate(ranked[::-1])[::-1].clip(max=1)
    return q

def stars(p):
    return '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''
print('Helpers ready.')

## Module 1 · Data ingestion and calendar validation
The raw panel is wide (`Name | Amount | date₁ … date_T`) and contains **only days on which the scraper ran**.
We (i) assign stable product IDs, (ii) build the observed matrix $P^{obs}$ (products × observed days) and the
full-calendar matrix $P^{cal}$ (products × every calendar day, `NaN` on scrape-gap days), and (iii) cross-check
prices against the earlier 2024 workbook, detecting days in that workbook that were filled by carry-forward.

In [ ]:
def _locate_input(path, label, required=True):
    p = Path(path)
    if p.exists(): return p
    if IN_COLAB and required:
        from google.colab import files
        print(f'{label} not found at {p}. Please upload it now ...')
        up = files.upload()
        for fn, b in up.items():
            (DIRS['data'] / fn).write_bytes(b)
        cand = list(DIRS['data'].glob(f'*{Path(path).suffix}'))
        if cand: return cand[0]
    if required: raise FileNotFoundError(f'{label} missing: {p}')
    return None

def m01_ingest():
    raw = pd.read_csv(_locate_input(CFG['raw_csv'], 'ChalDal_final.csv'))
    date_cols = [c for c in raw.columns if re.match(r'^\d{4}-\d{2}-\d{2}$', str(c))]
    meta = raw[['Name', 'Amount']].copy()
    meta['Name'] = meta['Name'].astype(str).str.strip()
    meta['Amount'] = meta['Amount'].astype(str).str.strip()
    meta.insert(0, 'product_id', [f'P{i:05d}' for i in range(len(meta))])
    obs = raw[date_cols].apply(pd.to_numeric, errors='coerce')
    obs.index = meta['product_id']; obs.columns = pd.to_datetime(date_cols)
    obs = obs.sort_index(axis=1)
    obs = obs.where(obs > 0)                                  # non-positive prices are invalid
    # stale-snapshot screening: a day on which (almost) no SKU differs from the previous scraped day is almost surely a
    # carried-forward/duplicated snapshot (with f≈3%/day the chance that none of ~5,000 SKUs changes is ≈0).
    stale = []
    if CFG['drop_stale_days']:
        V = obs.values
        share = np.r_[np.nan, (np.abs(np.diff(V, axis=1)) > 0).mean(0)]
        stale = list(obs.columns[np.where(share <= CFG['stale_threshold'])[0]])
        obs = obs.drop(columns=stale)
        print(f'Stale snapshot days removed: {len(stale)} (share of SKUs changed ≤ {CFG["stale_threshold"]:.2%})')
    cal_dates = pd.date_range(obs.columns.min(), obs.columns.max(), freq='D')
    cal = obs.reindex(columns=cal_dates)
    observed_day = pd.Series(cal_dates.isin(obs.columns), index=cal_dates, name='observed')

    # summary
    summ = pd.DataFrame({
        'Statistic': ['Products (SKUs)', 'Unique product names', 'First date', 'Last date', 'Calendar days',
                      'Stale snapshot days removed', 'Observed (valid) days', 'Missing days (gaps)', 'Share of calendar observed (%)',
                      'Price observations', 'Cell-level missing among observed days (%)', 'Median price (BDT)',
                      'Price range (BDT)'],
        'Value': [len(meta), meta['Name'].nunique(), str(cal_dates[0].date()), str(cal_dates[-1].date()), len(cal_dates),
                  len(stale), int(observed_day.sum()), int((~observed_day).sum()), round(100 * observed_day.mean(), 2),
                  int(obs.notna().sum().sum()), round(100 * obs.isna().mean().mean(), 3),
                  float(np.nanmedian(obs.values)), f'{np.nanmin(obs.values):,.0f} – {np.nanmax(obs.values):,.0f}']})
    summ['Value'] = summ['Value'].astype(str)
    save_table(summ, 'T01_dataset_summary', 'Summary of the Chaldal.com daily online price panel.',
               notes='Observed days are days with a valid complete snapshot. Stale days (no SKU price differs from the previous scraped day) are treated as missing because they are almost certainly carried-forward snapshots. All missing days are reconstructed in Module 4.')

    # cross-check with 2024 workbook (optional)
    xc = {}
    xp = _locate_input(CFG['raw_xlsx'], 'Dataset-2024.xlsx', required=False)
    if xp is not None:
        xl = pd.read_excel(xp)
        xd = [c for c in xl.columns if re.match(r'^\d{4}-\d{2}-\d{2}', str(c))]
        xl = xl[['Name'] + xd].copy(); xl['Name'] = xl['Name'].astype(str).str.strip()
        xl.columns = ['Name'] + list(pd.to_datetime([str(c)[:10] for c in xd]))
        xv = xl.set_index('Name').apply(pd.to_numeric, errors='coerce')
        xv = xv[~xv.index.duplicated(keep='first')]
        # carry-forward detection: days with zero price changes across ALL products
        chg = (xv.diff(axis=1).abs() > 0).mean()
        flat_days = chg.index[(chg == 0)][1:]
        not_in_final = [d for d in xv.columns if d not in obs.columns]
        # agreement on overlapping observed days for uniquely named products
        uniq = meta.drop_duplicates('Name', keep=False)
        m = uniq.merge(pd.DataFrame({'Name': xv.index}), on='Name')
        common_days = [d for d in xv.columns if d in obs.columns]
        a = obs.loc[m['product_id'], common_days].values
        b = xv.loc[m['Name'], common_days].values
        ok = ~np.isnan(a) & ~np.isnan(b)
        agree = float(np.mean(np.isclose(a[ok], b[ok])))
        xc = {'xlsx_products': int(len(xv)), 'xlsx_days': int(xv.shape[1]), 'matched_products': int(len(m)),
              'common_days': len(common_days), 'price_agreement_share': agree,
              'xlsx_days_absent_from_final': len(not_in_final),
              'absent_days_with_zero_changes_share': float(np.mean([chg.get(d, np.nan) == 0 for d in not_in_final])) if not_in_final else np.nan,
              'xlsx_flat_days': [str(d.date()) for d in flat_days]}
        print(f"Cross-check: {len(m)} products matched, agreement on {len(common_days)} common days = {agree:.4%}; "
              f"{xc['absent_days_with_zero_changes_share']:.0%} of days missing from the final panel are flat (carry-forward) in the workbook.")
    save_ck(meta, '01_meta')
    wide_to_ck(obs, '01_obs_wide')
    wide_to_ck(cal, '01_cal_wide')
    save_ck(observed_day.reset_index().rename(columns={'index': 'date'}), '01_calendar')
    xc['stale_days'] = [str(d.date()) for d in stale]
    save_ck(xc, '01_xcheck')
    show(summ)

run_module('m01', m01_ingest, ['01_meta', '01_obs_wide', '01_cal_wide', '01_calendar', '01_xcheck'])

## Module 2 · Data-quality audit
* **Pack-size parsing** – `Amount` → quantity in base units (g, ml, piece) and a unit price
  $P^{unit}_{it}=P_{it}/q_i$ for cross-sectional comparison (price *relatives* used in the indices are unit-free).
* **Category taxonomy** – the panel has no category field, so an ordered, transparent keyword dictionary assigns 18
  COICOP-style categories (edit `CATEGORY_RULES` and re-run with `FORCE_RERUN={'m02'}`).
* **Anomaly screening** – $r_{it}=\ln P_{it}-\ln P_{i,t-1}$ between consecutive observed days; extreme moves are
  *flagged → classified* (transient spike, persistent level shift, gap-spanning) – never deleted automatically.

In [ ]:
CATEGORY_RULES = [  # (category, is_food, regex) – first match wins; non-food first so that e.g. "Lemon floor cleaner" is not a fruit
    ('Sugar & sweeteners', 1, r'zero cal|sugar free|splenda|sweetener|sweetner'),          # priority food exceptions
    ('Dairy & eggs', 1, r'\beggs\b|\bdim\b\)'),
    ('Infant food & formula', 1, r'cerelac|formula|lactogen|optipro|\bnan\b|\bnido\b|growing up|follow up|infant milk|first infant|pureed|baby food|baby meal|pediasure|aptamil|biomil|eldomilk|eldobaby|heinz .*\(\d+ months?\)'),
    ('Pet care', 0, r'\bcat\b|\bdog\b|\bpet\b|whiskas|litter|smartheart|kitten|me-o|pedigree|drools|\bbird\b|budgie|rabbit food|royal canin|petslife|petme'),
    ('Baby care', 0, r'diaper|\bbaby\b(?! (corn|potato|carrot|spinach))|nappy|pacifier|feeder|formula milk|infant formula|cerelac|optipro|lactogen|infant|\(\d+ ?months?\)|\bm\+|toddler|huggies|mamypoko|avonee pants|breast pump|brest pump|\bnipple|cot sheet|follow up formula|growing up|pre nan|lacto fix|gastro-fix|pediasure'),
    ('Health & pharma', 0, r'tablet|capsule|ointment|face mask|\bmask\b|sanitizer|antiseptic|bandage|vitamin(?!.*eggs)|face shield|thermometer|inhaler|incepta|condom|first aid|chyawanprash|pain relief|\bbalm\b|orsaline|alcohol (swab|prep)|disinfectant|\beno\b|psyllium|isubgul|hand rub|oximeter|knee cap|glucomax|saline|triphala|oshwagandha|ashwagandha|licorice|josti modhu'),
    ('Personal care', 0, r'shampoo|conditioner|\bsoap|face ?wash|facial|lotion|body spray|deodorant|perfume|toothpaste|toothbrush|tooth powder|dental|(?<!angel )\bhair|(?<!ice )(?<!sour )(?<!whipping )(?<!cooking )(?<!fresh )(?<!full )(?<!coconut )(?<!sterilized )\bcream\b(?!\s*(&|and) onion)(?!\s*spread)(?!.*(biscuit|wafer|cracker|roll|cookie|cheese|cake|filled|sandwich|chocolate|stick|cone|milk powder|liquid milk))|razor|shav|sanitary|senora|joya|uptan|glycerin|petroleum jelly|vaseline|\blip\b|lip ?balm|(?<!with )\bskin\b(?!\s*(on|off|less))|serum|sunscreen|cologne|talc|handwash|hand wash|body wash|shower gel|(?<!virgin )coconut oil|cotton bud|wipes|\bnail|kajal|mehedi|henna|scrub(?!ber)|cleanser|cleansing|toner|moisturi|fairness|\bcomb\b|mouth ?wash|mouth freshener|beauty|cosmetic|fogg|dettol|\blux\b|\bdove\b|nivea|\bponds?\b|pond\'s|roll on|\bdeo\b|deo |cartridge|gillette|rezor|eyeliner|foundation|pomade|prickly heat|intimate|beard|cooling oil|argan|rosehip|marula|rose oil|massage oil|castor oil|neem oil|black seed oil|revlon|sensodyne|sweat pad|face care|shikakai|shimul|face magic|cotton wool|head cover|tea tree|pore strip|gel wash|perfume|attar'),
    ('Household & cleaning', 0, r'detergent|dishwash|dish wash|cleaner|harpic|\bvim\b|bleach|toilet|\bfloor|\bmop\b|broom|jharu|freshener|odonil|insect|mosquito|cockroach|aerosol|good knight|\bcoil\b|tissue|napkin|\bfoil\b|cling|garbage|trash bag|bucket|\bbowl\b|\brfl\b|plastic|(food|plastic|airtight|storage|lunch|powder) container|container (set|box)|\bbasket|lysol|phenyl|match ?box|matches|candle|batter(y|ies)|bulb|led light|\blamp|washing|laundry|stain|vanish|repellent|kills|\bpaper\b|scourer|scrubber|sponge(?! gourd)|\btray\b|hanger|\bcloth|\bmat\b|\bbin\b|towel|tiffin box|fabric (brightener|softner|softener|intelligent)|brightener|persil|clorox|polish|duster|incense|agarbatti|fragrance|glade|\brat\b|ukun|preservative|\bcoal\b|grill|lighter|\bplates?\b|spatula|spoon|chopping|food (cover|box)|dust pan|kitchen (tool|apron)|kitchen tools|flask|apron|gloves|zipbag|chefwrap|net bag|nozzle|comod|insulation|anti roach|\bmug\b(?!.*dal)|fry pan|fondant|silicon|leveler|water bottle|ceiling rose|\bsocket'),
    ('Stationery, apparel & misc.', 0, r'\bpen\b|pencil|stapler|\bfile\b|notebook|\bkhata\b|earphone|headphone|shuttlecock|\btoy|\bpillow\b(?!.*chips)|brush|charger|cable|motor oil|engine oil|(fully|semi) synthetic|syntium|\bglue|\btape\b|marker|calculator|racket|umbrella|\block\b|torch|\bdeli\b|\bkangaro\b|faber|stamp pad|\bclip|\bpins?\b|ruler|compass|\bbook|flag|jersey|cricket|badminton|doll|\bbear\b|train set|balloon|sash|lungi|saree|panjabi|cushion|three piece|ehram|knitting|thread|wrench|pliers|tools|scissor|pruning|\bbeej\b|malik seeds|mobil|omera|petronas|\d+w[- ]?\d+|mouse pad|raincoat|register|sticky notes|geometry|highlighter|eraser|card holder|leather|car wash|boxer|\bink\b|\blace\b|flash card|tupi|spider-man|needle|\bpages\b'),
    ('Edible oil & ghee', 1, r'(mustard|soybean|soyabean|sunflower|rice bran|palm|vegetable|canola|cooking|corn|sesame)\s*oil|\bghee\b|vanaspati|dalda|olive oil|saffola|edible oil|til\) oil|avocado oil|macadamia.*oil|virgin coconut oil'),
    ('Rice', 1, r'(?<!puffed )\brice\b(?!\s*(bran|flour|cooker|noodle|cracker|paper|cake))|\bchal\b|basmati|miniket|\batop\b|polao|chinigura|najirshail|nazirshail|kataribhog|katari|banglamoti|paijam'),
    ('Flour & bakery', 1, r'\batta\b|flour|maida|\bsuji\b|semolina|\bbread\b|\bbuns?\b|\brusk|toast|\bcake|muffin|brownie|croissant|beshon|besan|wheat|quinoa|sabudana|sagudana|weet-bix|bakorkhani|filo sheet|arrowroot'),
    ('Pulses', 1, r'\bdal\b|lentil|masur|moshur|moong|\bmug\b|chickpea|chola|khesari|kalai|\bboot\b'),
    ('Instant & ready food', 1, r'noodle|pasta|macaroni|spaghetti|\bsoup\b|vermicelli|shemai|semai|frozen|paratha|ready to|pizza|samosa|spring roll|nugget|popcorn chicken|chicken popcorn|french fries|sausage|meatball|singara|aloo puri|ramen|crab stick|curry mix|biryani|pilau|tikka|haleem mix|khichuri mix|ready mix'),
    ('Spices & condiments', 1, r'masala|spice|turmeric|holud|chil+i powder|morich gura|cumin|jira|coriander powder|dhonia gura|garam|cardamom|elach|cinnamon|clove|bay leaf|\bsalt\b|sauce|ketchup|vinegar|mayonnaise|pickle|achar|seasoning|\bpaste\b|food colou?r|baking|yeast|essence|stock cube|\bcube\b|tamarind|mustard seed|pepper|panch phoron|panch phoran|kalojira|chutney|ajwain|aji-no-moto|monosodium|(sesame|flax|chia|fenugreek|methi|poppy|tokma|radhuni|black) ?seeds?|flax ?seed|\bmace\b|nutmeg|saffron|chil+i (flakes|\(morich\) powder)|dried chil|coriander|kasundi|firni mix|jorda mix|falooda|dressing|\bmouri\b|fennel|posto|zarda|agar|gelatin|cocoa powder|pie filling|whipped topping|syrup|amlaki powder|jeera|shaadmishali|methi'),
    ('Beverages', 1, r'\btea\b|coffee|juice|\bdrink|\bcola\b|7 up|pepsi|coca|\bcoke\b|sprite|fanta|mojo|\b(mineral |drinking )?water\b(?! (pumpkin|spinach|lily|chestnut))|soda|horlicks|boost|ovaltine|\bensure\b|energy drink|sharbat|\btang\b|nescafe|electrolyte|beverage|squash|concentrate|\bcan\b|mountain dew|\bspeed\b|\bclemon\b|mirinda|rooh afza|glucose|mojito|fresh up|protein powder|ener-g|kola\b|lemongrass herbal|latte|cappuccino'),
    ('Snacks & confectionery', 1, r'biscuit|cookie|wafer|chips|crisps|chanachur|cracker|candy|chocolate|toffee|lollipop|jelly|dessert|custard|pudding|popcorn|\bnuts?\b|almond|cashew|peanut|raisin|\bdates\b|khejur|\bmuri\b|chira|cereal|\boats\b|cornflakes|corn flakes|muesli|\bjam\b|spread|crunch|\bbar\b|\bgum\b|mints?\b|pringles|lexus|oreo|kitkat|snickers|dairy milk|walnut|pistachio|badam|olives?\b|soan papdi|mallow|dalmoth|namkeen|kinder|cadbury|ferrero|\bplum\b|dried fruit|jhuribhaja|puffed rice|phuchka|twist|googly|mejanda|digestive|roll\b|ice cream|sweets'),
    ('Dairy & eggs', 1, r'\bmilk\b|cheese|butter(?!fly)|yogh?urt|\bdoi\b|\bcurd|\beggs?\b|paneer|lassi|\bcream\b|condensed|laban|\bmatha\b|roshogolla|rasgolla|\bnido\b'),
    ('Meat & fish', 1, r'chicken|beef|mutton|\bgoat\b|\bduck\b|\bfish\b|prawn|shrimp|chingri|\brui\b|hilsa|ilish|pangas|tilapia|\bkoi\b|squid|salami|\btuna\b|mrigel|katla|pabda|boal|\bmeat\b|kebab|\bkabab\b|keema|liver|gizzard|telapia|shing|magur|loitta|bata fish|\bpoa\b|\bcow\b|calamari|crab\b|paya'),
    ('Sugar & sweeteners', 1, r'\bsugar\b|chini|jaggery|\bgur\b|\bhoney\b|sweetener|sweetner|molasses|misri|splenda'),
    ('Fruits', 1, r'\bapple|banana|\bmango|orange|grape|guava|papaya|pineapple|daab|green coconut|malta|pomegranate|lichi|litchi|jackfruit|watermelon|\bkiwi|\bpear\b|sofeda|sapodilla|amra|boroi|\bkul\b|dragon fruit|strawberr|lemon\b|lime\b|\bjambura|pomelo|\bfruit\b|amloki|amla\b|peach|lychee|narikel|\bcoconut\b|paka pape|\blemu\b|chalta|\bpui fol|\bberry|avocado'),
    ('Vegetables', 1, r'potato|\balu\b|onion|piaj|peyaj|garlic|rosun|ginger|\bada\b|tomato|brinjal|begun|gourd|pumpkin|kumra|cabbage|cauliflower|carrot|cucumber|green chil|kacha morich|spinach|\bshak\b|\bsaag|radish|\bmula\b|\bbeans?\b|okra|dherosh|dheros|\blau\b|potol|chichinga|korola|karola|kakrol|capsicum|lettuce|coriander leaves|dhonia pata|mushroom|sweet corn|\bpepe\b|kochu|lemon grass|mint leaves|pudina|broccoli|beetroot|\bjhinga|\bdata\b|sajna|drumstick|green banana|kacha kola|green peas|motorshuti|kohlrabi|shalgom|asparagus|\bpui\b|\bpaan\b|betel|jalapeno|bombay chil+i|aloe vera|baby corn|gherkin'),
]
CAT_RE = [(c, f, re.compile(r, re.I)) for c, f, r in CATEGORY_RULES]
FOOD_CATS = [c for c, f, _ in CATEGORY_RULES if f == 1]

def classify_name(name):
    name = re.sub(r'\((free|buy|combo|with|extra)[^)]*\)', ' ', str(name), flags=re.I)   # strip free-gift / promo text
    for c, f, rx in CAT_RE:
        if rx.search(name):
            return c, f
    return 'Other / unclassified', 0

UNIT_MAP = {'gm': ('g', 1), 'g': ('g', 1), 'gram': ('g', 1), 'kg': ('g', 1000), 'ml': ('ml', 1), 'ltr': ('ml', 1000),
            'l': ('ml', 1000), 'litre': ('ml', 1000), 'liter': ('ml', 1000), 'pcs': ('piece', 1), 'pc': ('piece', 1),
            'pack': ('piece', 1), 'bundle': ('piece', 1), 'bundles': ('piece', 1), 'tablet': ('piece', 1), 'tablets': ('piece', 1)}

def parse_amount(a):
    s = str(a).lower().strip()
    if s in ('each', '1 each', 'nan', ''):
        return 1.0, 'piece', s != 'nan'
    m = re.match(r'^\s*([\d.]+)\s*([a-z]+)', s)
    if not m: return np.nan, 'unknown', False
    q, u = float(m.group(1)), m.group(2)
    if u not in UNIT_MAP: return np.nan, 'unknown', False
    base, mult = UNIT_MAP[u]
    return q * mult, base, True

def m02_quality():
    meta = load_ck('01_meta'); obs = wide_from_ck('01_obs_wide')
    cats = meta['Name'].map(classify_name)
    meta['category'] = [c for c, _ in cats]; meta['is_food'] = [f for _, f in cats]
    pa = meta['Amount'].map(parse_amount)
    meta['qty_base'] = [x[0] for x in pa]; meta['unit_base'] = [x[1] for x in pa]; meta['unit_parsed'] = [x[2] for x in pa]
    meta['variable_weight'] = meta['Name'].str.contains('±', regex=False)
    meta['combo_or_promo'] = meta['Name'].str.contains(r'combo|free|buy \d|offer|\+', case=False, regex=True)
    med = obs.median(axis=1)
    scale = np.where(meta['unit_base'].eq('g') | meta['unit_base'].eq('ml'), 1000.0, 1.0)  # per kg / litre / piece
    meta['median_price'] = med.values
    meta['median_unit_price'] = med.values / meta['qty_base'].values * scale
    meta['unit_price_basis'] = np.where(meta['unit_base'].eq('g'), 'BDT/kg', np.where(meta['unit_base'].eq('ml'), 'BDT/litre', 'BDT/piece'))

    # ---------------- anomaly screening on consecutive observed days ----------------
    days = obs.columns
    gap_after = np.r_[(np.diff(days.values).astype('timedelta64[D]').astype(int) > 1), False]   # gap between t and t+1
    L = np.log(obs.values)
    R = np.diff(L, axis=1)                        # r between obs day j and j+1
    thr, k = CFG['anomaly_logret'], CFG['reversal_window']
    rows = []
    ii, jj = np.where(np.abs(R) > thr)
    for i, j in zip(ii, jj):
        pre = L[i, j]; post = L[i, j + 1]
        nxt = L[i, j + 2:j + 2 + k]
        fut = L[i, j + 1:j + 1 + 7]
        if gap_after[j]:
            cls = 'Gap-spanning change'
        elif len(nxt) and np.any(np.abs(nxt - pre) < 0.02):
            cls = 'Transient spike (reverted)'
        elif len(fut) >= 5 and np.all(np.abs(fut - post) < 0.05):
            cls = 'Persistent level shift'
        else:
            cls = 'Unresolved'
        rows.append((meta['product_id'].iat[i], days[j], days[j + 1], float(np.exp(R[i, j]) - 1), cls))
    flags = pd.DataFrame(rows, columns=['product_id', 'date_from', 'date_to', 'pct_change', 'class'])
    flags = flags.merge(meta[['product_id', 'category']], on='product_id', how='left')

    # ---------------- tables ----------------
    cat_tab = (meta.groupby('category').agg(SKUs=('product_id', 'size'), Food=('is_food', 'max'),
                                             Median_price=('median_price', 'median'),
                                             Variable_weight=('variable_weight', 'mean'))
               .sort_values('SKUs', ascending=False).reset_index())
    cat_tab['Share (%)'] = 100 * cat_tab['SKUs'] / cat_tab['SKUs'].sum()
    cat_tab['Food'] = np.where(cat_tab['Food'] == 1, 'Yes', 'No')
    cat_tab['Variable_weight'] = 100 * cat_tab['Variable_weight']
    cat_tab = cat_tab.rename(columns={'category': 'Category', 'Median_price': 'Median price (BDT)', 'Variable_weight': 'Variable-weight SKUs (%)'})
    cat_tab = cat_tab[['Category', 'Food', 'SKUs', 'Share (%)', 'Median price (BDT)', 'Variable-weight SKUs (%)']]
    save_table(cat_tab, 'T02_category_composition', 'Category composition of the price panel.', float_fmt='{:.1f}',
               notes='Categories assigned by an ordered keyword taxonomy on product names (Module 2). Variable-weight SKUs are sold by approximate weight (±).')
    nz = R[np.abs(R) > 0]
    an = flags['class'].value_counts().rename_axis('Class').reset_index(name='Events')
    an['Share of flagged (%)'] = 100 * an['Events'] / max(len(flags), 1)
    an['Median |change| (%)'] = [100 * flags.loc[flags['class'] == c, 'pct_change'].abs().median() for c in an['Class']]
    an.loc[len(an)] = ['Total flagged', len(flags), 100.0, 100 * flags['pct_change'].abs().median() if len(flags) else np.nan]
    save_table(an, 'T03_anomaly_classification', 'Classification of extreme daily price movements (|Δln P| > ln 1.5).', float_fmt='{:.1f}',
               notes=f'Flagged among {np.sum(~np.isnan(R)):,} consecutive-observation price pairs ({len(nz):,} non-zero changes). Flagged values are retained in all datasets.')
    # unclassified diagnostics
    unc = meta.loc[meta['category'].eq('Other / unclassified'), 'Name']
    print(f'Unclassified SKUs: {len(unc)} ({100*len(unc)/len(meta):.1f}%). Examples:', unc.head(25).tolist())
    print(f'Unit parsed for {meta.unit_parsed.mean():.1%} of SKUs')

    # ---------------- figure ----------------
    fig, axes = plt.subplots(1, 2, figsize=(W2, 3.0), gridspec_kw={'width_ratios': [1.15, 1]})
    ax = axes[0]
    ct = cat_tab.sort_values('SKUs')
    ax.barh(ct['Category'], ct['SKUs'], color=[OKABE[0] if f == 'Yes' else GREY for f in ct['Food']], height=0.7)
    ax.set_xlabel('Number of SKUs'); ax.grid(axis='y', visible=False)
    ax.legend(handles=[matplotlib.patches.Patch(color=OKABE[0], label='Food'), matplotlib.patches.Patch(color=GREY, label='Non-food')], loc='lower right')
    ax.tick_params(axis='y', labelsize=6.5); panel_label(ax, 'a', x=-0.45)
    ax = axes[1]
    bins = np.linspace(-1, 1, 81)
    ax.hist(np.clip(nz, -1, 1), bins=bins, color=OKABE[0], alpha=0.85, edgecolor='white', linewidth=0.2)
    ax.set_yscale('log'); ax.axvline(thr, color=OKABE[1], ls='--', lw=0.8); ax.axvline(-thr, color=OKABE[1], ls='--', lw=0.8)
    ax.set_xlabel(r'Non-zero daily log price change $\Delta \ln P_{it}$'); ax.set_ylabel('Frequency (log scale)')
    ax.text(thr + 0.03, ax.get_ylim()[1] * 0.3, 'flag\nthreshold', fontsize=6.5, color=OKABE[1])
    panel_label(ax, 'b')
    fig.tight_layout()
    savefig(fig, 'F01_composition_and_changes', 'Panel composition by category (a) and distribution of non-zero daily log price changes (b).')

    save_ck(meta, '02_meta')
    save_ck(flags, '02_anomaly_flags')
    show(cat_tab); show(an)

run_module('m02', m02_quality, ['02_meta', '02_anomaly_flags'])

## Module 3 · Missingness audit and mechanism
Missingness matrix $M_{it}=1$ if $P_{it}$ is missing. Because the scraper either captures the *whole* catalogue or
nothing, $M_{it}$ is (almost) constant across $i$ on each day: missingness is **system-wide (date-level)** rather than
product-specific. We therefore report $MR$, $MR_i$, $MR_t$, $MR_c$, characterise every gap block, and test whether
gap onset is predictable from observed market conditions (a date-level MCAR/MAR diagnostic):
$$\Pr(\text{gap starts at } t+1)=\sigma(\beta_0+\beta_1\,\text{DOW}_t+\beta_2\,S_t+\beta_3\,|\bar r_t|+\beta_4\,\text{Event}_t+\beta_5\,t).$$
The **gap-spanning change test** compares the observed share of SKUs whose price differs across a gap of length
$L$ with the share implied by a constant daily hazard $f$: $1-(1-f)^{L+1}$.

In [ ]:
def m03_missingness():
    meta = load_ck('02_meta'); cal = wide_from_ck('01_cal_wide'); obs = wide_from_ck('01_obs_wide')
    calendar = load_ck('01_calendar'); calendar['date'] = pd.to_datetime(calendar['date'])
    observed = calendar.set_index('date')['observed'].astype(bool)
    M = cal.isna()
    MR = float(M.values.mean()); MR_t = M.mean(axis=0); MR_i = M.mean(axis=1)
    MR_c = M.groupby(meta.set_index('product_id')['category']).mean().mean(axis=1)

    # ------------- gap blocks -------------
    miss = (~observed).astype(int).values; dates = observed.index
    blocks, i = [], 0
    while i < len(miss):
        if miss[i]:
            j = i
            while j + 1 < len(miss) and miss[j + 1]: j += 1
            blocks.append((dates[i], dates[j], j - i + 1)); i = j + 1
        else:
            i += 1
    gaps = pd.DataFrame(blocks, columns=['start', 'end', 'length'])
    gaps['class'] = pd.cut(gaps['length'], [0, CFG['gap_short'], CFG['gap_long'], 10_000],
                           labels=[f'Short (≤{CFG["gap_short"]}d)', f'Medium ({CFG["gap_short"]+1}–{CFG["gap_long"]}d)', f'Long (>{CFG["gap_long"]}d)']).astype(str)
    # change across gap vs hazard benchmark
    L = np.log(obs.values); od = obs.columns
    consec = np.r_[np.diff(od.values).astype('timedelta64[D]').astype(int) == 1]
    dchg = (np.abs(np.diff(L, axis=1)) > 1e-9)
    f_daily = float(dchg[:, consec].mean())
    f_i = dchg[:, consec].mean(axis=1)                       # product-specific hazards (heterogeneous benchmark)
    rows = []
    for _, g in gaps.iterrows():
        jb = od.get_loc(g['start'] - pd.Timedelta(days=1)); ja = od.get_loc(g['end'] + pd.Timedelta(days=1))
        share = float(dchg[:, jb].mean()) if ja == jb + 1 else np.nan
        mean_abs = float(np.abs(L[:, ja] - L[:, jb]).mean())
        rows.append((share, 1 - (1 - f_daily) ** (g['length'] + 1), float(np.mean(1 - (1 - f_i) ** (g['length'] + 1))), 100 * mean_abs))
    gaps[['share_changed', 'share_expected_hazard', 'share_expected_het', 'mean_abs_logchg_x100']] = pd.DataFrame(rows, index=gaps.index)
    gaps['product_coverage'] = 0.0   # share of SKUs observed during the gap (system-wide gaps -> 0)

    # ------------- mechanism test (gap onset logit) -------------
    import statsmodels.api as sm
    S = pd.Series(np.r_[np.nan, dchg.mean(axis=0)], index=od)          # share changing vs previous observed day
    absr = pd.Series(np.r_[np.nan, np.abs(np.diff(L, axis=1)).mean(axis=0) * 100], index=od)
    nxt_missing = pd.Series([(d + pd.Timedelta(days=1)) in dates and not observed.get(d + pd.Timedelta(days=1), True) for d in od], index=od)
    X = pd.DataFrame({'sync_share': S, 'mean_abs_change': absr, 'trend_years': (od - od[0]).days / 365.25,
                      'weekend_fri_sat': od.dayofweek.isin([4, 5]).astype(int)}, index=od)
    ev = pd.Series(0, index=od)
    for a, b in CFG['events'].values():
        ev[(od >= a) & (od <= b)] = 1
    X['event_period'] = ev
    X['y'] = nxt_missing.astype(int)
    X = X.dropna()
    Xs = X.drop(columns='y').copy()
    for c in ['sync_share', 'mean_abs_change']:
        Xs[c] = (Xs[c] - Xs[c].mean()) / Xs[c].std()
    try:
        lg = sm.Logit(X['y'], sm.add_constant(Xs)).fit(disp=0, cov_type='HC1', method='bfgs', maxiter=2000)
        lr_p = float(lg.llr_pvalue)
        logit_tab = pd.DataFrame({'Variable': lg.params.index, 'Coefficient': lg.params.values, 'Robust SE': lg.bse.values,
                                  'z': lg.tvalues.values, 'p-value': lg.pvalues.values})
        logit_tab['Odds ratio'] = np.exp(logit_tab['Coefficient'])
        logit_tab['Sig.'] = [stars(p) for p in logit_tab['p-value']]
        names = {'const': 'Constant', 'sync_share': 'Synchronisation share (std.)', 'mean_abs_change': 'Mean |Δln P| (std.)',
                 'trend_years': 'Time trend (years)', 'weekend_fri_sat': 'Weekend (Fri/Sat)', 'event_period': 'Ramadan/Eid period'}
        logit_tab['Variable'] = logit_tab['Variable'].map(names)
        save_table(logit_tab, 'T05_missingness_mechanism', 'Logit model of scrape-gap onset (date-level missingness mechanism).',
                   notes=f'Dependent variable: next calendar day unobserved. N = {len(X)} observed days, {int(X.y.sum())} gap onsets. '
                         f'Heteroskedasticity-robust (HC1) SEs. Likelihood-ratio test of joint significance p = {lr_p:.3f}; McFadden pseudo-R² = {lg.prsquared:.3f}.')
    except Exception as e:
        print('Logit failed:', e); logit_tab = pd.DataFrame(); lr_p = np.nan
    # KS test: market activity on the day before a gap vs other days
    pre = X.loc[X.y == 1, 'mean_abs_change']; oth = X.loc[X.y == 0, 'mean_abs_change']
    ks = stats.ks_2samp(pre, oth) if len(pre) > 2 else None

    # ------------- tables -------------
    gtab = gaps.copy()
    gtab['start'] = gtab['start'].dt.strftime('%Y-%m-%d'); gtab['end'] = gtab['end'].dt.strftime('%Y-%m-%d')
    gtab = gtab.rename(columns={'start': 'Start', 'end': 'End', 'length': 'Days', 'class': 'Class',
                                'share_changed': 'SKUs changed (%)', 'share_expected_hazard': 'Hazard-implied (%)',
                                'share_expected_het': 'Het.-hazard-implied (%)',
                                'mean_abs_logchg_x100': 'Mean |Δln P|×100'})
    for c in ['SKUs changed (%)', 'Hazard-implied (%)', 'Het.-hazard-implied (%)']: gtab[c] *= 100
    gtab = gtab[['Start', 'End', 'Days', 'Class', 'SKUs changed (%)', 'Hazard-implied (%)', 'Het.-hazard-implied (%)', 'Mean |Δln P|×100']]
    save_table(gtab, 'T04_gap_inventory', 'Inventory of scrape gaps (system-wide missing days).', float_fmt='{:.2f}',
               notes=f'Hazard-implied share assumes a common daily price-change probability f = {f_daily:.4f} estimated on consecutive observed days: 1-(1-f)^(L+1); the heterogeneous version averages 1-(1-f_i)^(L+1) over SKU-specific hazards. Shares far below both benchmarks indicate clustered (batch) repricing rather than a memoryless hazard. No SKU is observed on any gap day (system-wide missingness).')
    cls = gaps.groupby('class').agg(Blocks=('length', 'size'), Days=('length', 'sum'), Max=('length', 'max')).reset_index().rename(columns={'class': 'Gap class'})
    cls['Share of calendar (%)'] = 100 * cls['Days'] / len(dates)
    save_table(cls, 'T04b_gap_classes', 'Scrape gaps by length class.', float_fmt='{:.2f}')
    summary = {'MR_overall': MR, 'MR_product_min': float(MR_i.min()), 'MR_product_max': float(MR_i.max()),
               'MR_category': MR_c.to_dict(), 'n_gap_blocks': int(len(gaps)), 'gap_days': int(gaps['length'].sum()),
               'longest_gap': int(gaps['length'].max()), 'f_daily': f_daily, 'logit_LR_p': lr_p,
               'ks_pre_gap_activity_p': float(ks.pvalue) if ks else np.nan,
               'system_wide': bool(MR_i.std() < 1e-9)}
    print(json.dumps({k: v for k, v in summary.items() if k != 'MR_category'}, indent=1, default=float))

    # ------------- figure -------------
    fig = plt.figure(figsize=(W2, 4.2))
    gs = fig.add_gridspec(2, 2, height_ratios=[0.8, 1.25], hspace=0.6, wspace=0.28)
    ax = fig.add_subplot(gs[0, :])
    ax.bar(MR_t.index, 1 - MR_t.values, width=1.0, color=OKABE[0], lw=0)
    for _, g in gaps[gaps['length'] >= 10].iterrows():
        mid = g['start'] + (g['end'] - g['start']) / 2
        ax.annotate(f"{g['length']} d", (mid, 1.04), ha='center', fontsize=6.5, color=OKABE[1], annotation_clip=False)
    ax.set_ylim(0, 1.0); ax.set_yticks([0, 1]); ax.set_yticklabels(['missing', 'observed'])
    ax.grid(False); date_axis(ax); ax.set_xlim(dates[0], dates[-1])
    ax.set_title(f'Daily observation status of the catalogue (MR = {100*MR:.1f}% of SKU-days)', pad=12)
    panel_label(ax, 'a', x=-0.06, y=1.08)
    ax = fig.add_subplot(gs[1, 0])
    gl = gaps.sort_values('length')
    colors = {c: col for c, col in zip(sorted(gaps['class'].unique()), [OKABE[1], OKABE[2], OKABE[0]])}
    bins = pd.cut(gaps['length'], [0, 1, 2, 7, 30, 10_000], labels=['1', '2', '3–7', '8–30', '>30'])
    cnt = bins.value_counts().sort_index(); days_ = gaps.groupby(bins, observed=False)['length'].sum()
    bars = ax.bar(cnt.index.astype(str), cnt.values, color=OKABE[0], width=0.65)
    for r, n_, d_ in zip(bars, cnt.values, days_.values):
        ax.text(r.get_x() + r.get_width() / 2, n_ + 0.3, f'{int(d_)} d', ha='center', fontsize=6.5, color=GREY)
    ax.set_xlabel('Gap length class (days)'); ax.set_ylabel('Number of gap blocks'); ax.grid(axis='x', visible=False)
    ax.set_ylim(0, cnt.max() * 1.18)
    panel_label(ax, 'b', x=-0.2)
    ax = fig.add_subplot(gs[1, 1])
    xx = np.arange(1, gaps['length'].max() + 2)
    ax.plot(xx, 100 * (1 - (1 - f_daily) ** (xx + 1)), color=GREY, ls='--', lw=0.9, label='Common hazard')
    ax.plot(xx, [100 * np.mean(1 - (1 - f_i) ** (x + 1)) for x in xx], color=OKABE[0], ls=':', lw=1.1, label='Heterogeneous hazard')
    ax.scatter(gaps['length'], 100 * gaps['share_changed'], s=14, color=OKABE[1], zorder=3, label='Observed across gap', edgecolor='white', lw=0.4)
    ax.set_xscale('log'); ax.set_xlabel('Gap length L (days, log scale)'); ax.set_ylabel('SKUs changed across gap (%)')
    ax.set_xticks([1, 2, 5, 10, 20, 50, 137]); ax.get_xaxis().set_major_formatter(matplotlib.ticker.ScalarFormatter())
    ax.legend(loc='upper left'); panel_label(ax, 'c', x=-0.2)
    savefig(fig, 'F02_missingness', 'Missingness structure. (a) Observation status by calendar day with gap lengths (≥10 days) annotated. (b) Number of gap blocks by length class (labels: total missing days). (c) Share of SKUs whose price differs across a gap versus common- and heterogeneous-hazard benchmarks.')

    save_ck(gaps, '03_gap_inventory')
    save_ck(summary, '03_missingness_summary')
    show(gtab, 40)

run_module('m03', m03_missingness, ['03_gap_inventory', '03_missingness_summary'])

## Module 4 · Missing-price reconstruction library
All methods operate on log prices $y_{it}=\ln P_{it}$ and fill gap blocks $[a,b]$ (length $L=b-a+1$) bridged by the
last pre-gap value $y_{i,a-1}$ and first post-gap value $y_{i,b+1}$; $\tau_k=k/(L+1)$.

| Level | Method | Bridge |
|---|---|---|
| 1 | LOCF / NOCB | $\hat y_{a+k-1}=y_{a-1}$ / $y_{b+1}$ |
| 1 | Linear | $\hat y=(1-\tau_k)y_{a-1}+\tau_k y_{b+1}$ |
| 2 | Seasonal–trend (DOW) | linear trend + SKU day-of-week profile $s_i(\text{dow})$ from detrended observed data |
| 3 | Kalman (local level) | RTS smoother, SKU-specific $q_i,r_i$ by method of moments |
| 4 | Sticky step (midpoint) | single price change at the gap midpoint – the MAE-optimal step under uniform change timing |
| 4 | Sticky stochastic (MI) | single change at a random day, $M$ draws (multiple imputation; its expectation equals linear) |
| 5 | Learned bridge (LightGBM) | cross-sectional model of $y_{t}-y_{a-1}$ given $\tau$, $L$, endpoint change, SKU hazard/volatility, recency of last change, **category- and market-wide change shares across the gap**, calendar |

*Why no plain matrix completion?* Every gap is **system-wide** – no SKU is observed inside it – so the time factors of a
low-rank model are unidentified within the gap and any factor bridge collapses to linear interpolation.
Cross-sectional information therefore enters through the learned bridge (Level 5), which pools gap behaviour across
thousands of SKUs and conditions on how many SKUs in the same category/market changed across the gap.

The library cell always runs (functions are needed by later modules); the execution cell is checkpointed.

In [ ]:
EPS = 1e-9

def blocks_from_cols(missing_cols):
    m = np.asarray(missing_cols, bool); out, i = [], 0
    while i < len(m):
        if m[i]:
            j = i
            while j + 1 < len(m) and m[j + 1]: j += 1
            out.append((i, j)); i = j + 1
        else:
            i += 1
    return out

def product_stats(Y):
    """SKU statistics from consecutive observed log prices (NaN-aware)."""
    D = Y[:, 1:] - Y[:, :-1]
    V = ~np.isnan(D)
    ch = (np.abs(np.nan_to_num(D)) > EPS) & V
    nv = V.sum(1).clip(min=1)
    f = ch.sum(1) / nv
    Dz = np.where(V, D, 0.0)
    mu = Dz.sum(1) / nv
    vol = np.sqrt(((Dz - mu[:, None]) ** 2 * V).sum(1) / nv)
    nz = np.where(ch, np.abs(Dz), np.nan)
    size = np.nan_to_num(np.nanmean(nz, axis=1)) if np.any(ch) else np.zeros(len(Y))
    up = np.nan_to_num(np.where(ch.sum(1) > 0, ((Dz > EPS) & V).sum(1) / ch.sum(1).clip(min=1), 0.5))
    # lag-1 autocovariance for Kalman moments
    D0 = Dz[:, 1:]; D1 = Dz[:, :-1]; V2 = V[:, 1:] & V[:, :-1]
    g1 = (((D0 - mu[:, None]) * (D1 - mu[:, None])) * V2).sum(1) / V2.sum(1).clip(min=1)
    return {'f': f, 'vol': vol, 'size': size, 'up': up, 'g0': vol ** 2, 'g1': g1, 'change_mask': ch}

def dow_profile(Y, dates, window=15):
    """SKU day-of-week seasonal profile from observed data detrended by a centred rolling median."""
    df = pd.DataFrame(Y.T, index=dates)
    trend = df.rolling(window, center=True, min_periods=3).median()
    R = (df - trend)
    prof = R.groupby(dates.dayofweek).mean().T.values          # N x 7
    prof = np.nan_to_num(prof - np.nanmean(prof, axis=1, keepdims=True))
    return prof

def _bounds(Y, a, b):
    T = Y.shape[1]
    yb = Y[:, a - 1] if a > 0 else Y[:, b + 1]
    ya = Y[:, b + 1] if b + 1 < T else yb
    return yb, ya

def fill_locf(Y, blocks, ctx):
    Z = Y.copy()
    for a, b in blocks:
        yb, _ = _bounds(Y, a, b); Z[:, a:b + 1] = yb[:, None]
    return Z

def fill_nocb(Y, blocks, ctx):
    Z = Y.copy()
    for a, b in blocks:
        _, ya = _bounds(Y, a, b); Z[:, a:b + 1] = ya[:, None]
    return Z

def fill_linear(Y, blocks, ctx):
    Z = Y.copy()
    for a, b in blocks:
        yb, ya = _bounds(Y, a, b); L = b - a + 1
        tau = np.arange(1, L + 1) / (L + 1)
        Z[:, a:b + 1] = yb[:, None] * (1 - tau) + ya[:, None] * tau
    return Z

def fill_seasonal(Y, blocks, ctx):
    Z = fill_linear(Y, blocks, ctx); prof = ctx['dow']; dows = ctx['dates'].dayofweek.values; T = Y.shape[1]
    for a, b in blocks:
        L = b - a + 1; tau = np.arange(1, L + 1) / (L + 1)
        sb = prof[:, dows[a - 1]] if a > 0 else 0; sa = prof[:, dows[b + 1]] if b + 1 < T else sb
        Z[:, a:b + 1] += prof[:, dows[a:b + 1]] - ((1 - tau) * np.asarray(sb)[:, None] + tau * np.asarray(sa)[:, None])
    return Z

def fill_kalman(Y, blocks, ctx):
    """Vectorised local-level Kalman filter + RTS smoother; only missing cells inside `blocks` are replaced."""
    st = ctx['stats']; N, T = Y.shape
    r = np.clip(-st['g1'], 1e-8, None); q = np.clip(st['g0'] - 2 * r, 1e-7, None)
    obs = ~np.isnan(Y)
    af = np.zeros((N, T)); Pf = np.zeros((N, T)); ap = np.zeros((N, T)); Pp = np.zeros((N, T))
    first = np.nanmean(Y[:, :5], axis=1)
    a, P = first, np.full(N, 1e-2)
    for t in range(T):
        ap[:, t] = a; Pp[:, t] = P + q
        o = obs[:, t]
        K = np.where(o, Pp[:, t] / (Pp[:, t] + r), 0.0)
        yt = np.where(o, Y[:, t], ap[:, t])
        a = ap[:, t] + K * (yt - ap[:, t]); P = (1 - K) * Pp[:, t]
        af[:, t] = a; Pf[:, t] = P
    As = af.copy()
    for t in range(T - 2, -1, -1):
        J = Pf[:, t] / Pp[:, t + 1]
        As[:, t] = af[:, t] + J * (As[:, t + 1] - ap[:, t + 1])
    Z = Y.copy()
    for a_, b_ in blocks:
        Z[:, a_:b_ + 1] = As[:, a_:b_ + 1]
    return Z

def fill_sticky_mid(Y, blocks, ctx):
    Z = Y.copy()
    for a, b in blocks:
        yb, ya = _bounds(Y, a, b); L = b - a + 1
        k = np.arange(1, L + 1)
        Z[:, a:b + 1] = np.where(k[None, :] <= (L + 1) / 2, yb[:, None], ya[:, None])
    return Z

def fill_sticky_draw(Y, blocks, ctx, rng=None):
    rng = rng or np.random.default_rng(ctx.get('seed', 0))
    Z = Y.copy()
    for a, b in blocks:
        yb, ya = _bounds(Y, a, b); L = b - a + 1
        j = rng.integers(1, L + 2, size=len(yb))                 # change happens on gap day j (L+1 = after gap)
        k = np.arange(1, L + 1)
        Z[:, a:b + 1] = np.where(k[None, :] < j[:, None], yb[:, None], ya[:, None])
    return Z

# ---------------- learned bridge ----------------
BRIDGE_FEATURES = ['tau', 'L', 'k', 'to_end', 'd', 'absd', 'f', 'vol', 'size', 'up', 'dsl', 'cat', 'cat_share', 'cat_mean_d',
                   'mkt_share', 'mkt_mean_absd', 'dow', 'month', 'level']
FEATURE_GROUPS = {'temporal_position': ['tau', 'k', 'to_end', 'L'], 'endpoint_change': ['d', 'absd'],
                  'product_history': ['f', 'vol', 'size', 'up', 'dsl', 'level'], 'category': ['cat', 'cat_share', 'cat_mean_d'],
                  'cross_section_market': ['mkt_share', 'mkt_mean_absd'], 'calendar': ['dow', 'month']}

def days_since_change(ch_mask, a):
    """days since the last observed price change before column a (capped at 365)."""
    sub = ch_mask[:, :max(a - 1, 1)]
    any_ = sub.any(1)
    last = sub.shape[1] - 1 - np.argmax(sub[:, ::-1], axis=1)
    return np.where(any_, np.minimum(a - 1 - last, 365), 365).astype(float)

def bridge_features(Y, a, b, ctx, rows=None):
    yb, ya = _bounds(Y, a, b); L = b - a + 1; N = len(yb)
    rows = np.arange(N) if rows is None else rows
    st = ctx['stats']; cat = ctx['cat_codes']; dates = ctx['dates']
    d = ya - yb; chg = np.abs(d) > EPS
    cs = pd.Series(chg).groupby(cat).mean(); cm = pd.Series(d).groupby(cat).mean()
    dsl = days_since_change(st['change_mask'], a)
    n, k = len(rows), np.arange(1, L + 1)
    R = np.repeat(rows, L); K = np.tile(k, n)
    F = pd.DataFrame({
        'tau': K / (L + 1), 'L': L, 'k': K, 'to_end': L + 1 - K, 'd': d[R], 'absd': np.abs(d[R]),
        'f': st['f'][R], 'vol': st['vol'][R], 'size': st['size'][R], 'up': st['up'][R], 'dsl': dsl[R],
        'cat': cat[R], 'cat_share': cs.reindex(cat).values[R], 'cat_mean_d': cm.reindex(cat).values[R],
        'mkt_share': chg.mean(), 'mkt_mean_absd': np.abs(d).mean(),
        'dow': dates[a + K - 1].dayofweek, 'month': dates[a + K - 1].month, 'level': yb[R]})
    F['cat'] = F['cat'].astype('category')
    return F, R, K

def fill_bridge(Y, blocks, ctx, model=None, features=None):
    model = model or ctx['bridge_model']; features = features or BRIDGE_FEATURES
    Z = Y.copy()
    for a, b in blocks:
        F, R, K = bridge_features(Y, a, b, ctx)
        yb, ya = _bounds(Y, a, b)
        pred = model.predict(F[features])
        Z[R, a + K - 1] = yb[R] + pred
    return Z

METHODS = {'LOCF': fill_locf, 'NOCB': fill_nocb, 'Linear': fill_linear, 'Seasonal-trend (DOW)': fill_seasonal,
           'Kalman (local level)': fill_kalman, 'Sticky step (midpoint)': fill_sticky_mid,
           'Sticky stochastic (MI)': fill_sticky_draw, 'Learned bridge (LightGBM)': fill_bridge}
METHOD_FAMILY = {'LOCF': 'baseline', 'NOCB': 'baseline', 'Linear': 'baseline', 'Seasonal-trend (DOW)': 'statistical',
                 'Kalman (local level)': 'statistical', 'Sticky step (midpoint)': 'sticky', 'Sticky stochastic (MI)': 'sticky',
                 'Learned bridge (LightGBM)': 'learned'}
METHOD_COLOR = dict(zip(METHODS, [GREY, '#999999', OKABE[0], OKABE[5], OKABE[2], OKABE[4], OKABE[3], OKABE[1]]))

def make_ctx(Y, dates, cat_codes, seed=0, bridge_model=None):
    return {'stats': product_stats(Y), 'dow': dow_profile(Y, dates), 'dates': dates, 'cat_codes': cat_codes,
            'seed': seed, 'bridge_model': bridge_model}

def candidate_starts(observed, L, margin=8):
    """Start columns s such that s-margin .. s+L+margin-1 are all originally observed (window fully inside observed data)."""
    o = np.asarray(observed, bool); T = len(o); cs = np.r_[0, np.cumsum(o)]
    out = []
    for s in range(margin, T - L - margin):
        lo, hi = s - margin, s + L + margin
        if cs[hi] - cs[lo] == hi - lo: out.append(s)
    return np.array(out)

def product_split(n, seed):
    r = np.random.default_rng(seed).permutation(n)
    return np.sort(r[: n // 2]), np.sort(r[n // 2:])

def train_bridge(Ycal, observed, dates, cat_codes, train_rows, n_windows, seed, max_rows, lengths=None, features=None, verbose=True):
    import lightgbm as lgb
    rng = np.random.default_rng(seed)
    lengths = lengths or [3, 7, 14, 21, 30, 45, 60, 90, 137]
    Xs, ys, ws = [], [], []
    per_win = max_rows // max(n_windows, 1)
    for w in range(n_windows):
        L = int(rng.choice(lengths)); cand = candidate_starts(observed, L)
        if len(cand) == 0: continue
        s = int(rng.choice(cand)); a, b = s, s + L - 1
        Ym = Ycal.copy(); Ym[:, a:b + 1] = np.nan
        ctx = make_ctx(Ym, dates, cat_codes, seed=seed)
        F, R, K = bridge_features(Ym, a, b, ctx, rows=train_rows)
        y = Ycal[R, a + K - 1] - Ym[R, a - 1]
        keep_p = np.where(F['absd'].values > EPS, 1.0, 0.25)
        m = rng.random(len(F)) < keep_p
        if m.sum() > per_win:
            idx = np.where(m)[0]; m[:] = False; m[rng.choice(idx, per_win, replace=False)] = True
        Xs.append(F[m]); ys.append(y[m]); ws.append(1 / keep_p[m])
    X = pd.concat(Xs, ignore_index=True); X['cat'] = X['cat'].astype('category')
    y = np.concatenate(ys); wgt = np.concatenate(ws)
    features = features or BRIDGE_FEATURES
    model = lgb.LGBMRegressor(objective='l1', n_estimators=500, learning_rate=0.05, num_leaves=63, min_child_samples=200,
                              subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, random_state=seed, verbose=-1)
    model.fit(X[features], y, sample_weight=wgt)
    if verbose: print(f'  bridge trained on {len(X):,} rows from {len(Xs)} masked windows; features={len(features)}')
    return model

In [ ]:
def m04_impute():
    meta = load_ck('02_meta'); cal = wide_from_ck('01_cal_wide')
    calendar = load_ck('01_calendar'); observed = calendar['observed'].astype(bool).values
    dates = cal.columns; Y = np.log(cal.values)
    cat_codes = pd.Categorical(meta['category']).codes
    blocks = blocks_from_cols(np.isnan(Y).all(0))
    train_rows, test_rows = product_split(len(Y), CFG['seed'])
    model = train_bridge(Y, observed, dates, cat_codes, train_rows, CFG['bridge_train_windows'], CFG['seed'], CFG['bridge_max_rows'])
    pickle.dump(model, open(DIRS['models'] / 'bridge_lgbm.pkl', 'wb'))
    ctx = make_ctx(Y, dates, cat_codes, seed=CFG['seed'], bridge_model=model)
    for name, fn in METHODS.items():
        t0 = time.time()
        Z = fn(Y, blocks, ctx)
        wide_to_ck(pd.DataFrame(np.exp(Z), index=cal.index, columns=dates), f'04_imp_{re.sub(r"[^A-Za-z]+", "_", name).strip("_")}')
        print(f'  {name:28s} filled {len(blocks)} blocks in {time.time()-t0:.1f}s')
    draws = np.stack([fill_sticky_draw(Y, blocks, ctx, rng=np.random.default_rng(CFG['seed'] + m)) for m in range(CFG['mi_draws'])]).astype('float32')
    save_ck({'draws_log': draws, 'index': list(cal.index), 'dates': [str(d.date()) for d in dates]}, '04_mi_draws')
    imp = pd.DataFrame(model.feature_importances_, index=BRIDGE_FEATURES, columns=['gain_splits']).sort_values('gain_splits', ascending=False)
    save_ck({'train_rows': train_rows.tolist(), 'test_rows': test_rows.tolist(), 'blocks': blocks,
             'methods': list(METHODS), 'feature_importance': imp['gain_splits'].to_dict()}, '04_imputation_meta')

def imp_name(m): return f'04_imp_{re.sub(r"[^A-Za-z]+", "_", m).strip("_")}'

run_module('m04', m04_impute, ['04_imputation_meta', '04_mi_draws'] + [imp_name(m) for m in METHODS])

## Module 5 · Masked-gap validation
Known data are artificially removed as **system-wide** blocks (all SKUs, $L\in\{7,14,30,45,137\}$ days) inside fully
observed stretches, reconstructed by every method, and compared with the truth on the **held-out half of SKUs**
(the learned bridge is trained on the other half – product-disjoint, no leakage).

Metrics go beyond point accuracy – a reconstruction must preserve *level, distribution, stickiness, volatility,
boundary continuity and the aggregate index*:
* point: MAE, RMSE, MAPE, sMAPE, WAPE (price level); MAE of log price; MAE conditional on SKUs that changed price
* distribution of daily log changes: KS, Wasserstein-1, KL divergence (binned)
* stickiness: |true − reconstructed| frequency of price change (pp); volatility error
* boundary: jump error at gap entry/exit, slope continuity $|\hat S-S|$ over the first $k=7$ days
* aggregate: RMSE of the Jevons index path (index points), mean category-index RMSE

In [ ]:
def score_window(Yt, Z, a, b, rows, cat_codes):
    L = b - a + 1
    T_ = Yt[rows, a - 1:b + 2]; H = Z[rows, a - 1:b + 2].copy(); H[:, 0] = T_[:, 0]; H[:, -1] = T_[:, -1]
    yt, yh = T_[:, 1:-1], H[:, 1:-1]
    P, Ph = np.exp(yt), np.exp(yh)
    dT, dH = np.diff(T_, axis=1), np.diff(H, axis=1)
    cT, cH = np.abs(dT) > EPS, np.abs(dH) > EPS
    changed = cT.any(1)
    out = {'MAE (BDT)': m_mae(P, Ph), 'RMSE (BDT)': m_rmse(P, Ph), 'MAPE (%)': m_mape(P, Ph), 'sMAPE (%)': m_smape(P, Ph),
           'WAPE (%)': m_wape(P, Ph), 'MAE log ×100': 100 * m_mae(yt, yh),
           'MAE log ×100 | changed': 100 * m_mae(yt[changed], yh[changed]) if changed.any() else np.nan,
           'Share SKUs changed (%)': 100 * changed.mean(),
           'Change-freq. error (pp)': 100 * float(np.mean(np.abs(cT.mean(1) - cH.mean(1)))),
           'Volatility error ×100': 100 * float(np.mean(np.abs(dT.std(1) - dH.std(1))))}
    rng = np.random.default_rng(0)
    a1, a2 = dT.ravel(), dH.ravel()
    s1 = a1 if len(a1) < 40000 else rng.choice(a1, 40000, replace=False)
    s2 = a2 if len(a2) < 40000 else rng.choice(a2, 40000, replace=False)
    out['KS'] = float(stats.ks_2samp(s1, s2).statistic)
    out['Wasserstein ×100'] = 100 * float(stats.wasserstein_distance(a1, a2))
    bins = np.r_[-10, np.linspace(-0.3, 0.3, 61), 10]
    h1 = np.histogram(a1, bins)[0] + 0.5; h2 = np.histogram(a2, bins)[0] + 0.5
    h1, h2 = h1 / h1.sum(), h2 / h2.sum()
    out['KL divergence'] = float(np.sum(h1 * np.log(h1 / h2)))
    out['Boundary jump error ×100'] = 100 * float(np.mean(np.abs(dH[:, 0] - dT[:, 0]) + np.abs(dH[:, -1] - dT[:, -1])) / 2)
    k = min(7, L)
    out['Slope error ×100'] = 100 * float(np.mean(np.abs((yh[:, k - 1] - H[:, 0]) / k - (yt[:, k - 1] - T_[:, 0]) / k)))
    It = 100 * (yt - T_[:, [0]]).mean(0); Ih = 100 * (yh - T_[:, [0]]).mean(0)
    out['Index RMSE (pts)'] = float(np.sqrt(np.mean((It - Ih) ** 2)))
    out['Index max abs err (pts)'] = float(np.max(np.abs(It - Ih)))
    cc = cat_codes[rows]; errs = []
    for c in np.unique(cc):
        m = cc == c
        if m.sum() >= 5:
            errs.append(np.sqrt(np.mean((100 * (yt[m] - T_[m][:, [0]]).mean(0) - 100 * (yh[m] - T_[m][:, [0]]).mean(0)) ** 2)))
    out['Category-index RMSE (pts)'] = float(np.mean(errs))
    perprod = np.abs(yt - yh).mean(1)
    return out, perprod

KEY_METRICS = ['WAPE (%)', 'Index RMSE (pts)', 'Change-freq. error (pp)', 'Wasserstein ×100']

def m05_validate():
    import lightgbm as lgb
    meta = load_ck('02_meta'); cal = wide_from_ck('01_cal_wide'); im = load_ck('04_imputation_meta')
    observed = load_ck('01_calendar')['observed'].astype(bool).values
    dates = cal.columns; Y = np.log(cal.values); cat_codes = pd.Categorical(meta['category']).codes
    test_rows = np.array(im['test_rows'])
    model = pickle.load(open(DIRS['models'] / 'bridge_lgbm.pkl', 'rb'))
    rng = np.random.default_rng(CFG['seed'] + 1000)
    recs, perprod, illus = [], {}, None
    for L in CFG['mask_lengths']:
        cand = candidate_starts(observed, L)
        if len(cand) == 0:
            print(f'  L={L}: no fully observed window – skipped'); continue
        starts = rng.choice(cand, size=min(CFG['mask_windows_per_length'], len(cand)), replace=False)
        for w, s in enumerate(starts):
            a, b = int(s), int(s + L - 1)
            Ym = Y.copy(); Ym[:, a:b + 1] = np.nan
            ctx = make_ctx(Ym, dates, cat_codes, seed=int(s), bridge_model=model)
            Zs = {}
            for name, fn in METHODS.items():
                Z = fn(Ym, [(a, b)], ctx)
                Zs[name] = Z
                sc, pp = score_window(Y, Z, a, b, test_rows, cat_codes)
                recs.append({'L': L, 'window': w, 'start': str(dates[a].date()), 'method': name, **sc})
                perprod.setdefault((L, name), []).append(pp)
            if illus is None and L == 45:
                pad = 20; lo, hi = max(a - pad, 0), min(b + pad, len(dates) - 1)
                tr = test_rows
                inside = np.abs(np.diff(Y[tr, a - 1:b + 2], axis=1)) > EPS
                chg_rows = tr[(np.abs(Y[tr, b + 1] - Y[tr, a - 1]) > 0.03) & (inside.sum(1) <= 2)]
                pick = int(chg_rows[np.argmax(np.abs(Y[chg_rows, b + 1] - Y[chg_rows, a - 1]) * (meta['is_food'].values[chg_rows] == 1))]) if len(chg_rows) else int(tr[0])
                illus = {'a': a, 'b': b, 'lo': lo, 'hi': hi, 'dates': [str(d.date()) for d in dates[lo:hi + 1]], 'pick': pick,
                         'name': meta['Name'].iat[pick] + ' ' + meta['Amount'].iat[pick],
                         'truth_prod': np.exp(Y[pick, lo:hi + 1]).tolist(),
                         'prod': {m: np.exp(Zs[m][pick, lo:hi + 1]).tolist() for m in METHODS},
                         'truth_idx': (100 * np.exp((Y[tr, lo:hi + 1] - Y[tr][:, [lo]]).mean(0))).tolist(),
                         'idx': {m: (100 * np.exp((Zs[m][tr, lo:hi + 1] - Y[tr][:, [lo]]).mean(0))).tolist() for m in METHODS}}
        print(f'  L={L:3d}: {len(starts)} windows evaluated')
    res = pd.DataFrame(recs)
    num = res.drop(columns=['window', 'start'])
    agg = num.groupby(['L', 'method']).agg(['mean', 'std'])
    mean = num.groupby(['L', 'method']).mean()
    # composite rank across key metrics and lengths
    rk = mean[KEY_METRICS].groupby(level='L').rank()
    wts = pd.Series(CFG['selection_weights'])[KEY_METRICS]
    comp = (rk * wts).sum(axis=1).div(wts.sum()).groupby(level='method').mean().sort_values()
    best = comp.index[0]
    alt = next(m for m in comp.index[1:] if METHOD_FAMILY[m] != METHOD_FAMILY[best])
    # paired tests: product-level mean |log error| best vs alternatives (pooled over windows), Wilcoxon signed-rank
    tests = []
    for L in sorted(res['L'].unique()):
        e_best = np.mean(perprod[(L, best)], axis=0)
        for m in METHODS:
            if m == best: continue
            e_m = np.mean(perprod[(L, m)], axis=0)
            d = e_m - e_best
            p = stats.wilcoxon(e_m, e_best, zero_method='zsplit').pvalue if np.any(d != 0) else 1.0
            tests.append({'L': L, 'Comparator': m, 'Mean Δ|log err|×100': 100 * d.mean(), 'Share SKUs best better (%)': 100 * np.mean(d > 0), 'p-value': p})
    tests = pd.DataFrame(tests); tests['q (BH)'] = bh_fdr(tests['p-value'].values); tests['Sig.'] = [stars(q) for q in tests['q (BH)']]

    # ------------- tables -------------
    order = list(comp.index)
    wide = []
    for m in order:
        row = {'Method': m, 'Family': METHOD_FAMILY[m]}
        for L in sorted(res['L'].unique()):
            row[f'WAPE L={L}'] = mean.loc[(L, m), 'WAPE (%)']
        for L in sorted(res['L'].unique()):
            row[f'Index RMSE L={L}'] = mean.loc[(L, m), 'Index RMSE (pts)']
        row['Mean rank'] = comp[m]
        wide.append(row)
    wide = pd.DataFrame(wide)
    save_table(wide, 'T06_imputation_validation_main', 'Masked-gap validation: price-level WAPE (%) and Jevons-index RMSE (index points) by gap length.',
               float_fmt='{:.3f}', notes=f'Means over {CFG["mask_windows_per_length"]} system-wide masked windows per length, evaluated on {len(test_rows):,} held-out SKUs. '
                                        f'Mean rank: weighted average of ranks over WAPE (1), index RMSE (2), change-frequency error (1) and Wasserstein distance (1) across lengths; index accuracy is double-weighted because the DPI is the primary output. Preferred: {best}; alternative: {alt}.')
    long = mean.reset_index()
    cols = ['L', 'method', 'WAPE (%)', 'sMAPE (%)', 'MAE log ×100', 'MAE log ×100 | changed', 'Change-freq. error (pp)', 'Volatility error ×100',
            'KS', 'Wasserstein ×100', 'KL divergence', 'Boundary jump error ×100', 'Slope error ×100', 'Index RMSE (pts)', 'Category-index RMSE (pts)']
    long = long[cols].rename(columns={'method': 'Method'})
    save_table(long, 'T07_imputation_validation_full', 'Full masked-gap validation results (all metrics).', float_fmt='{:.3f}',
               notes='Price-level, distributional, stickiness, boundary and aggregate-index metrics; see Module 5 for definitions.')
    save_table(tests, 'T08_imputation_tests', f'Paired Wilcoxon signed-rank tests: {best} versus each comparator (SKU-level mean absolute log error).',
               float_fmt='{:.4f}', notes='Positive Δ: comparator has larger error. q-values: Benjamini–Hochberg across all comparisons. *** q<0.01, ** q<0.05, * q<0.10.')

    # ------------- figures -------------
    Ls = sorted(res['L'].unique())
    fig, axes = plt.subplots(2, 2, figsize=(W2, 4.8))
    for ax, met, lab in zip(axes.ravel(), KEY_METRICS, 'abcd'):
        for m in order:
            mu = [mean.loc[(L, m), met] for L in Ls]; sd = [agg.loc[(L, m), (met, 'std')] for L in Ls]
            lw, z = (1.8, 5) if m == best else (0.9, 3)
            ax.errorbar(Ls, mu, yerr=sd, color=METHOD_COLOR[m], lw=lw, marker='o', ms=3, capsize=1.5, elinewidth=0.5, label=m, zorder=z)
        ax.set_xscale('log'); ax.set_xticks(Ls); ax.get_xaxis().set_major_formatter(matplotlib.ticker.ScalarFormatter())
        ax.set_xlabel('Masked gap length (days)'); ax.set_ylabel(met); panel_label(ax, lab, x=-0.17)
        if met in ('WAPE (%)', 'Index RMSE (pts)', 'Wasserstein ×100'):
            ax.set_yscale('log'); ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:g}')); ax.yaxis.set_minor_formatter(FuncFormatter(lambda v, _: ''))
    h, l = axes[0, 0].get_legend_handles_labels()
    fig.legend(h, l, loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.06))
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    savefig(fig, 'F03_imputation_validation', 'Masked-gap validation by gap length: (a) price WAPE, (b) Jevons-index RMSE, (c) error in the frequency of price changes, (d) Wasserstein distance between true and reconstructed daily log-change distributions. Error bars: ±1 s.d. across masked windows; thick line: preferred method.')

    if illus:
        dd = pd.to_datetime(illus['dates']); a, b, lo = illus['a'], illus['b'], illus['lo']
        show_m = ['LOCF', 'Linear', 'Kalman (local level)', 'Sticky step (midpoint)', 'Learned bridge (LightGBM)']
        fig, axes = plt.subplots(1, 2, figsize=(W2, 2.7))
        for ax, tkey, mkey, lab, ttl in [(axes[0], 'truth_prod', 'prod', 'a', 'Single SKU'), (axes[1], 'truth_idx', 'idx', 'b', 'Jevons index (held-out SKUs)')]:
            ax.axvspan(dd[a - lo], dd[b - lo], color=LIGHT, alpha=0.6, lw=0, label='Masked window')
            ax.plot(dd, illus[tkey], color='black', lw=1.6, label='Observed truth', drawstyle='steps-post' if tkey == 'truth_prod' else 'default')
            for m in show_m:
                v = np.array(illus[mkey][m]); seg = slice(a - lo - 1, b - lo + 2)
                ax.plot(dd[seg], v[seg], color=METHOD_COLOR[m], lw=1.1, ls='--' if m in ('LOCF', 'Linear') else '-', label=m,
                        drawstyle='steps-post' if (tkey == 'truth_prod' and 'Sticky' in m) else 'default')
            ax.set_title(ttl); date_axis(ax, '%d %b\n%Y', interval=1); panel_label(ax, lab, x=-0.15)
        axes[0].ticklabel_format(axis='y', useOffset=False, style='plain'); axes[0].set_ylabel('Price (BDT)'); axes[1].set_ylabel('Index (window start = 100)')
        h, l = axes[1].get_legend_handles_labels()
        fig.legend(h, l, loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.12))
        fig.tight_layout(rect=(0, 0.08, 1, 1))
        savefig(fig, 'F04_imputation_example', f'Illustrative {b-a+1}-day masked window. (a) {illus["name"]}; (b) aggregate Jevons index of held-out SKUs. Grey: masked period.')
    save_ck(res, '05_validation_results'); save_ck(tests, '05_validation_tests')
    save_ck({'preferred': best, 'alternative': alt, 'composite_rank': comp.to_dict(), 'illustration': illus}, '05_selection')
    print('Composite ranking:\n', comp.round(2).to_string()); print(f'Preferred = {best}; alternative = {alt}')
    show(wide)

run_module('m05', m05_validate, ['05_validation_results', '05_validation_tests', '05_selection'])

## Module 6 · Analytical datasets for robustness
| Dataset | Content | Used for |
|---|---|---|
| **A** Observed-only | scraped days only, gaps left missing | descriptive facts, stickiness, validation, robustness |
| **B** Preferred reconstruction | best method from Module 5 | continuous index, inflation, forecasting |
| **C** Alternative reconstruction | best method from a different family | sensitivity analysis |
| **D** Conservative | B, but gaps longer than `gap_long` days set to missing | robustness to long-gap reconstruction |
| **MI** | $M$ draws of the stochastic sticky bridge | uncertainty from reconstruction |

In [ ]:
def m06_datasets():
    sel = load_ck('05_selection'); cal = wide_from_ck('01_cal_wide'); gaps = load_ck('03_gap_inventory')
    B = wide_from_ck(imp_name(sel['preferred'])); C = wide_from_ck(imp_name(sel['alternative']))
    D = B.copy()
    for _, g in gaps[gaps['length'] > CFG['gap_long']].iterrows():
        D.loc[:, (D.columns >= g['start']) & (D.columns <= g['end'])] = np.nan
    for k, W in {'A': cal, 'B': B, 'C': C, 'D': D}.items():
        wide_to_ck(W, f'06_dataset_{k}')
    rows = []
    for k, W, desc in [('A', cal, 'Observed only'), ('B', B, f'Preferred: {sel["preferred"]}'), ('C', C, f'Alternative: {sel["alternative"]}'),
                       ('D', D, f'Conservative: gaps >{CFG["gap_long"]} d excluded')]:
        obs_days = int((~W.isna().all(0)).sum())
        rows.append({'Dataset': k, 'Definition': desc, 'Calendar days covered': obs_days,
                     'SKU-day cells': int(W.notna().sum().sum()), 'Reconstructed cells (%)': 100 * float((W.notna() & cal.isna()).sum().sum() / max(W.notna().sum().sum(), 1))})
    tab = pd.DataFrame(rows)
    save_table(tab, 'T09_analytical_datasets', 'Analytical datasets used throughout the analysis.', float_fmt='{:.2f}',
               notes='Dataset MI (multiple imputation, Module 4) provides additional draws of the stochastic sticky bridge for reconstruction-uncertainty bands.')
    save_ck({'preferred': sel['preferred'], 'alternative': sel['alternative']}, '06_datasets_meta')
    show(tab)

run_module('m06', m06_datasets, ['06_dataset_A', '06_dataset_B', '06_dataset_C', '06_dataset_D', '06_datasets_meta'])

## Module 7 · Digital Price Index (DPI) construction
**Elementary level** (within category $c$, fixed base $0$):
* Jevons $J_{ct}=\exp\big[\tfrac1{N_c}\sum_i\ln(P_{it}/P_{i0})\big]$ (ILO/IMF CPI Manual recommended elementary index)
* Dutot $D_{ct}=\sum_iP_{it}/\sum_iP_{i0}$
* Laspeyres $L_{ct}=\sum_i s_{i0}\,P_{it}/P_{i0}$ and Paasche $P_{ct}=\big[\sum_i s_{it}\,P_{i0}/P_{it}\big]^{-1}$ with
  **equal expenditure shares** (quantities are not observed online), i.e. Carli and harmonic means of price relatives;
  Fisher $F_{ct}=\sqrt{L_{ct}P_{ct}}$ (superlative; ≈ Jevons).

**Upper level** with category weights $w_c$ (`'sku'` = SKU shares, `'equal'`, or `'hies'` from `hies_weights.csv`):
geometric for Jevons, $\ln DPI_t=\sum_c w_c\ln J_{ct}$; arithmetic for Laspeyres/Dutot; harmonic for Paasche.
Indices are produced for All items, **Food (FPI)**, Non-food and every category, for datasets A–D and three weighting schemes.

In [ ]:
def elementary(Pw, base_col, kind):
    R = Pw / Pw[:, [base_col]]
    if kind == 'Jevons': return np.exp(np.nanmean(np.log(R), axis=0))
    if kind == 'Dutot': return np.nansum(Pw, axis=0) / np.nansum(Pw[:, base_col]) * np.where(np.isnan(Pw).all(0), np.nan, 1)
    if kind == 'Laspeyres': return np.nanmean(R, axis=0)
    if kind == 'Paasche': return 1 / np.nanmean(1 / R, axis=0)
    raise ValueError(kind)

def upper(Ic, w, kind):
    """Ic: dict cat->array; w: dict cat->weight (normalised over available cats)."""
    cats = [c for c in Ic if w.get(c, 0) > 0]
    W = np.array([w[c] for c in cats]); W = W / W.sum(); M = np.vstack([Ic[c] for c in cats])
    if kind == 'Jevons': return np.exp(W @ np.log(M))
    if kind == 'Paasche': return 1 / (W @ (1 / M))
    return W @ M

def category_weights(meta, scheme):
    cnt = meta['category'].value_counts()
    if scheme == 'equal': return {c: 1.0 for c in cnt.index}
    if scheme == 'hies':
        p = Path(CFG['hies_weights_csv'])
        if p.exists():
            hw = pd.read_csv(p); return dict(zip(hw['category'], hw['weight']))
        return None
    return (cnt / cnt.sum()).to_dict()

def build_indices(W, meta, base_col=0, formulas=('Jevons', 'Dutot', 'Laspeyres', 'Paasche'), schemes=('sku', 'equal', 'hies')):
    P = W.values; cats = meta['category'].values; food = meta['is_food'].values == 1
    out = {}
    Icat = {f: {} for f in formulas}
    for c in np.unique(cats):
        m = cats == c
        for f in formulas:
            Icat[f][c] = elementary(P[m], base_col, f)
            if f == 'Jevons': out[(c, 'Jevons', 'sku')] = Icat[f][c]
    for sch in schemes:
        wts = category_weights(meta, sch)
        if wts is None: continue
        for scope, sel in [('All items', np.ones_like(food)), ('Food', food), ('Non-food', ~food)]:
            cs = set(np.unique(cats[sel.astype(bool)]))
            w = {c: v for c, v in wts.items() if c in cs}
            for f in formulas:
                if sch != 'sku' and f != 'Jevons': continue
                out[(scope, f, sch)] = upper(Icat[f], w, f)
        if sch == 'sku' and 'Laspeyres' in formulas and 'Paasche' in formulas:
            for scope in ['All items', 'Food', 'Non-food']:
                out[(scope, 'Fisher', 'sku')] = np.sqrt(out[(scope, 'Laspeyres', 'sku')] * out[(scope, 'Paasche', 'sku')])
    return out

def annualised(series, dates):
    s = pd.Series(series, index=dates).dropna()
    yrs = (s.index[-1] - s.index[0]).days / 365.25
    return 100 * (np.log(s.iloc[-1] / s.iloc[0]) / yrs)

def m07_index():
    meta = load_ck('02_meta')
    frames = []
    for k in ['A', 'B', 'C', 'D']:
        W = wide_from_ck(f'06_dataset_{k}')
        base_col = 0 if CFG['base_date'] is None else W.columns.get_loc(pd.Timestamp(CFG['base_date']))
        res = build_indices(W, meta, base_col)
        for (scope, f, sch), v in res.items():
            frames.append(pd.DataFrame({'date': W.columns, 'dataset': k, 'scope': scope, 'formula': f, 'weights': sch, 'index': 100 * v}))
    idx = pd.concat(frames, ignore_index=True)
    save_ck(idx, '07_indices')
    dates = pd.to_datetime(sorted(idx['date'].unique()))

    def get(ds='B', scope='All items', f='Jevons', w='sku'):
        s = idx[(idx.dataset == ds) & (idx.scope == scope) & (idx.formula == f) & (idx.weights == w)].set_index('date')['index']
        return s.reindex(dates)

    # ---- table: formula comparison ----
    rows = []
    for scope in ['All items', 'Food', 'Non-food']:
        J = get(scope=scope)
        for f in ['Jevons', 'Dutot', 'Laspeyres', 'Paasche', 'Fisher']:
            s = get(scope=scope, f=f); dl = 100 * np.log(s).diff()
            rows.append({'Scope': scope, 'Formula': f + (' (=Carli)' if f == 'Laspeyres' else ' (=harmonic)' if f == 'Paasche' else ''),
                         'Index at end': s.dropna().iloc[-1], 'Annualised change (%)': annualised(s, dates),
                         'SD daily Δln ×100': dl.std(), 'Max drawdown (%)': 100 * float((s / s.cummax() - 1).min()),
                         'Corr. with Jevons (Δln)': np.corrcoef(dl.dropna(), 100 * np.log(J).diff().dropna())[0, 1]})
    ft = pd.DataFrame(rows)
    save_table(ft, 'T10_index_formulas', f'Digital Price Index by formula and scope (dataset B, SKU-share weights, base {dates[0].date()} = 100).',
               float_fmt='{:.3f}', notes='Laspeyres and Paasche use equal expenditure shares (base and current period) because quantities are not observed; Fisher is their geometric mean. Annualised change = 100·ln(I_T/I_0)/years.')
    # ---- table: category indices ----
    rows = []
    for c in meta['category'].value_counts().index:
        s = get(scope=c); sA = get('A', c)
        rows.append({'Category': c, 'Food': 'Yes' if meta.loc[meta.category == c, 'is_food'].iloc[0] == 1 else 'No',
                     'SKUs': int((meta.category == c).sum()), 'Index at end': s.dropna().iloc[-1],
                     'Annualised change (%)': annualised(s, dates), 'Latest YoY (%)': 100 * float(np.log(s.dropna().iloc[-1] / s.reindex([s.dropna().index[-1] - pd.Timedelta(days=365)]).iloc[0])),
                     'SD daily Δln ×100': (100 * np.log(s).diff()).std()})
    ct = pd.DataFrame(rows).sort_values('Annualised change (%)', ascending=False)
    save_table(ct, 'T11_category_indices', 'Category-level Jevons indices (dataset B).', float_fmt='{:.2f}',
               notes='Annualised log change over the full sample; latest YoY = 100·ln(I_T/I_{T-365}).')
    # ---- table: weighting sensitivity ----
    rows = []
    for scope in ['All items', 'Food', 'Non-food']:
        for w in idx.loc[idx.formula.eq('Jevons') & idx.scope.eq(scope), 'weights'].unique():
            s = get(scope=scope, w=w)
            rows.append({'Scope': scope, 'Weights': {'sku': 'SKU shares', 'equal': 'Equal', 'hies': 'HIES expenditure'}[w],
                         'Index at end': s.dropna().iloc[-1], 'Annualised change (%)': annualised(s, dates)})
    save_table(pd.DataFrame(rows), 'T12_weight_sensitivity', 'Sensitivity of the Jevons DPI to upper-level category weights (dataset B).', float_fmt='{:.3f}')

    # ---- figure: headline DPI ----
    fig, axes = plt.subplots(1, 2, figsize=(W2, 2.9), gridspec_kw={'width_ratios': [1.35, 1]})
    ax = axes[0]; shade_gaps(ax)
    for scope, col in [('All items', OKABE[0]), ('Food', OKABE[1]), ('Non-food', OKABE[2])]:
        ax.plot(dates, get(scope=scope), color=col, lw=1.1, label=f'{scope} (B)')
        ax.plot(dates, get('C', scope), color=col, lw=0.6, ls=':', alpha=0.9)
        a = get('A', scope); ax.scatter(a.index, a.values, s=0.8, color=col, alpha=0.35, lw=0)
    ax.plot([], [], color=GREY, ls=':', lw=0.8, label='Alternative reconstruction (C)')
    ax.scatter([], [], s=4, color=GREY, label='Observed days (A)')
    ax.set_ylabel(f'Index ({dates[0].strftime("%d %b %Y")} = 100)'); date_axis(ax); ax.legend(loc='lower right', ncol=1)
    ax.set_title('Jevons Digital Price Index'); panel_label(ax, 'a', x=-0.12)
    ax = axes[1]; shade_gaps(ax, label=False)
    for f, col, ls in [('Jevons', OKABE[0], '-'), ('Dutot', OKABE[4], '-'), ('Laspeyres', OKABE[1], '--'), ('Paasche', OKABE[2], '--'), ('Fisher', 'black', ':')]:
        ax.plot(dates, get(scope='Food', f=f), color=col, ls=ls, lw=1.0, label=f + (' (Carli)' if f == 'Laspeyres' else ' (harmonic)' if f == 'Paasche' else ''))
    ax.set_ylabel('Food index'); date_axis(ax, interval=4); ax.legend(loc='upper left'); ax.set_title('Food DPI by index formula')
    panel_label(ax, 'b', x=-0.15)
    fig.tight_layout()
    savefig(fig, 'F05_dpi_headline', 'Digital Price Index. (a) All-items, food and non-food Jevons indices from the preferred reconstruction (solid), the alternative reconstruction (dotted) and observed days only (points); grey bands mark scrape gaps. (b) Food index under alternative elementary formulas.')

    # ---- figure: category small multiples ----
    order = ct.sort_values(['Food', 'Annualised change (%)'], ascending=[False, False])['Category'].tolist()
    n = len(order); ncol = 6; nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(W2, 1.3 * nrow + 0.3), sharex=True)
    for ax, c in zip(axes.ravel(), order):
        shade_gaps(ax, label=False)
        s = get(scope=c); ax.plot(dates, s, color=OKABE[1] if meta.loc[meta.category == c, 'is_food'].iloc[0] else OKABE[0], lw=0.9)
        ax.axhline(100, color=GREY, lw=0.4)
        ax.set_title(f'{c}\n({annualised(s, dates):+.1f}% p.a.)', fontsize=6.5, fontweight='normal', loc='left')
        ax.tick_params(labelsize=5.5, labelbottom=True)
        ax.xaxis.set_major_locator(mdates.YearLocator()); ax.xaxis.set_major_formatter(mdates.DateFormatter("'%y"))
    for ax in axes.ravel()[n:]: ax.axis('off')
    fig.tight_layout(h_pad=0.6, w_pad=0.4)
    savefig(fig, 'F06_category_indices', 'Category Jevons indices (dataset B; base = 100). Orange: food categories; blue: non-food. Annualised log change in parentheses. Grey bands: scrape gaps.')
    show(ft); show(ct, 25)

run_module('m07', m07_index, ['07_indices'])

## Module 8 · Price dynamics: stickiness, volatility, synchronisation, seasonality, structural breaks
All micro statistics use **dataset A** and only transitions between *consecutive calendar days that are both
observed*, so no statistic is contaminated by reconstruction.
* Price-change indicator $I_{it}=\mathbb 1[P_{it}\neq P_{i,t-1}]$; frequency $f_i$; implied duration $d_i=-1/\ln(1-f_i)$
* Empirical hazard $h(d)=\Pr(\text{change after } d \text{ days}\mid\text{unchanged for } d-1)$ from price spells,
  with right-censoring at the end of each observed run (Kaplan–Meier)
* Synchronisation $S_t$ = share of SKUs changing price on day $t$; Fisher–Konieczny index
  $FK=\sqrt{\operatorname{Var}(S_t)/[\bar f(1-\bar f)]}$ (0 = perfectly staggered, 1 = perfectly synchronised)
* Structural breaks: PELT change points in the 7-day DPI inflation rate (dataset B) + OLS-CUSUM stability test

In [ ]:
def transitions(W):
    """Δln P between consecutive calendar days that are both observed (NaN otherwise)."""
    L = np.log(W.values); D = np.full_like(L, np.nan); D[:, 1:] = L[:, 1:] - L[:, :-1]
    return D

def spells(D):
    """Price spells per SKU from change indicators; returns arrays (duration, event, row)."""
    dur, ev, rows = [], [], []
    N, T = D.shape
    valid = ~np.isnan(D)
    for i in range(N):
        v = valid[i]; c = np.abs(np.nan_to_num(D[i])) > EPS
        t = 0
        while t < T:
            if not v[t]: t += 1; continue
            s = t                                     # run of valid transitions [s, e)
            while t < T and v[t]: t += 1
            e = t
            idx = np.where(c[s:e])[0]
            if len(idx) == 0: continue
            for k in range(1, len(idx)):              # complete spells (left-censored first spell dropped)
                dur.append(idx[k] - idx[k - 1]); ev.append(1); rows.append(i)
            tail = (e - s) - idx[-1] - 1              # right-censored
            if tail > 0: dur.append(tail); ev.append(0); rows.append(i)
    return np.array(dur), np.array(ev), np.array(rows)

def km_hazard(dur, ev, dmax=120):
    h, S, s = [], [], 1.0
    for d in range(1, dmax + 1):
        at_risk = np.sum(dur >= d); events = np.sum((dur == d) & (ev == 1))
        hd = events / at_risk if at_risk > 0 else np.nan
        h.append(hd); s *= (1 - (hd if np.isfinite(hd) else 0)); S.append(s)
    return np.array(h), np.array(S)

def m08_dynamics():
    import ruptures as rpt
    from statsmodels.stats.diagnostic import breaks_cusumolsresid
    meta = load_ck('02_meta'); A = wide_from_ck('06_dataset_A'); idx = load_ck('07_indices')
    dates = A.columns; D = transitions(A); V = ~np.isnan(D); C = (np.abs(np.nan_to_num(D)) > EPS) & V
    up = (np.nan_to_num(D) > EPS) & V; dn = (np.nan_to_num(D) < -EPS) & V
    nv = V.sum(1).clip(min=1)
    f = C.sum(1) / nv
    Dz = np.where(V, D, np.nan)
    prod = pd.DataFrame({'product_id': meta['product_id'], 'category': meta['category'], 'is_food': meta['is_food'],
                         'freq': f, 'duration': np.where(f > 0, -1 / np.log(1 - np.clip(f, 1e-9, 0.999)), np.nan),
                         'n_changes': C.sum(1), 'share_up': np.where(C.sum(1) > 0, up.sum(1) / C.sum(1).clip(min=1), np.nan),
                         'size_up': 100 * np.nanmean(np.where(up, Dz, np.nan), axis=1),
                         'size_down': -100 * np.nanmean(np.where(dn, Dz, np.nan), axis=1),
                         'abs_size': 100 * np.nanmean(np.where(C, np.abs(Dz), np.nan), axis=1),
                         'vol': 100 * np.nanstd(Dz, axis=1),
                         'cv_level': A.std(axis=1).values / A.mean(axis=1).values,
                         'kurt_nonzero': [stats.kurtosis(Dz[i][C[i]]) if C[i].sum() > 3 else np.nan for i in range(len(Dz))],
                         'max_abs_change': 100 * np.nanmax(np.abs(Dz), axis=1)})
    # synchronisation
    S = pd.Series(C.sum(0) / V.sum(0).clip(min=1), index=dates).where(V.sum(0) > 0)
    fbar = float(C.sum() / V.sum())
    FK = float(np.sqrt(S.var() / (fbar * (1 - fbar))))
    top5 = S.dropna().quantile(0.95)
    share_changes_top5 = float(C[:, (S >= top5).values].sum() / C.sum())
    # per category table
    rows = []
    for c, g in prod.groupby('category'):
        m = (meta['category'] == c).values
        Sc = C[m].sum(0) / V[m].sum(0).clip(min=1); Sc = Sc[V[m].sum(0) > 0]; fc = C[m].sum() / V[m].sum()
        rows.append({'Category': c, 'Food': 'Yes' if g.is_food.iloc[0] else 'No', 'SKUs': len(g),
                     'Freq. of change (%/day)': 100 * fc, 'Median implied duration (days)': g['duration'].median(),
                     'SKUs never changing (%)': 100 * (g['n_changes'] == 0).mean(), 'Increases (% of changes)': 100 * up[m].sum() / max(C[m].sum(), 1),
                     'Mean increase (%)': np.nanmean(g['size_up']), 'Mean decrease (%)': np.nanmean(g['size_down']),
                     'Daily volatility ×100': g['vol'].mean(), 'FK sync.': np.sqrt(np.var(Sc) / (fc * (1 - fc))) if 0 < fc < 1 else np.nan})
    ptab = pd.DataFrame(rows).sort_values('Freq. of change (%/day)', ascending=False)
    allrow = {'Category': 'All SKUs', 'Food': '', 'SKUs': len(prod), 'Freq. of change (%/day)': 100 * fbar,
              'Median implied duration (days)': prod['duration'].median(), 'SKUs never changing (%)': 100 * (prod['n_changes'] == 0).mean(),
              'Increases (% of changes)': 100 * up.sum() / C.sum(), 'Mean increase (%)': np.nanmean(prod['size_up']),
              'Mean decrease (%)': np.nanmean(prod['size_down']), 'Daily volatility ×100': prod['vol'].mean(), 'FK sync.': FK}
    ptab = pd.concat([ptab, pd.DataFrame([allrow])], ignore_index=True)
    save_table(ptab, 'T13_price_setting', 'Price-setting statistics by category (observed consecutive days, dataset A).', float_fmt='{:.2f}',
               notes=f'Implied duration d = -1/ln(1-f) at SKU level (SKUs that never change excluded from the median). FK = Fisher–Konieczny synchronisation index. '
                     f'{100*share_changes_top5:.1f}% of all price changes occur on the 5% of days with the highest synchronisation.')

    # hazard
    dur, ev, srow = spells(D)
    food_rows = set(np.where(meta['is_food'].values == 1)[0])
    isf = np.array([r in food_rows for r in srow])
    hz = {k: km_hazard(dur[m], ev[m]) for k, m in [('All', np.ones_like(isf)), ('Food', isf), ('Non-food', ~isf)]}

    # seasonality & events
    dfS = pd.DataFrame({'S': S, 'up': pd.Series(up.sum(0) / V.sum(0).clip(min=1), index=dates).where(V.sum(0) > 0)})
    dfS['dow'] = dates.dayofweek; dfS['month'] = dates.month
    ev_mask = pd.Series(False, index=dates)
    for a, b in CFG['events'].values(): ev_mask |= (dates >= a) & (dates <= b)
    dfS['event'] = ev_mask
    dd = dfS.dropna()
    kw_dow = stats.kruskal(*[g['S'].values for _, g in dd.groupby('dow')])
    kw_mon = stats.kruskal(*[g['S'].values for _, g in dd.groupby('month')])
    mw_ev = stats.mannwhitneyu(dd.loc[dd.event, 'S'], dd.loc[~dd.event, 'S']) if dd.event.any() else None
    # chi-square on pooled change counts: event vs non-event days
    ce = C[:, ev_mask.values].sum(); ve = V[:, ev_mask.values].sum(); cn = C[:, ~ev_mask.values].sum(); vn = V[:, ~ev_mask.values].sum()
    chi = stats.chi2_contingency([[ce, ve - ce], [cn, vn - cn]]) if ve > 0 else None
    seas = pd.DataFrame([
        {'Test': 'Day-of-week effect on change frequency', 'Statistic': 'Kruskal–Wallis H', 'Value': kw_dow.statistic, 'p-value': kw_dow.pvalue},
        {'Test': 'Month-of-year effect on change frequency', 'Statistic': 'Kruskal–Wallis H', 'Value': kw_mon.statistic, 'p-value': kw_mon.pvalue},
        {'Test': 'Ramadan/Eid vs other days (daily S_t)', 'Statistic': 'Mann–Whitney U', 'Value': mw_ev.statistic if mw_ev else np.nan, 'p-value': mw_ev.pvalue if mw_ev else np.nan},
        {'Test': f'Ramadan/Eid vs other days (pooled; {100*ce/max(ve,1):.2f}% vs {100*cn/vn:.2f}%)', 'Statistic': 'χ²', 'Value': chi[0] if chi else np.nan, 'p-value': chi[1] if chi else np.nan}])
    seas['Sig.'] = [stars(p) for p in seas['p-value']]
    save_table(seas, 'T14_seasonality_tests', 'Calendar and event effects on the frequency of price changes.', float_fmt='{:.4g}',
               notes='S_t is the share of SKUs changing price on day t (dataset A, consecutive observed days). Event periods as configured in CFG["events"].')

    # structural breaks on DPI inflation (dataset B)
    br_rows, brk = [], {}
    gaps = load_ck('03_gap_inventory')
    gap_edges = pd.to_datetime(list(gaps['start']) + list(gaps['end']))
    for scope in ['All items', 'Food', 'Non-food']:
        s = idx[(idx.dataset == 'B') & (idx.scope == scope) & (idx.formula == 'Jevons') & (idx.weights == 'sku')].set_index('date')['index']
        s.index = pd.to_datetime(s.index)
        x = (100 * np.log(s).diff(7)).dropna()                      # 7-day inflation, %
        pen = CFG['pelt_penalty'] or 2 * np.log(len(x)) * float(np.var(x.values))
        algo = rpt.Pelt(model='l2', min_size=30, jump=1).fit(x.values.reshape(-1, 1))
        bk = [b for b in algo.predict(pen=pen) if b < len(x)]
        brk[scope] = (x, bk)
        cus = breaks_cusumolsresid(x.values - x.values.mean())
        for b in bk:
            d = x.index[b]
            inside = any((g['start'] <= d <= g['end']) for _, g in gaps.iterrows())
            near_gap = np.min(np.abs((gap_edges - d).days)) <= 7
            loc = 'Inside gap' if inside else ('Gap edge (±7 d)' if near_gap else 'Observed')
            near_ev = [k for k, (a1, b1) in CFG['events'].items() if pd.Timestamp(a1) - pd.Timedelta(days=14) <= d <= pd.Timestamp(b1) + pd.Timedelta(days=14)]
            br_rows.append({'Scope': scope, 'Break date': str(d.date()), 'Mean before (%/7d)': x.iloc[max(0, b - 30):b].mean(),
                            'Mean after (%/7d)': x.iloc[b:b + 30].mean(), 'Location': loc,
                            'Near event': ', '.join(near_ev) if near_ev else '', 'CUSUM p (scope)': cus[1]})
    brtab = pd.DataFrame(br_rows)
    save_table(brtab, 'T15_structural_breaks', 'PELT change points in 7-day DPI inflation (dataset B, Jevons).', float_fmt='{:.3f}',
               notes='Penalty = 2·ln(T)·Var(x) (BIC-type), minimum segment 30 days. Breaks located inside or within 7 days of a scrape gap may be reconstruction artefacts and are not interpreted economically. CUSUM: OLS-residual CUSUM test of parameter stability.')

    # ---------------- figures ----------------
    fig, axes = plt.subplots(1, 3, figsize=(W2, 2.5))
    ax = axes[0]
    dpos = prod['duration'].dropna()
    ax.hist(np.log10(dpos.clip(upper=2000)), bins=40, color=OKABE[0], edgecolor='white', lw=0.2)
    ax.set_xticks(np.log10([1, 7, 30, 90, 365])); ax.set_xticklabels(['1', '7', '30', '90', '365'])
    ax.axvline(np.log10(dpos.median()), color=OKABE[1], ls='--', lw=0.9)
    ax.text(np.log10(dpos.median()) + 0.05, ax.get_ylim()[1] * 0.9, f'median\n{dpos.median():.0f} d', fontsize=6.5, color=OKABE[1], va='top')
    ax.set_xlabel('Implied price duration (days, log)'); ax.set_ylabel('Number of SKUs'); panel_label(ax, 'a', x=-0.25)
    ax = axes[1]
    for k, col in [('Food', OKABE[1]), ('Non-food', OKABE[0])]:
        h, _ = hz[k]; dd_ = np.arange(1, len(h) + 1)
        sm = pd.Series(h).rolling(5, center=True, min_periods=1).mean()
        ax.plot(dd_, 100 * h, color=col, lw=0.4, alpha=0.4); ax.plot(dd_, 100 * sm, color=col, lw=1.2, label=k)
    ax.set_xlabel('Days since last price change'); ax.set_ylabel('Hazard of a change (%)'); ax.legend(); ax.set_xlim(1, 120)
    ax.set_yscale('log'); ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:g}'))
    panel_label(ax, 'b', x=-0.25)
    ax = axes[2]
    ct = ptab[ptab.Category != 'All SKUs']
    ax.scatter(ct['Freq. of change (%/day)'], (ct['Mean increase (%)'] + ct['Mean decrease (%)']) / 2, s=ct['SKUs'] / 6,
               c=[OKABE[1] if x == 'Yes' else OKABE[0] for x in ct['Food']], alpha=0.75, edgecolor='white', lw=0.4)
    for _, r in ct.iterrows():
        if True:
            ax.annotate(r['Category'].split(' &')[0].split(',')[0], (r['Freq. of change (%/day)'], (r['Mean increase (%)'] + r['Mean decrease (%)']) / 2),
                        fontsize=4.8, xytext=(3, 2), textcoords='offset points')
    ax.set_xscale('log'); ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:g}'))
    ax.set_xlabel('Frequency of change (%/day, log)'); ax.set_ylabel('Mean |size| of change (%)'); panel_label(ax, 'c', x=-0.25)
    fig.tight_layout()
    savefig(fig, 'F07_price_stickiness', 'Price stickiness. (a) SKU-level implied price durations; (b) Kaplan–Meier hazard of a price change by spell age (thin: raw, thick: 5-day smoothed); (c) frequency versus average size of price changes by category (bubble area ∝ SKUs; orange = food).')

    fig = plt.figure(figsize=(W2, 4.3)); gsp = fig.add_gridspec(2, 3, height_ratios=[1.1, 1], hspace=0.6, wspace=0.45)
    ax = fig.add_subplot(gsp[0, :]); shade_gaps(ax); shade_events(ax)
    ax.plot(S.index, 100 * S, color=OKABE[0], lw=0.4, alpha=0.6, label='Daily')
    ax.plot(S.index, 100 * S.rolling(7, min_periods=3).mean(), color=OKABE[0], lw=1.2, label='7-day mean')
    ax.axhline(100 * top5, color=OKABE[1], ls=':', lw=0.8, label='95th percentile')
    ax.set_yscale('log'); ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:g}'))
    ax.set_ylabel('SKUs changing price (%)'); date_axis(ax); ax.legend(ncol=4, loc='lower left', fontsize=6)
    ax.set_title(f'Synchronisation of price changes (FK = {FK:.2f})'); panel_label(ax, 'a', x=-0.06, y=1.06)
    ax = fig.add_subplot(gsp[1, 0])
    g = dd.groupby('dow')['S'].agg(['mean', 'sem'])
    ax.bar(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'], 100 * g['mean'], yerr=196 * g['sem'], color=OKABE[0], capsize=1.5, error_kw={'lw': 0.6})
    ax.set_ylabel('SKUs changing (%)'); ax.set_title(f'Day of week (p={kw_dow.pvalue:.3f})', fontweight='normal', fontsize=7); ax.tick_params(axis='x', labelsize=6)
    ax.grid(axis='x', visible=False); panel_label(ax, 'b', x=-0.3)
    ax = fig.add_subplot(gsp[1, 1])
    g = dd.groupby('month')['S'].agg(['mean', 'sem'])
    ax.bar(g.index, 100 * g['mean'], yerr=196 * g['sem'], color=OKABE[2], capsize=1.5, error_kw={'lw': 0.6})
    ax.set_xticks(range(1, 13)); ax.set_xticklabels('JFMAMJJASOND'); ax.set_title(f'Month (p={kw_mon.pvalue:.3f})', fontweight='normal', fontsize=7)
    ax.grid(axis='x', visible=False); panel_label(ax, 'c', x=-0.3)
    ax = fig.add_subplot(gsp[1, 2])
    mm = pd.DataFrame({'up': up.sum(0), 'dn': dn.sum(0)}, index=dates).resample('MS').sum()
    mm = mm[(mm.up + mm.dn) > 0]
    shr = 100 * mm.up / (mm.up + mm.dn)
    ax.plot(mm.index, shr, color=OKABE[1], marker='o', ms=2, lw=0.9); ax.axhline(50, color=GREY, lw=0.6, ls='--')
    ax.set_ylabel('Increases (%)'); date_axis(ax, "%b\n'%y", interval=6); ax.set_title('Share of increases', fontweight='normal', fontsize=7)
    panel_label(ax, 'd', x=-0.3)
    savefig(fig, 'F08_synchronisation_seasonality', 'Timing of price changes. (a) Share of SKUs changing price per day (log scale; orange bands: Ramadan/Eid; grey: scrape gaps); (b) day-of-week and (c) month-of-year means with 95% CIs (Kruskal–Wallis p-values); (d) share of price changes that are increases, by month.')

    fig, axes = plt.subplots(1, 3, figsize=(W2, 2.3), sharey=False)
    for ax, (scope, (x, bk)), lab in zip(axes, brk.items(), 'abc'):
        shade_gaps(ax, label=False)
        ax.plot(x.index, x.values, color=OKABE[0], lw=0.7)
        seg = [0] + bk + [len(x)]
        for s0, s1 in zip(seg[:-1], seg[1:]):
            ax.hlines(x.iloc[s0:s1].mean(), x.index[s0], x.index[s1 - 1], color=OKABE[1], lw=1.4)
        for b in bk: ax.axvline(x.index[b], color=OKABE[1], ls=':', lw=0.7)
        ax.axhline(0, color=GREY, lw=0.4); ax.set_title(scope, fontweight='normal'); date_axis(ax, "%b\n'%y", interval=6)
        panel_label(ax, lab, x=-0.2)
    axes[0].set_ylabel('7-day inflation (%)')
    fig.tight_layout()
    savefig(fig, 'F09_structural_breaks', 'PELT change points (dotted) and segment means (orange) of 7-day DPI inflation for (a) all items, (b) food and (c) non-food (dataset B). Grey: scrape gaps.')

    save_ck(prod, '08_product_dynamics')
    save_ck(pd.DataFrame({'date': S.index, 'sync': S.values}), '08_sync')
    save_ck({'FK': FK, 'fbar': fbar, 'share_changes_top5pct_days': share_changes_top5,
             'hazard': {k: [list(map(float, v[0])), list(map(float, v[1]))] for k, v in hz.items()},
             'breaks': brtab.to_dict('records'), 'n_spells': int(len(dur)), 'censored_share': float(1 - ev.mean())}, '08_dynamics_summary')
    show(ptab, 25); show(brtab)

run_module('m08', m08_dynamics, ['08_product_dynamics', '08_sync', '08_dynamics_summary'])

## Module 9 · Heterogeneous price behaviour: product regime clustering
Feature vector per SKU (observed data only):
$X_i=[\text{logit } f_i,\ \text{share up}_i,\ \ln|\text{size}|_i,\ \ln\sigma_i,\ CV_i,\ \text{trend}_i,\ \text{seasonality}_i,\ \text{shock}_i,\ \text{batch share}_i]$,
where *trend* is the annualised log price change, *seasonality* the share of detrended variance explained by month,
*shock* the largest absolute daily change and *batch share* the share of a SKU's changes that fall on the 5% most
synchronised days. SKUs that never change price form a separate **fixed-price** regime. K-means (k chosen by
silhouette), Gaussian mixtures (BIC) and HDBSCAN are compared; stability is assessed by bootstrap adjusted Rand index
and economic meaning by association with categories (Cramér's V) and by regime-level price indices.

In [ ]:
def cramers_v(x, y):
    ct = pd.crosstab(x, y); chi2 = stats.chi2_contingency(ct)[0]; n = ct.values.sum()
    r, k = ct.shape
    return float(np.sqrt(chi2 / (n * (min(r, k) - 1))))

def m09_clusters():
    from sklearn.preprocessing import StandardScaler
    from sklearn.cluster import KMeans, HDBSCAN
    from sklearn.mixture import GaussianMixture
    from sklearn.decomposition import PCA
    from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score, adjusted_rand_score
    meta = load_ck('02_meta'); prod = load_ck('08_product_dynamics'); A = wide_from_ck('06_dataset_A'); B = wide_from_ck('06_dataset_B')
    sync = load_ck('08_sync').set_index('date')['sync']
    dates = A.columns; LA = np.log(A.values)
    D = transitions(A); V = ~np.isnan(D); C = (np.abs(np.nan_to_num(D)) > EPS) & V
    yrs = (dates - dates[0]).days.values / 365.25
    obs_cols = ~np.isnan(LA).all(0)
    X = yrs[obs_cols]; Yl = LA[:, obs_cols]
    slope = ((Yl - Yl.mean(1, keepdims=True)) * (X - X.mean())).sum(1) / ((X - X.mean()) ** 2).sum()
    # monthly seasonality share: variance of month means of detrended log price / total detrended variance
    resid = Yl - (Yl.mean(1, keepdims=True) + slope[:, None] * (X - X.mean()))
    mon = pd.DatetimeIndex(dates[obs_cols]).month.values
    mm = np.stack([resid[:, mon == m].mean(1) for m in np.unique(mon)], 1)
    seas = np.nan_to_num(mm.var(1) / resid.var(1).clip(min=1e-12))
    hi_days = (sync >= sync.quantile(0.95)).reindex(dates).fillna(False).values
    batch = np.where(C.sum(1) > 0, C[:, hi_days].sum(1) / C.sum(1).clip(min=1), np.nan)
    F = pd.DataFrame({'product_id': prod['product_id'], 'freq': prod['freq'], 'share_up': prod['share_up'], 'abs_size': prod['abs_size'],
                      'vol': prod['vol'], 'cv_level': prod['cv_level'], 'trend': 100 * slope, 'seasonality': seas,
                      'shock': prod['max_abs_change'], 'batch_share': batch})
    fixed = F['freq'] <= 0
    Z = F.loc[~fixed].copy()
    T = pd.DataFrame({'logit_freq': np.log(Z['freq'] / (1 - Z['freq'])), 'share_up': Z['share_up'], 'log_size': np.log(Z['abs_size'].clip(lower=0.1)),
                      'log_vol': np.log(Z['vol'].clip(lower=1e-3)), 'cv_level': np.log(Z['cv_level'].clip(lower=1e-4)),
                      'trend': Z['trend'].clip(-60, 60), 'seasonality': Z['seasonality'], 'log_shock': np.log(Z['shock'].clip(lower=0.1)),
                      'batch_share': Z['batch_share']}).fillna(0)
    Xs = StandardScaler().fit_transform(T.values)
    rng = np.random.default_rng(CFG['seed'])
    samp = rng.choice(len(Xs), min(4000, len(Xs)), replace=False)
    rows = []
    for k in CFG['k_range']:
        km = KMeans(k, n_init=20, random_state=CFG['seed']).fit(Xs)
        gm = GaussianMixture(k, covariance_type='full', random_state=CFG['seed'], n_init=3).fit(Xs)
        rows.append({'k': k, 'Silhouette': silhouette_score(Xs[samp], km.labels_[samp]), 'Calinski–Harabasz': calinski_harabasz_score(Xs, km.labels_),
                     'Davies–Bouldin': davies_bouldin_score(Xs, km.labels_), 'GMM BIC': gm.bic(Xs)})
    sel = pd.DataFrame(rows)
    k_best = int(sel.loc[sel['Silhouette'].idxmax(), 'k']); k_bic = int(sel.loc[sel['GMM BIC'].idxmin(), 'k'])
    km = KMeans(k_best, n_init=50, random_state=CFG['seed']).fit(Xs); lab = km.labels_
    gm = GaussianMixture(k_best, covariance_type='full', random_state=CFG['seed'], n_init=3).fit(Xs)
    hd = HDBSCAN(min_cluster_size=max(30, len(Xs) // 100)).fit(Xs)
    # bootstrap stability
    aris = []
    for bsd in range(20):
        r2 = np.random.default_rng(bsd); ix = r2.choice(len(Xs), int(0.8 * len(Xs)), replace=False)
        kb = KMeans(k_best, n_init=10, random_state=bsd).fit(Xs[ix])
        aris.append(adjusted_rand_score(lab[ix], kb.labels_))
    # naming
    prof = Z.assign(cluster=lab).groupby('cluster').median(numeric_only=True)
    fr_rank = prof['freq'].rank(); vol_rank = prof['vol'].rank()
    names = {}
    for c, r in prof.iterrows():
        a1 = 'Frequent' if fr_rank[c] > len(prof) - max(1, len(prof) // 3) else ('Sticky' if fr_rank[c] <= max(1, len(prof) // 3) else 'Moderate')
        a2 = 'rising' if r['trend'] > 2 else ('falling' if r['trend'] < -2 else 'stable')
        a3 = ', volatile' if vol_rank[c] == len(prof) else (', batch-driven' if r['batch_share'] == prof['batch_share'].max() else '')
        names[c] = f'R{c + 1}: {a1}, {a2}{a3}'
    regime = pd.Series('R0: No observed change', index=F.index)
    regime.loc[~fixed] = [names[l] for l in lab]
    F['regime'] = regime.values; F['category'] = meta['category'].values; F['is_food'] = meta['is_food'].values
    F['gmm'] = -1; F.loc[~fixed, 'gmm'] = gm.predict(Xs); F['hdbscan'] = -2; F.loc[~fixed, 'hdbscan'] = hd.labels_
    agree = {'ARI KMeans–GMM': adjusted_rand_score(lab, gm.predict(Xs)),
             'ARI KMeans–HDBSCAN (non-noise)': adjusted_rand_score(lab[hd.labels_ >= 0], hd.labels_[hd.labels_ >= 0]) if (hd.labels_ >= 0).sum() > 10 else np.nan,
             'HDBSCAN clusters': int(hd.labels_.max() + 1), 'HDBSCAN noise share': float((hd.labels_ < 0).mean()),
             'Bootstrap ARI mean': float(np.mean(aris)), 'Bootstrap ARI sd': float(np.std(aris)),
             "Cramér's V (regime × category)": cramers_v(F['regime'], F['category']),
             "Cramér's V (regime × food)": cramers_v(F['regime'], F['is_food'])}
    # regime-level index (dataset B) & inflation
    LB = np.log(B.values); rix = {}
    for rname in sorted(F['regime'].unique()):
        m = (F['regime'] == rname).values
        rix[rname] = 100 * np.exp(np.nanmean(LB[m] - LB[m][:, [0]], axis=0))
    kw = stats.kruskal(*[g['trend'].values for _, g in F.groupby('regime')])
    # tables
    sel_t = sel.copy(); sel_t['Chosen'] = np.where(sel_t['k'] == k_best, 'Silhouette', '') + np.where(sel_t['k'] == k_bic, ' BIC', '')
    save_table(sel_t, 'T16_cluster_selection', 'Selection of the number of price-behaviour regimes.', float_fmt='{:.3f}',
               notes=f'K-means on standardised features of {len(Xs):,} SKUs with at least one price change; {int(fixed.sum()):,} SKUs with no price change on consecutive observed days form regime R0.')
    rp = F.groupby('regime').agg(SKUs=('product_id', 'size'), Food_share=('is_food', 'mean'), Freq=('freq', 'median'), Up=('share_up', 'median'),
                                 Size=('abs_size', 'median'), Vol=('vol', 'median'), Trend=('trend', 'median'), Seas=('seasonality', 'median'),
                                 Batch=('batch_share', 'median')).reset_index()
    rp['Food_share'] *= 100; rp['Freq'] *= 100; rp['Up'] *= 100; rp['Batch'] *= 100
    top = F.groupby('regime')['category'].agg(lambda s: ', '.join(s.value_counts().index[:2]))
    rp['Main categories'] = rp['regime'].map(top)
    rp['Regime index at end'] = rp['regime'].map({k: v[-1] for k, v in rix.items()})
    rp = rp.rename(columns={'regime': 'Regime', 'Food_share': 'Food (%)', 'Freq': 'Freq. (%/day)', 'Up': 'Increases (%)', 'Size': '|Size| (%)',
                            'Vol': 'Volatility ×100', 'Trend': 'Trend (% p.a.)', 'Seas': 'Seasonality share', 'Batch': 'Batch share (%)'})
    save_table(rp, 'T17_regime_profiles', 'Price-behaviour regimes: median characteristics.', float_fmt='{:.2f}',
               notes=f"Kruskal–Wallis test of equal trend across regimes: H = {kw.statistic:.1f}, p = {kw.pvalue:.2g}. Regime index: Jevons index of member SKUs (dataset B, start = 100).")
    ag = pd.DataFrame({'Diagnostic': list(agree), 'Value': list(agree.values())})
    save_table(ag, 'T18_regime_validity', 'Robustness and economic validity of the regime classification.', float_fmt='{:.3f}')

    # figure
    pca = PCA(2, random_state=0).fit(Xs); P2 = pca.transform(Xs)
    rnames = sorted(set(names.values()))
    cmap = {n: OKABE[i % len(OKABE)] for i, n in enumerate(sorted(rix))}
    fig = plt.figure(figsize=(W2, 5.6)); gsp = fig.add_gridspec(2, 2, height_ratios=[1, 1], hspace=0.42, wspace=0.35)
    ax = fig.add_subplot(gsp[0, 0])
    for c in range(k_best):
        m = lab == c
        ax.scatter(P2[m, 0], P2[m, 1], s=2, color=cmap[names[c]], alpha=0.5, lw=0, label=names[c].split(':')[0])
    ax.set_xlabel(f'PC1 ({100*pca.explained_variance_ratio_[0]:.0f}%)'); ax.set_ylabel(f'PC2 ({100*pca.explained_variance_ratio_[1]:.0f}%)')
    ax.legend(markerscale=4, loc='best', fontsize=6); panel_label(ax, 'a', x=-0.18)
    ax = fig.add_subplot(gsp[0, 1])
    cen = pd.DataFrame(km.cluster_centers_, columns=T.columns, index=[names[c].split(':')[0] for c in range(k_best)])
    im = ax.imshow(cen.T.values, cmap='RdBu_r', vmin=-2, vmax=2, aspect='auto')
    ax.set_yticks(range(len(T.columns))); ax.set_yticklabels(['logit freq', 'share up', 'ln size', 'ln vol', 'ln CV', 'trend', 'seasonality', 'ln shock', 'batch share'], fontsize=6)
    ax.set_xticks(range(k_best)); ax.set_xticklabels(cen.index, fontsize=6.5); ax.grid(False)
    for (j, i), v in np.ndenumerate(cen.T.values): ax.text(i, j, f'{v:.1f}', ha='center', va='center', fontsize=5, color='white' if abs(v) > 1.2 else 'black')
    cb = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02); cb.set_label('Standardised centroid', fontsize=6.5); cb.ax.tick_params(labelsize=6)
    panel_label(ax, 'b', x=-0.3)
    ax = fig.add_subplot(gsp[1, :]); shade_gaps(ax, label=False)
    for rname, v in rix.items():
        ax.plot(B.columns, v, color=cmap.get(rname, 'black'), lw=1.1, label=f"{rname} (n={int((F.regime == rname).sum())})")
    ax.axhline(100, color=GREY, lw=0.4); ax.set_ylabel('Regime Jevons index'); date_axis(ax)
    ax.legend(fontsize=6, ncol=3, loc='upper center', bbox_to_anchor=(0.5, -0.2))
    panel_label(ax, 'c', x=-0.07)
    savefig(fig, 'F10_price_regimes', f'Price-behaviour regimes. (a) SKUs in the first two principal components of the feature space; (b) standardised k-means centroids (k = {k_best}); (c) Jevons index of each regime (dataset B), including regime R0 (no observed change; its index moves only across scrape gaps).')
    save_ck(F, '09_regimes'); save_ck({'k': k_best, 'k_bic': k_bic, 'names': {int(k): v for k, v in names.items()}, **agree}, '09_cluster_summary')
    show(rp)

run_module('m09', m09_clusters, ['09_regimes', '09_cluster_summary'])

## Module 10 · Inflation dynamics
* $\pi^{d}_t=100\,\Delta\ln DPI_t$, $\pi^{MoM}_t=100[\ln DPI_t-\ln DPI_{t-30}]$, $\pi^{YoY}_t=100[\ln DPI_t-\ln DPI_{t-365}]$;
  monthly versions use the monthly average index (comparable with official CPI).
* **Category contributions** (exact under geometric aggregation): $\pi^{YoY}_{t}=\sum_c w_c\,\pi^{YoY}_{c,t}$.
* **Persistence**: AR($p$) on weekly inflation, $p$ by BIC; persistence $\rho=\sum_j\phi_j$ and half-life $\ln 0.5/\ln\rho$.
* **Shock propagation** across food categories: pairwise Granger causality on weekly category inflation estimated on a
  *complete-case* sample that drops every regression row touching a mostly reconstructed week (avoids spurious
  co-movement manufactured by imputation), Benjamini–Hochberg FDR control, and a VAR with orthogonalised impulse responses.

In [ ]:
def granger_cc(y, x, valid, p=2):
    """Granger F-test of x→y with lags 1..p; rows dropped if any of y_t, y_{t-j}, x_{t-j} is in an invalid week."""
    n = len(y); rows = [t for t in range(p, n) if valid[t - p:t + 1].all()]
    if len(rows) < 3 * p + 10: return np.nan, np.nan, len(rows)
    Yv = np.array([y[t] for t in rows])
    Xr = np.array([[1] + [y[t - j] for j in range(1, p + 1)] for t in rows])
    Xu = np.array([[1] + [y[t - j] for j in range(1, p + 1)] + [x[t - j] for j in range(1, p + 1)] for t in rows])
    rss = lambda X: np.sum((Yv - X @ np.linalg.lstsq(X, Yv, rcond=None)[0]) ** 2)
    rr, ru = rss(Xr), rss(Xu); df2 = len(rows) - Xu.shape[1]
    F = ((rr - ru) / p) / (ru / df2)
    return float(F), float(1 - stats.f.cdf(F, p, df2)), len(rows)

def ar_persistence(x, pmax=8):
    import statsmodels.api as sm
    x = np.asarray(x); best = None
    for p in range(1, pmax + 1):
        Y = x[p:]; X = sm.add_constant(np.column_stack([x[p - j:len(x) - j] for j in range(1, p + 1)]))
        r = sm.OLS(Y, X).fit()
        if best is None or r.bic < best[1].bic: best = (p, r)
    p, r = best; rho = float(r.params[1:].sum())
    return p, rho, (np.log(0.5) / np.log(rho) if 0 < rho < 1 else np.nan)

def m10_inflation():
    import statsmodels.api as sm
    from statsmodels.tsa.api import VAR
    meta = load_ck('02_meta'); idx = load_ck('07_indices'); gaps = load_ck('03_gap_inventory')
    calendar = load_ck('01_calendar'); observed = calendar.set_index('date')['observed'].astype(bool); observed.index = pd.to_datetime(observed.index)
    def get(ds='B', scope='All items', f='Jevons', w='sku'):
        s = idx[(idx.dataset == ds) & (idx.scope == scope) & (idx.formula == f) & (idx.weights == w)].set_index('date')['index']
        s.index = pd.to_datetime(s.index); return s
    dates = get().index
    infl = {}
    for scope in ['All items', 'Food', 'Non-food']:
        s = get(scope=scope); ls = np.log(s)
        mon = s.resample('MS').mean(); lm = np.log(mon)
        infl[scope] = {'daily': 100 * ls.diff(), 'mom30': 100 * (ls - ls.shift(30)), 'yoy': 100 * (ls - ls.shift(365)),
                       'm_mom': 100 * lm.diff(), 'm_yoy': 100 * (lm - lm.shift(12)), 'weekly': 100 * np.log(s.resample('W').mean()).diff()}
    # monthly table export (for macro module)
    mt = pd.DataFrame({f'{sc}_{k}': infl[sc][k] for sc in infl for k in ['m_mom', 'm_yoy']})
    mt.index.name = 'month'
    imp_share_m = (~observed).resample('MS').mean()
    mt['imputed_day_share'] = imp_share_m.reindex(mt.index).values
    # persistence & summary
    rows = []
    wk_imp = (~observed).resample('W').mean()
    for sc in infl:
        w = infl[sc]['weekly'].dropna()
        p, rho, hl = ar_persistence(w.values)
        wv = w[wk_imp.reindex(w.index).fillna(1) < 0.5]
        p2, rho2, hl2 = ar_persistence(wv.values) if len(wv) > 30 else (np.nan, np.nan, np.nan)
        y = infl[sc]['yoy'].dropna()
        rows.append({'Scope': sc, 'Mean monthly MoM (%)': infl[sc]['m_mom'].mean(), 'SD monthly MoM (%)': infl[sc]['m_mom'].std(),
                     'YoY latest (%)': y.iloc[-1], 'YoY max (%)': y.max(), 'YoY min (%)': y.min(),
                     'AR lag (BIC)': p, 'Persistence ρ': rho, 'Half-life (weeks)': hl, 'ρ (observed weeks)': rho2,
                     'Skewness weekly': stats.skew(w), 'Share weeks >0 (%)': 100 * (w > 0).mean()})
    st = pd.DataFrame(rows)
    save_table(st, 'T19_inflation_summary', 'Digital inflation: level, volatility, persistence and asymmetry (dataset B, Jevons).', float_fmt='{:.3f}',
               notes='Weekly inflation = 100·Δln of weekly mean index. Persistence ρ = sum of AR coefficients; "observed weeks" re-estimates ρ on weeks with <50% reconstructed days (rows are contiguous fragments concatenated; indicative only).')

    # contributions to food YoY (exact geometric decomposition, SKU-share weights)
    food_cats = [c for c in meta.loc[meta.is_food == 1, 'category'].value_counts().index]
    wts = meta.loc[meta.is_food == 1, 'category'].value_counts(normalize=True)
    contrib = pd.DataFrame({c: wts[c] * 100 * (np.log(get(scope=c)) - np.log(get(scope=c)).shift(365)) for c in food_cats})
    cm = contrib.resample('MS').mean().dropna(how='all')
    total = cm.sum(1)
    topc = cm.abs().mean().sort_values(ascending=False).index[:7].tolist()
    cplot = cm[topc].copy(); cplot['Other food'] = cm.drop(columns=topc).sum(1)
    ctab = pd.DataFrame({'Category': cm.columns, 'Weight (%)': 100 * wts[cm.columns].values,
                         'Mean contribution to YoY (pp)': cm.mean().values,
                         'Share of mean food YoY (%)': 100 * cm.mean().values / total.mean()}).sort_values('Mean contribution to YoY (pp)', ascending=False)
    save_table(ctab, 'T20_food_contributions', 'Category contributions to year-on-year food DPI inflation (dataset B).', float_fmt='{:.3f}',
               notes='Contribution_c = w_c·100·[ln I_c,t − ln I_c,t−365], exact for the geometric (Jevons) upper level; averaged over months with YoY available.')

    # shock propagation among food categories (weekly)
    big = [c for c in food_cats if (meta.category == c).sum() >= 40][:10]
    Wk = pd.DataFrame({c: 100 * np.log(get(scope=c).resample('W').mean()).diff() for c in big}).dropna()
    valid = (wk_imp.reindex(Wk.index).fillna(1) < 0.5).values
    gr = []
    for cx in big:
        for cy in big:
            if cx == cy: continue
            F_, p_, n_ = granger_cc(Wk[cy].values, Wk[cx].values, valid, p=2)
            gr.append({'Cause': cx, 'Effect': cy, 'F': F_, 'p-value': p_, 'N': n_})
    gr = pd.DataFrame(gr); gr['q (BH)'] = bh_fdr(gr['p-value'].values); gr['Sig.'] = [stars(q) for q in gr['q (BH)']]
    save_table(gr.sort_values('p-value').head(25), 'T21_granger_network', 'Granger causality between weekly food-category inflation rates (complete-case, 2 lags).',
               float_fmt='{:.3f}', notes='Rows of the test regression are dropped whenever the target week or any lag week has ≥50% reconstructed days. Top 25 pairs by p-value; q = Benjamini–Hochberg.')
    # VAR on most important categories (contiguous weekly B; caveat: includes reconstructed weeks)
    vc = [c for c in ['Edible oil & ghee', 'Rice', 'Meat & fish', 'Vegetables', 'Dairy & eggs', 'Spices & condiments'] if c in Wk.columns][:5]
    irf_data = None
    try:
        vm = VAR(Wk[vc]).fit(maxlags=4, ic='bic')
        if vm.k_ar == 0:                                   # BIC selects no dynamics -> report VAR(1) for the IRF
            vm = VAR(Wk[vc]).fit(1)
        irf = vm.irf(12)
        irf_data = {'vars': vc, 'irfs': irf.orth_irfs.tolist(), 'lag': int(vm.k_ar)}
    except Exception as e:
        print('VAR failed:', e)

    # ---------------- figures ----------------
    fig, axes = plt.subplots(1, 2, figsize=(W2, 2.7), gridspec_kw={'width_ratios': [1.3, 1]})
    ax = axes[0]; shade_gaps(ax)
    for sc, col in [('All items', OKABE[0]), ('Food', OKABE[1]), ('Non-food', OKABE[2])]:
        ax.plot(dates, infl[sc]['yoy'], color=col, lw=1.1, label=sc)
    ax.axhline(0, color=GREY, lw=0.5); ax.set_ylabel('Year-on-year inflation (%)'); date_axis(ax)
    ax.set_xlim(dates[0] + pd.Timedelta(days=360), dates[-1]); ax.legend(loc='lower left', fontsize=6.3); panel_label(ax, 'a', x=-0.12)
    ax = axes[1]
    mm = infl['Food']['m_mom'].dropna()
    imp = imp_share_m.reindex(mm.index).fillna(0)
    ax.bar(mm.index, mm.values, width=20, color=[OKABE[1] if v >= 0 else OKABE[0] for v in mm.values])
    for d_, v_, s_ in zip(mm.index, mm.values, imp.values):
        if s_ > 0.5: ax.bar(d_, v_, width=20, facecolor='white', hatch='////', edgecolor=GREY, lw=0.6)
    ax.axhline(0, color=GREY, lw=0.5); ax.set_ylabel('Food MoM inflation (%)'); date_axis(ax, "%b\n'%y", interval=6)
    ax.legend(handles=[matplotlib.patches.Patch(facecolor='white', hatch='////', edgecolor=GREY, label='>50% reconstructed days')], loc='upper right', fontsize=6.3)
    panel_label(ax, 'b', x=-0.18)
    fig.tight_layout()
    savefig(fig, 'F11_inflation', 'Digital inflation. (a) Year-on-year DPI inflation (daily, dataset B; grey bands: scrape gaps); (b) monthly month-on-month food inflation (hatched: months dominated by reconstructed days).')

    fig, ax = plt.subplots(figsize=(W2, 2.9))
    pos = np.zeros(len(cplot)); neg = np.zeros(len(cplot))
    cols = OKABE[:7] + [LIGHT]
    for c, col in zip(cplot.columns, cols):
        v = cplot[c].fillna(0).values
        ax.bar(cplot.index, np.where(v > 0, v, 0), bottom=pos, width=22, color=col, label=c, lw=0)
        ax.bar(cplot.index, np.where(v < 0, v, 0), bottom=neg, width=22, color=col, lw=0)
        pos += np.where(v > 0, v, 0); neg += np.where(v < 0, v, 0)
    ax.plot(total.index, total.values, color='black', marker='o', ms=2.5, lw=1.0, label='Food YoY (total)')
    ax.axhline(0, color=GREY, lw=0.5); ax.set_ylabel('Contribution to food YoY (pp)'); date_axis(ax, "%b\n%Y", interval=2)
    ax.legend(ncol=5, fontsize=6, loc='lower center', bbox_to_anchor=(0.5, 1.0))
    savefig(fig, 'F12_food_contributions', 'Category contributions to year-on-year food DPI inflation (monthly averages; exact geometric decomposition, dataset B).')

    if irf_data:
        vc = irf_data['vars']; ir = np.array(irf_data['irfs'])
        k = len(vc)
        fig, axes = plt.subplots(1, 2, figsize=(W2, 2.9), gridspec_kw={'width_ratios': [1, 1.1]})
        ax = axes[0]
        M = np.full((len(big), len(big)), np.nan)
        for _, r in gr.iterrows():
            M[big.index(r['Effect']), big.index(r['Cause'])] = -np.log10(max(r['q (BH)'], 1e-6))
        im = ax.imshow(M, cmap='Blues', vmin=0, vmax=3)
        short = [c.split(' &')[0].split(',')[0] for c in big]
        ax.set_xticks(range(len(big))); ax.set_xticklabels(short, rotation=55, ha='right', fontsize=6)
        ax.set_yticks(range(len(big))); ax.set_yticklabels(short, fontsize=6); ax.grid(False)
        ax.set_xlabel('Cause'); ax.set_ylabel('Effect')
        for (i, j), v in np.ndenumerate(M):
            if np.isfinite(v) and v > -np.log10(0.10): ax.text(j, i, '•', ha='center', va='center', color='white' if v > 1.5 else 'black', fontsize=7)
        cb = fig.colorbar(im, ax=ax, fraction=0.045, pad=0.02); cb.set_label('−log₁₀ q (BH)', fontsize=6.5); cb.ax.tick_params(labelsize=6)
        panel_label(ax, 'a', x=-0.45)
        ax = axes[1]
        src = 0
        for j in range(k):
            ax.plot(range(ir.shape[0]), ir[:, j, src], color=OKABE[j % 8], lw=1.1, marker='o', ms=2, label=vc[j].split(' &')[0])
        ax.axhline(0, color=GREY, lw=0.5); ax.set_xlabel('Weeks after shock'); ax.set_ylabel('Response (pp weekly inflation)')
        ax.set_title(f'Orthogonalised IRF to a {vc[src].split(" &")[0].lower()} shock', fontweight='normal'); ax.legend(fontsize=6)
        panel_label(ax, 'b', x=-0.18)
        fig.tight_layout()
        savefig(fig, 'F13_shock_propagation', f'Shock propagation among food categories. (a) Complete-case Granger causality network (weekly, 2 lags; • q<0.10); (b) orthogonalised impulse responses to a one-s.d. {vc[0].lower()} inflation shock from a VAR({irf_data["lag"]}) (Cholesky order as listed).')

    save_ck(mt.reset_index(), '10_monthly_inflation')
    save_ck(pd.DataFrame({f'{sc}_{k}': infl[sc][k] for sc in infl for k in ['daily', 'mom30', 'yoy']}).reset_index().rename(columns={'index': 'date'}), '10_daily_inflation')
    save_ck({'summary': st.to_dict('records'), 'granger': gr.to_dict('records'), 'irf': irf_data, 'contrib': ctab.to_dict('records')}, '10_inflation_summary')
    show(st); show(ctab)

run_module('m10', m10_inflation, ['10_monthly_inflation', '10_daily_inflation', '10_inflation_summary'])

## Module 11 · Official CPI/FPI linkage, leading-indicator and nowcasting tests
Requires `data/bbs_cpi_monthly.csv` (a template is written on first run). Columns:
`month` (YYYY-MM), `cpi_general_index`, `cpi_food_index`, optional `cpi_nonfood_index`, `exchange_rate_bdt_usd`,
`fuel_price_index` – fill from the BBS monthly CPI release and Bangladesh Bank. **No values are fabricated**: if the
file is missing or empty the module writes the template and skips.

Analyses: (i) contemporaneous agreement (Pearson/Spearman, RMSE/MAE of YoY, turning-point accuracy);
(ii) **lead–lag**: $\operatorname{Corr}(\overline{DPI}^{(k)}_m, CPI_m)$ where $\overline{DPI}^{(k)}_m$ is the monthly mean of the
daily DPI shifted by $k=0,\dots,90$ days; (iii) complete-case Granger causality DPI→CPI;
(iv) **nowcasting** CPI food MoM with an expanding-window bridge equation (AR(1) + current-month DPI) vs AR(1),
compared by RMSE ratio and the Diebold–Mariano test; (v) correlation with the exchange rate at monthly lags.

In [ ]:
CPI_COLS = ['month', 'cpi_general_index', 'cpi_food_index', 'cpi_nonfood_index', 'exchange_rate_bdt_usd', 'fuel_price_index']

def load_cpi():
    p = Path(CFG['cpi_csv'])
    if not p.exists():
        months = pd.date_range('2023-01-01', '2026-12-01', freq='MS').strftime('%Y-%m')
        pd.DataFrame({'month': months, **{c: np.nan for c in CPI_COLS[1:]}}).to_csv(p, index=False)
        print(f'Template written to {p}. Fill it with official BBS CPI (and Bangladesh Bank FX) data, then re-run Module 11 with FORCE_RERUN={{"m11"}}.')
        return None
    df = pd.read_csv(p)
    df['month'] = pd.to_datetime(df['month'].astype(str).str[:7])
    df = df.set_index('month').sort_index()
    if df[['cpi_food_index', 'cpi_general_index']].notna().sum().min() < 13:
        print('CPI file has <13 non-missing months – module skipped.'); return None
    return df

def m11_macro():
    cpi = load_cpi()
    if cpi is None:
        save_ck({'status': 'skipped – no CPI data'}, '11_macro_summary'); return
    idx = load_ck('07_indices'); calendar = load_ck('01_calendar')
    observed = calendar.set_index('date')['observed'].astype(bool); observed.index = pd.to_datetime(observed.index)
    imp_m = (~observed).resample('MS').mean()
    def get(scope, ds='B'):
        s = idx[(idx.dataset == ds) & (idx.scope == scope) & (idx.formula == 'Jevons') & (idx.weights == 'sku')].set_index('date')['index']
        s.index = pd.to_datetime(s.index); return s
    out_rows, lead_curves, now = [], {}, {}
    pairs = [('Food', 'cpi_food_index'), ('All items', 'cpi_general_index')] + ([('Non-food', 'cpi_nonfood_index')] if cpi['cpi_nonfood_index'].notna().sum() > 12 else [])
    for scope, col in pairs:
        d = get(scope); dm = d.resample('MS').mean()
        c = cpi[col].dropna()
        M = pd.DataFrame({'dpi': dm, 'cpi': c}).dropna()
        M['dpi_mom'] = 100 * np.log(M.dpi).diff(); M['cpi_mom'] = 100 * np.log(M.cpi).diff()
        M['dpi_yoy'] = 100 * np.log(M.dpi / M.dpi.shift(12)); M['cpi_yoy'] = 100 * np.log(M.cpi / M.cpi.shift(12))
        M['imp'] = imp_m.reindex(M.index).fillna(1)
        for var in ['mom', 'yoy']:
            sub = M[[f'dpi_{var}', f'cpi_{var}', 'imp']].dropna()
            if len(sub) < 6: continue
            pr = stats.pearsonr(sub[f'dpi_{var}'], sub[f'cpi_{var}']); sr = stats.spearmanr(sub[f'dpi_{var}'], sub[f'cpi_{var}'])
            so = sub[sub.imp < 0.5]
            pro = stats.pearsonr(so[f'dpi_{var}'], so[f'cpi_{var}']) if len(so) > 5 else (np.nan, np.nan)
            dd = np.sign(sub[f'dpi_{var}'].diff()) == np.sign(sub[f'cpi_{var}'].diff())
            out_rows.append({'Scope': scope, 'Measure': var.upper(), 'N months': len(sub), 'Pearson r': pr[0], 'p (Pearson)': pr[1],
                             'Spearman ρ': sr[0], 'p (Spearman)': sr[1], 'Pearson r (observed months)': pro[0],
                             'RMSE (pp)': m_rmse(sub[f'cpi_{var}'].values, sub[f'dpi_{var}'].values), 'MAE (pp)': m_mae(sub[f'cpi_{var}'].values, sub[f'dpi_{var}'].values),
                             'Direction agreement (%)': 100 * dd.iloc[1:].mean()})
        # lead curve (daily shift k)
        ks = list(range(0, 91, 3)); cur = []
        for k in ks:
            dk = d.copy(); dk.index = dk.index + pd.Timedelta(days=k)
            mk = dk.resample('MS').mean()
            Mk = pd.DataFrame({'dpi': 100 * np.log(mk).diff(), 'cpi': M['cpi_mom']}).dropna()
            if len(Mk) > 6:
                r = np.corrcoef(Mk.dpi, Mk.cpi)[0, 1]; z = np.arctanh(r); se = 1 / np.sqrt(len(Mk) - 3)
                cur.append((k, r, np.tanh(z - 1.96 * se), np.tanh(z + 1.96 * se)))
        lead_curves[scope] = cur
        # Granger DPI -> CPI (complete case, monthly)
        valid = (M['imp'] < 0.5).values
        g1 = granger_cc(M['cpi_mom'].fillna(0).values, M['dpi_mom'].fillna(0).values, valid & M['cpi_mom'].notna().values, p=1)
        g2 = granger_cc(M['dpi_mom'].fillna(0).values, M['cpi_mom'].fillna(0).values, valid & M['cpi_mom'].notna().values, p=1)
        # nowcast (expanding window)
        Mn = M[['cpi_mom', 'dpi_mom']].dropna()
        e_ar, e_br, dts = [], [], []
        for t in range(10, len(Mn)):
            tr = Mn.iloc[:t]; y = tr['cpi_mom'].values[1:]
            Xa = np.column_stack([np.ones(len(y)), tr['cpi_mom'].values[:-1]])
            Xb = np.column_stack([Xa, tr['dpi_mom'].values[1:]])
            ba = np.linalg.lstsq(Xa, y, rcond=None)[0]; bb = np.linalg.lstsq(Xb, y, rcond=None)[0]
            prev = Mn['cpi_mom'].iloc[t - 1]; cur_d = Mn['dpi_mom'].iloc[t]; truth = Mn['cpi_mom'].iloc[t]
            e_ar.append(truth - (ba[0] + ba[1] * prev)); e_br.append(truth - (bb[0] + bb[1] * prev + bb[2] * cur_d)); dts.append(Mn.index[t])
        if len(e_ar) >= 5:
            dm_s, dm_p = dm_test(e_ar, e_br, h=1)
            now[scope] = {'dates': [str(x.date()) for x in dts], 'e_ar': e_ar, 'e_br': e_br, 'truth': Mn['cpi_mom'].iloc[10:].tolist()}
            out_rows.append({'Scope': scope, 'Measure': 'Nowcast', 'N months': len(e_ar), 'RMSE AR(1)': m_rmse(np.array(e_ar), 0),
                             'RMSE bridge': m_rmse(np.array(e_br), 0), 'RMSE ratio': m_rmse(np.array(e_br), 0) / m_rmse(np.array(e_ar), 0),
                             'DM stat': dm_s, 'DM p': dm_p, 'Granger DPI→CPI p': g1[1], 'Granger CPI→DPI p': g2[1]})
        if 'exchange_rate_bdt_usd' in cpi and cpi['exchange_rate_bdt_usd'].notna().sum() > 13:
            fx = 100 * np.log(cpi['exchange_rate_bdt_usd']).diff(12)
            for lag in range(0, 7):
                sub = pd.DataFrame({'d': M['dpi_yoy'], 'fx': fx.shift(lag)}).dropna()
                if len(sub) > 6:
                    out_rows.append({'Scope': scope, 'Measure': f'Corr(DPI YoY, FX YoY lag {lag}m)', 'N months': len(sub), 'Pearson r': np.corrcoef(sub.d, sub.fx)[0, 1]})
    res = pd.DataFrame(out_rows)
    agree = res[res.Measure.isin(['MOM', 'YOY'])].dropna(axis=1, how='all')
    save_table(agree, 'T22_dpi_vs_cpi', 'Agreement between the Digital Price Index and official BBS CPI (monthly).', float_fmt='{:.3f}',
               notes='DPI monthly values are averages of the daily index (dataset B). "Observed months": months with <50% reconstructed days.')
    nc = res[res.Measure.eq('Nowcast')].dropna(axis=1, how='all')
    if len(nc):
        save_table(nc, 'T23_nowcasting', 'Out-of-sample nowcasting of official month-on-month CPI inflation with the DPI.', float_fmt='{:.3f}',
                   notes='Expanding window; bridge = AR(1) + same-month DPI MoM (available before the official release). DM: Diebold–Mariano (HLN-corrected), positive = bridge more accurate. Granger tests are complete-case with 1 lag.')
    fx = res[res.Measure.str.startswith('Corr')].dropna(axis=1, how='all')
    if len(fx):
        save_table(fx, 'T24_exchange_rate', 'Correlation of DPI year-on-year inflation with BDT/USD depreciation at monthly lags.', float_fmt='{:.3f}')
    fig, axes = plt.subplots(1, 3, figsize=(W2, 2.5))
    ax = axes[0]
    d = get('Food'); dm = d.resample('MS').mean(); c = cpi['cpi_food_index'].dropna()
    ax.plot(dm.index, 100 * np.log(dm / dm.shift(12)), color=OKABE[1], marker='o', ms=2, label='Food DPI (online)')
    ax.plot(c.index, 100 * np.log(c / c.shift(12)), color='black', marker='s', ms=2, label='BBS food CPI')
    ax.set_xlim(dm.index[12], dm.index[-1]); ax.set_ylabel('YoY inflation (%)'); date_axis(ax, "%b\n'%y", interval=4); ax.legend(fontsize=6)
    panel_label(ax, 'a', x=-0.22)
    ax = axes[1]
    for (scope, cur), col in zip(lead_curves.items(), [OKABE[1], OKABE[0], OKABE[2]]):
        if not cur: continue
        k, r, lo, hi = map(np.array, zip(*cur))
        ax.plot(k, r, color=col, lw=1.2, label=scope); ax.fill_between(k, lo, hi, color=col, alpha=0.15, lw=0)
    ax.axhline(0, color=GREY, lw=0.5); ax.set_xlabel('Lead of DPI over CPI, k (days)'); ax.set_ylabel('Corr. of MoM inflation'); ax.legend(fontsize=6)
    panel_label(ax, 'b', x=-0.22)
    ax = axes[2]
    if 'Food' in now:
        nd = pd.to_datetime(now['Food']['dates']); tr_ = np.array(now['Food']['truth'])
        ax.plot(nd, tr_, color='black', marker='s', ms=2, label='Official')
        ax.plot(nd, tr_ - np.array(now['Food']['e_ar']), color=GREY, ls='--', label='AR(1)')
        ax.plot(nd, tr_ - np.array(now['Food']['e_br']), color=OKABE[1], label='AR(1) + DPI')
        ax.set_ylabel('Food CPI MoM (%)'); date_axis(ax, "%b\n'%y", interval=3); ax.legend(fontsize=6)
    panel_label(ax, 'c', x=-0.22)
    fig.tight_layout()
    savefig(fig, 'F14_dpi_vs_cpi', 'Online versus official inflation. (a) Year-on-year food inflation; (b) correlation between monthly CPI inflation and the monthly mean of the daily DPI shifted forward by k days (95% CI); (c) expanding-window nowcasts of food CPI month-on-month inflation.')
    save_ck({'status': 'ok', 'results': res.to_dict('records'), 'lead_curves': lead_curves, 'nowcast': now}, '11_macro_summary')
    show(res)

run_module('m11', m11_macro, ['11_macro_summary'])

## Module 12 · Forecasting framework (rolling origin)
**Rolling-origin evaluation** – `n_origins` forecast origins spaced `origin_step` days apart at the end of the sample;
models are re-estimated at every origin using only past data; horizons $h\in\{1,7,14,30,60\}$ days.
Forecast errors are **scored only on target days that were actually observed** (never against reconstructed values).

*Aggregate track* – log DPI (All, Food, Non-food): Naive, Seasonal naive (7), Drift, ETS (damped, additive),
ARIMA (order by AIC), LightGBM and XGBoost (global direct models pooled over aggregate + category indices),
LSTM and GRU (global multi-horizon sequence models).

*Product track* – a regime-stratified sample of SKUs: Naive, Seasonal naive, 7-day moving average, global LightGBM,
global LSTM. Because most prices are sticky, we also report **change-aware metrics**: MAE conditional on a price change
and AUPRC for detecting whether a change occurs by the horizon. The question is *which model works best in which
price-behaviour regime*.

Metrics: MAE, RMSE, sMAPE, MASE (scaled by in-sample one-step naive MAE), WAPE; 80% interval coverage and width;
Diebold–Mariano (HLN) tests against the naive benchmark.

In [ ]:
def make_origins(T, obs_mask, n_origins, step, hmax):
    last = T - 1 - hmax
    out = []
    o = last
    while len(out) < n_origins and o > 365:
        oo = o
        while oo > 0 and not (obs_mask[oo] and obs_mask[oo + 1]): oo -= 1
        out.append(oo); o -= step
    return sorted(set(out))

GLOBAL_LAGS = [1, 2, 3, 7, 14, 21, 28, 60]

def global_features(Ymat, t, sid, dates):
    """Features at origin t for every series (rows of Ymat, log×100)."""
    f = {f'chg_{l}': Ymat[:, t] - Ymat[:, t - l] for l in GLOBAL_LAGS}
    d = np.diff(Ymat[:, t - 30:t + 1], axis=1)
    f['vol30'] = d.std(1); f['nz30'] = (np.abs(d) > 1e-6).mean(1)
    f['dow'] = np.full(len(Ymat), dates[t].dayofweek); f['month'] = np.full(len(Ymat), dates[t].month)
    f['sid'] = sid
    return pd.DataFrame(f)

def train_global_tree(Ymat, sid, dates, origin, h, kind='lgbm', stride=2, quantile=None, extra=None):
    rows, ys = [], []
    for t in range(61, origin - h + 1, stride):
        F = global_features(Ymat, t, sid, dates)
        if extra is not None: F = pd.concat([F, extra], axis=1)
        rows.append(F); ys.append(Ymat[:, t + h] - Ymat[:, t])
    X = pd.concat(rows, ignore_index=True); y = np.concatenate(ys)
    X['sid'] = X['sid'].astype('category')
    if kind == 'lgbm':
        import lightgbm as lgb
        params = dict(n_estimators=300, learning_rate=0.03, num_leaves=31, min_child_samples=30, subsample=0.8, subsample_freq=1,
                      colsample_bytree=0.8, random_state=CFG['seed'], verbose=-1)
        if quantile is not None: params.update(objective='quantile', alpha=quantile)
        m = lgb.LGBMRegressor(**params).fit(X, y)
    else:
        import xgboost as xgb
        m = xgb.XGBRegressor(n_estimators=300, learning_rate=0.03, max_depth=4, subsample=0.8, colsample_bytree=0.8,
                             enable_categorical=True, tree_method='hist', random_state=CFG['seed']).fit(X, y)
    return m

def predict_global_tree(m, Ymat, sid, dates, origin, extra=None):
    F = global_features(Ymat, origin, sid, dates)
    if extra is not None: F = pd.concat([F, extra], axis=1)
    F['sid'] = F['sid'].astype('category')
    return m.predict(F)

def train_seq(Ymat, origin, hmax, cell='LSTM', win=60, epochs=60, seed=0):
    import torch, torch.nn as nn
    torch.manual_seed(seed); np.random.seed(seed)
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    D = np.diff(Ymat, axis=1)
    sc = D[:, :origin].std(1, keepdims=True) + 1e-3
    Xs, Ys = [], []
    for t in range(win, origin - hmax + 1):
        Xs.append(D[:, t - win:t] / sc)
        Ys.append((Ymat[:, t + 1:t + hmax + 1] - Ymat[:, [t]]) / sc)
    X = torch.tensor(np.concatenate(Xs), dtype=torch.float32).unsqueeze(-1); Y = torch.tensor(np.concatenate(Ys), dtype=torch.float32)
    class Net(nn.Module):
        def __init__(s):
            super().__init__(); s.rnn = (nn.LSTM if cell == 'LSTM' else nn.GRU)(1, 32, batch_first=True); s.out = nn.Linear(32, hmax)
        def forward(s, x):
            o, _ = s.rnn(x); return s.out(o[:, -1])
    net = Net().to(dev); opt = torch.optim.Adam(net.parameters(), lr=2e-3); lossf = nn.MSELoss()
    n = len(X); bs = 256
    for ep in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, bs):
            b = perm[i:i + bs]; opt.zero_grad()
            l = lossf(net(X[b].to(dev)), Y[b].to(dev)); l.backward(); opt.step()
    net.eval()
    with torch.no_grad():
        xo = torch.tensor(D[:, origin - win:origin] / sc, dtype=torch.float32).unsqueeze(-1).to(dev)
        pred = net(xo).cpu().numpy() * sc
    return Ymat[:, [origin]] + pred                                  # (n_series, hmax) level forecasts

def fit_ets_arima(y, hmax, order=None):
    from statsmodels.tsa.exponential_smoothing.ets import ETSModel
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    out = {}
    try:
        e = ETSModel(pd.Series(y), error='add', trend='add', damped_trend=True).fit(disp=False)
        pr = e.get_prediction(start=len(y), end=len(y) + hmax - 1).summary_frame(alpha=0.2)
        out['ETS'] = (pr['mean'].values, pr['pi_lower'].values, pr['pi_upper'].values)
    except Exception as ex:
        out['ETS'] = None
    if order is None:
        best = None
        for p in range(3):
            for q in range(3):
                try:
                    r = SARIMAX(y, order=(p, 1, q), trend='c').fit(disp=False)
                    if best is None or r.aic < best[0]: best = (r.aic, (p, 1, q))
                except Exception: pass
        order = best[1] if best else (1, 1, 0)
    try:
        r = SARIMAX(y, order=order, trend='c').fit(disp=False)
        fc = r.get_forecast(hmax); ci = fc.conf_int(alpha=0.2)
        out['ARIMA'] = (fc.predicted_mean, ci[:, 0], ci[:, 1])
    except Exception:
        out['ARIMA'] = None
    return out, order

def score_block(df):
    y, p = df['y_true'].values, df['y_pred'].values
    r = {'MAE': m_mae(y, p), 'RMSE': m_rmse(y, p), 'sMAPE (%)': m_smape(y, p), 'WAPE (%)': m_wape(y, p),
         'MASE': float(np.mean(np.abs(y - p) / df['scale'].values))}
    m = df['lo'].notna().values
    r['Coverage 80% (%)'] = 100 * float(np.mean((y[m] >= df['lo'].values[m]) & (y[m] <= df['hi'].values[m]))) if m.any() else np.nan
    r['Width 80%'] = float(np.mean(df['hi'].values[m] - df['lo'].values[m])) if m.any() else np.nan
    return pd.Series(r)

def m12_forecast():
    meta = load_ck('02_meta'); idx = load_ck('07_indices'); regimes = load_ck('09_regimes')
    calendar = load_ck('01_calendar'); obs_mask = calendar['observed'].astype(bool).values
    hz = CFG['horizons']; hmax = max(hz)
    # ---------------- aggregate track ----------------
    B = idx[(idx.dataset == 'B') & (idx.formula == 'Jevons') & (idx.weights == 'sku')]
    piv = B.pivot_table(index='date', columns='scope', values='index'); piv.index = pd.to_datetime(piv.index)
    dates = piv.index; targets = ['All items', 'Food', 'Non-food']
    series = targets + [c for c in piv.columns if c not in targets]
    Ymat = 100 * np.log(piv[series].values.T / 100)                 # log×100, rows = series
    sid = np.arange(len(series)); T = Ymat.shape[1]
    origins = make_origins(T, obs_mask, CFG['n_origins'], CFG['origin_step'], hmax)
    print(f'  aggregate track: {len(origins)} origins from {dates[origins[0]].date()} to {dates[origins[-1]].date()}')
    recs = []; arima_orders = {}
    for oi, o in enumerate(origins):
        fc = {}
        for j, s in enumerate(targets):
            y = Ymat[j, :o + 1]
            fc[('Naive', s)] = (np.repeat(y[-1], hmax), None, None)
            fc[('Seasonal naive', s)] = (np.array([y[-7 + ((h - 1) % 7)] for h in range(1, hmax + 1)]), None, None)
            drift = np.mean(np.diff(y[-365:])); fc[('Drift', s)] = (y[-1] + drift * np.arange(1, hmax + 1), None, None)
            ea, arima_orders[s] = fit_ets_arima(y, hmax, arima_orders.get(s))
            for k, v in ea.items():
                if v is not None: fc[(k, s)] = v
        for kind, name in [('lgbm', 'LightGBM'), ('xgb', 'XGBoost')]:
            for h in hz:
                m = train_global_tree(Ymat, sid, dates, o, h, kind)
                ph = predict_global_tree(m, Ymat, sid, dates, o)
                if kind == 'lgbm':
                    ql = predict_global_tree(train_global_tree(Ymat, sid, dates, o, h, 'lgbm', quantile=0.1), Ymat, sid, dates, o)
                    qh = predict_global_tree(train_global_tree(Ymat, sid, dates, o, h, 'lgbm', quantile=0.9), Ymat, sid, dates, o)
                for j, s in enumerate(targets):
                    fc.setdefault((name, s), (np.full(hmax, np.nan), np.full(hmax, np.nan), np.full(hmax, np.nan)))
                    fc[(name, s)][0][h - 1] = Ymat[j, o] + ph[j]
                    if kind == 'lgbm':
                        fc[(name, s)][1][h - 1] = Ymat[j, o] + min(ql[j], qh[j]); fc[(name, s)][2][h - 1] = Ymat[j, o] + max(ql[j], qh[j])
        for cell in ['LSTM', 'GRU']:
            P = train_seq(Ymat, o, hmax, cell, epochs=CFG['lstm_epochs'], seed=CFG['seed'] + oi)
            for j, s in enumerate(targets): fc[(cell, s)] = (P[j], None, None)
        for (mname, s), (mean_, lo_, hi_) in fc.items():
            j = targets.index(s); scale = np.mean(np.abs(np.diff(Ymat[j, :o + 1])))
            for h in hz:
                t = o + h
                if not obs_mask[t]: continue
                recs.append({'track': 'aggregate', 'series': s, 'origin': str(dates[o].date()), 'h': h, 'model': mname,
                             'y_true': 100 * np.exp(Ymat[j, t] / 100), 'y_pred': 100 * np.exp(mean_[h - 1] / 100),
                             'lo': 100 * np.exp(lo_[h - 1] / 100) if lo_ is not None and np.isfinite(lo_[h - 1]) else np.nan,
                             'hi': 100 * np.exp(hi_[h - 1] / 100) if hi_ is not None and np.isfinite(hi_[h - 1]) else np.nan,
                             'scale': scale * 100 * np.exp(Ymat[j, o] / 100) / 100, 'y_origin': 100 * np.exp(Ymat[j, o] / 100)})
        print(f'    origin {oi + 1}/{len(origins)} done')
    # ---------------- product track ----------------
    Bw = wide_from_ck('06_dataset_B')
    rg = regimes.set_index('product_id')['regime']
    n_per = max(5, CFG['n_sample_products'] // rg.nunique())
    rs = np.random.default_rng(CFG['seed'])
    pick = np.concatenate([rs.choice(np.where(rg.values == r)[0], min(n_per, (rg.values == r).sum()), replace=False) for r in sorted(rg.unique())])
    Yp = 100 * np.log(Bw.values[pick]); psid = np.arange(len(pick))
    pmeta = pd.DataFrame({'product_id': Bw.index[pick], 'regime': rg.values[pick], 'category': meta['category'].values[pick]})
    extra = pd.DataFrame({'regime': pd.Categorical(pmeta['regime']).codes, 'cat': pd.Categorical(pmeta['category']).codes})
    for oi, o in enumerate(origins):
        P_lstm = train_seq(Yp, o, hmax, 'LSTM', epochs=max(10, CFG['lstm_epochs'] // 3), seed=CFG['seed'] + oi)
        preds = {'Naive': np.repeat(Yp[:, [o]], hmax, 1),
                 'Seasonal naive': np.stack([Yp[:, o - 7 + ((h - 1) % 7)] for h in range(1, hmax + 1)], 1),
                 'Moving average (7d)': np.repeat(Yp[:, o - 6:o + 1].mean(1, keepdims=True), hmax, 1), 'LSTM': P_lstm}
        lg = np.full((len(pick), hmax), np.nan)
        for h in hz:
            m = train_global_tree(Yp, psid, dates, o, h, 'lgbm', stride=4, extra=extra)
            lg[:, h - 1] = Yp[:, o] + predict_global_tree(m, Yp, psid, dates, o, extra=extra)
        preds['LightGBM'] = lg
        scale = np.mean(np.abs(np.diff(Yp[:, :o + 1], axis=1)), axis=1) + 1e-3
        for mname, M_ in preds.items():
            for h in hz:
                t = o + h
                if not obs_mask[t]: continue
                for i in range(len(pick)):
                    lvl = np.exp(Yp[i, o] / 100)
                    recs.append({'track': 'product', 'series': pmeta['product_id'].iat[i], 'regime': pmeta['regime'].iat[i], 'origin': str(dates[o].date()),
                                 'h': h, 'model': mname, 'y_true': np.exp(Yp[i, t] / 100), 'y_pred': np.exp(M_[i, h - 1] / 100), 'lo': np.nan, 'hi': np.nan,
                                 'scale': scale[i] / 100 * lvl, 'y_origin': lvl})
        print(f'    product origin {oi + 1}/{len(origins)} done')
    fcs = pd.DataFrame(recs)
    save_ck(fcs, '12_forecasts'); save_ck({'origins': [str(dates[o].date()) for o in origins], 'arima_orders': {k: list(v) for k, v in arima_orders.items()},
                                          'n_products': int(len(pick))}, '12_forecast_meta')
    report_forecasts(fcs)

def report_forecasts(fcs):
    from sklearn.metrics import average_precision_score
    agg = fcs[fcs.track == 'aggregate']
    acc = pd.DataFrame([{'series': k[0], 'model': k[1], 'h': k[2], **score_block(g).to_dict()} for k, g in agg.groupby(['series', 'model', 'h'])])
    for s in ['Food', 'All items']:
        t = acc[acc.series == s].pivot_table(index='model', columns='h', values='MASE')
        t.columns = [f'MASE h={c}' for c in t.columns]
        t2 = acc[acc.series == s].pivot_table(index='model', columns='h', values='sMAPE (%)'); t2.columns = [f'sMAPE h={c}' for c in t2.columns]
        tt = pd.concat([t, t2], axis=1).sort_values(t.columns[-2]).reset_index().rename(columns={'model': 'Model'})
        save_table(tt, f'T25_forecast_accuracy_{s.split()[0].lower()}', f'Rolling-origin forecast accuracy for the {s.lower()} DPI.', float_fmt='{:.3f}',
                   notes='MASE < 1: better than the in-sample one-step naive forecast. Scored only on observed target days.')
    # DM tests vs naive
    dm = []
    for s in ['All items', 'Food', 'Non-food']:
        for h in sorted(agg.h.unique()):
            base = agg[(agg.series == s) & (agg.h == h) & (agg.model == 'Naive')].set_index('origin')
            for mname in agg.model.unique():
                if mname == 'Naive': continue
                cmp = agg[(agg.series == s) & (agg.h == h) & (agg.model == mname)].set_index('origin')
                j = base.index.intersection(cmp.index)
                st, p = dm_test((base.loc[j, 'y_true'] - base.loc[j, 'y_pred']).values, (cmp.loc[j, 'y_true'] - cmp.loc[j, 'y_pred']).values, h=1)
                dm.append({'Series': s, 'h': h, 'Model': mname, 'DM stat': st, 'p-value': p,
                           'RMSE ratio vs naive': m_rmse(cmp.loc[j, 'y_true'].values, cmp.loc[j, 'y_pred'].values) / max(m_rmse(base.loc[j, 'y_true'].values, base.loc[j, 'y_pred'].values), 1e-12)})
    dm = pd.DataFrame(dm); dm['Sig.'] = [stars(p) if np.isfinite(p) else '' for p in dm['p-value']]
    save_table(dm[dm.Series == 'Food'], 'T26_dm_tests_food', 'Diebold–Mariano tests against the naive forecast (food DPI).', float_fmt='{:.3f}',
               notes='Positive DM statistic: model more accurate than naive (squared loss, HLN correction). Few origins → low power; interpret jointly with RMSE ratios.')
    # product track
    pr = fcs[fcs.track == 'product'].copy()
    pr['changed'] = np.abs(np.log(pr['y_true'] / pr['y_origin'])) > 1e-6
    pr['ae'] = np.abs(pr['y_true'] - pr['y_pred']) / pr['y_origin'] * 100
    rows = []
    for (reg, mname, h), g in pr.groupby(['regime', 'model', 'h']):
        score = np.abs(np.log(g['y_pred'] / g['y_origin']))
        ap = average_precision_score(g['changed'], score) if g['changed'].any() and score.std() > 0 else g['changed'].mean()
        rows.append({'Regime': reg, 'Model': mname, 'h': h, 'MAE (% of price)': g['ae'].mean(), 'MAE | change (%)': g.loc[g.changed, 'ae'].mean(),
                     'Share changed (%)': 100 * g['changed'].mean(), 'AUPRC (change)': ap,
                     'MASE': float(np.mean(np.abs(g['y_true'] - g['y_pred']) / g['scale']))})
    pt = pd.DataFrame(rows)
    pt7 = pt[pt.h.isin([7, 30])].pivot_table(index=['Regime', 'Model'], columns='h', values=['MAE (% of price)', 'MAE | change (%)', 'AUPRC (change)'])
    pt7.columns = [f'{a} h={b}' for a, b in pt7.columns]; pt7 = pt7.reset_index()
    save_table(pt7, 'T27_product_forecasts_by_regime', 'Product-level forecast accuracy by price-behaviour regime (h = 7 and 30 days).', float_fmt='{:.3f}',
               notes='MAE in % of the price at origin. "MAE | change": only forecasts where the true price differs from the origin price. AUPRC scores the forecast change magnitude as a detector of a price change (naive = base rate).')
    save_ck(acc, '12_accuracy_aggregate'); save_ck(pt, '12_accuracy_product'); save_ck(dm, '12_dm_tests')
    # figures
    order = acc[acc.series == 'Food'].groupby('model')['MASE'].mean().sort_values().index.tolist()
    cmap = dict(zip(order, OKABE + ['#999999', '#444444', '#bbbbbb']))
    fig, axes = plt.subplots(1, 3, figsize=(W2, 2.6))
    for ax, s, lab in zip(axes[:2], ['Food', 'All items'], 'ab'):
        for mname in [m for m in order if m != 'Naive']:
            t = dm[(dm.Series == s) & (dm.Model == mname)].sort_values('h')
            ax.plot(t['h'], t['RMSE ratio vs naive'], marker='o', ms=2.5, lw=1.0, color=cmap[mname], label=mname,
                    ls='--' if mname in ('Seasonal naive', 'Drift') else '-')
        ax.axhline(1, color='black', lw=0.7, ls=':')
        ax.set_xscale('log'); ax.set_xticks(CFG['horizons']); ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:g}'))
        ax.set_yscale('log'); ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:g}')); ax.yaxis.set_minor_formatter(FuncFormatter(lambda v, _: ''))
        ax.set_xlabel('Horizon (days)'); ax.set_ylabel('RMSE relative to naive'); ax.set_title(f'{s} DPI', fontweight='normal'); panel_label(ax, lab, x=-0.25)
    ax = axes[2]
    hm = pt[pt.h == 30].pivot_table(index='Regime', columns='Model', values='MAE (% of price)')
    im = ax.imshow(hm.values, cmap='viridis_r', aspect='auto')
    ax.set_xticks(range(hm.shape[1])); ax.set_xticklabels([c.replace(' (7d)', '') for c in hm.columns], rotation=45, ha='right', fontsize=6)
    ax.set_yticks(range(hm.shape[0])); ax.set_yticklabels([r.split(':')[0] for r in hm.index], fontsize=6); ax.grid(False)
    vmax = np.nanmax(hm.values)
    for (i, j), v in np.ndenumerate(hm.values): ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=5.5, color='white' if v > 0.55 * vmax else 'black')
    ax.set_title('SKU MAE, % of price (h=30)', fontweight='normal'); panel_label(ax, 'c', x=-0.3)
    h_, l_ = axes[0].get_legend_handles_labels(); fig.legend(h_, l_, loc='lower center', ncol=5, bbox_to_anchor=(0.4, -0.1), fontsize=6.3)
    fig.tight_layout(rect=(0, 0.08, 1, 1))
    savefig(fig, 'F15_forecast_accuracy', 'Forecast accuracy. RMSE relative to the naive forecast by horizon for (a) food and (b) all-items DPI (rolling origins, observed target days only; <1 = better than naive); (c) SKU-level mean absolute error (% of price) by price-behaviour regime and model at a 30-day horizon.')
    show(dm[dm.Series == 'Food'].head(20))

run_module('m12', m12_forecast, ['12_forecasts', '12_forecast_meta', '12_accuracy_aggregate', '12_accuracy_product', '12_dm_tests'])

## Module 13 · Hierarchical reconciliation and uncertainty quantification
**Hierarchy.** Under geometric aggregation the food DPI is *exactly linear in logs*:
$\ln FPI_t=\sum_c w_c\ln I_{ct}$, so $\mathbf y_t=\mathbf S\,\mathbf b_t$ with $\mathbf S=[\mathbf w^\top;\ \mathbf I]$.
Base forecasts (damped ETS, every node, every origin) are reconciled by bottom-up (BU), top-down with historical
betas (TD, $\hat b_c=\hat y_{tot}\,\beta_c$, $\sum_c w_c\beta_c=1$), OLS, WLS (structural) and MinT-shrink
($\tilde{\mathbf y}=\mathbf S(\mathbf S^\top\mathbf W^{-1}\mathbf S)^{-1}\mathbf S^\top\mathbf W^{-1}\hat{\mathbf y}$, $\mathbf W$ = shrunk residual covariance).

**Uncertainty.** (i) Jackknife+-style conformal intervals across rolling origins for every model in Module 12;
(ii) pinball loss of LightGBM quantile forecasts; (iii) **reconstruction uncertainty** – the DPI recomputed on each
multiple-imputation draw; (iv) moving-block bootstrap CIs for annualised inflation.

In [ ]:
def shrink_cov(R):
    """Schäfer–Strimmer shrinkage of the residual covariance toward its diagonal."""
    R = R - R.mean(0); n = len(R)
    S = R.T @ R / n; d = np.sqrt(np.diag(S)).clip(min=1e-12)
    Z = R / d; Rc = Z.T @ Z / n
    W = (Z[:, :, None] * Z[:, None, :]); vr = n / (n - 1) ** 2 * ((W - Rc) ** 2).sum(0)
    np.fill_diagonal(vr, 0); off = Rc.copy(); np.fill_diagonal(off, 0)
    lam = float(np.clip(vr.sum() / max((off ** 2).sum(), 1e-12), 0, 1))
    Rs = (1 - lam) * Rc; np.fill_diagonal(Rs, 1)
    return Rs * np.outer(d, d), lam

def reconcile(yhat, S, W):
    Wi = np.linalg.pinv(W)
    G = np.linalg.pinv(S.T @ Wi @ S) @ S.T @ Wi
    return S @ (G @ yhat)

def block_bootstrap_ann(logidx, dates, B=1000, block=30, seed=0):
    d = np.diff(logidx); n = len(d); rng = np.random.default_rng(seed)
    yrs = (dates[-1] - dates[0]).days / 365.25; out = []
    for _ in range(B):
        st = rng.integers(0, n - block, int(np.ceil(n / block)))
        s = np.concatenate([d[i:i + block] for i in st])[:n]
        out.append(100 * s.sum() / yrs)
    return np.percentile(out, [2.5, 97.5])

def m13_hier_unc():
    from statsmodels.tsa.exponential_smoothing.ets import ETSModel
    meta = load_ck('02_meta'); idx = load_ck('07_indices'); fcs = load_ck('12_forecasts'); fm = load_ck('12_forecast_meta')
    calendar = load_ck('01_calendar'); obs_mask = calendar['observed'].astype(bool).values
    B_ = idx[(idx.dataset == 'B') & (idx.formula == 'Jevons') & (idx.weights == 'sku')]
    piv = B_.pivot_table(index='date', columns='scope', values='index'); piv.index = pd.to_datetime(piv.index); dates = piv.index
    food_cats = list(meta.loc[meta.is_food == 1, 'category'].value_counts().index)
    w = meta.loc[meta.is_food == 1, 'category'].value_counts(normalize=True)[food_cats].values
    nodes = ['Food'] + food_cats
    Y = 100 * np.log(piv[nodes].values.T / 100)
    S = np.vstack([w, np.eye(len(food_cats))])
    origins = [dates.get_loc(pd.Timestamp(o)) for o in fm['origins']]; hz = CFG['horizons']; hmax = max(hz)
    rec = []
    for o in origins:
        base = np.zeros((len(nodes), hmax)); resid = []
        for j in range(len(nodes)):
            try:
                e = ETSModel(pd.Series(Y[j, :o + 1]), error='add', trend='add', damped_trend=True).fit(disp=False)
                base[j] = e.forecast(hmax); resid.append(e.resid[-365:])
            except Exception:
                base[j] = Y[j, o]; resid.append(np.diff(Y[j, o - 365:o + 1]))
        Rm = np.column_stack(resid)
        Wm, lam = shrink_cov(Rm)
        Wdiag = np.diag(np.diag(Wm)); Wstruct = np.diag(S.sum(1))
        for h in hz:
            if not obs_mask[o + h]: continue
            yh = base[:, h - 1]
            hchg = np.array([Y[j, h:o + 1] - Y[j, :o + 1 - h] for j in range(len(nodes))])[:, -365:]
            beta = np.array([np.cov(hchg[j + 1], hchg[0])[0, 1] / max(np.var(hchg[0]), 1e-12) for j in range(len(food_cats))])
            beta = beta / max(w @ beta, 1e-12)
            td_b = Y[1:, o] + (yh[0] - Y[0, o]) * beta
            meth = {'Base (incoherent)': yh, 'Bottom-up': S @ yh[1:], 'Top-down (β)': S @ td_b,
                    'OLS': reconcile(yh, S, np.eye(len(nodes))), 'WLS (structural)': reconcile(yh, S, Wstruct),
                    'MinT (shrink)': reconcile(yh, S, Wm)}
            for mname, v in meth.items():
                for j, nd in enumerate(nodes):
                    rec.append({'origin': str(dates[o].date()), 'h': h, 'method': mname, 'node': nd, 'level': 'Total' if j == 0 else 'Category',
                                'err': 100 * np.exp(Y[j, o + h] / 100) - 100 * np.exp(v[j] / 100)})
    R = pd.DataFrame(rec)
    ht = R.groupby(['level', 'method', 'h'])['err'].apply(lambda e: np.sqrt(np.mean(e ** 2))).unstack('h')
    ht.columns = [f'RMSE h={c}' for c in ht.columns]; ht = ht.reset_index().rename(columns={'level': 'Level', 'method': 'Method'})
    save_table(ht, 'T28_hierarchical_reconciliation', 'Hierarchical forecast reconciliation of the food DPI (food total and its categories; damped-ETS base forecasts).',
               float_fmt='{:.3f}', notes='RMSE in index points over rolling origins (observed target days only); category level = pooled over categories. MinT uses a Schäfer–Strimmer shrunk covariance of in-sample residuals.')

    # ---------- conformal intervals (leave-one-origin-out) ----------
    agg = fcs[fcs.track == 'aggregate'].copy()
    rows = []
    for (s, mname, h), g in agg.groupby(['series', 'model', 'h']):
        e = (g['y_true'] - g['y_pred']).abs().values
        if len(e) < 3: continue
        for alpha, lab in [(0.2, '80'), (0.05, '95')]:
            cov, wid = [], []
            for k in range(len(e)):
                q = np.quantile(np.delete(e, k), min(1, np.ceil((1 - alpha) * len(e)) / (len(e) - 1)))
                cov.append(e[k] <= q); wid.append(2 * q)
            rows.append({'Series': s, 'Model': mname, 'h': h, 'Nominal (%)': int(lab), 'Coverage (%)': 100 * np.mean(cov), 'Mean width': np.mean(wid)})
    cf = pd.DataFrame(rows, columns=['Series', 'Model', 'h', 'Nominal (%)', 'Coverage (%)', 'Mean width'])
    pin = []
    lg = agg[(agg.model == 'LightGBM') & agg.lo.notna()]
    for (s, h), g in lg.groupby(['series', 'h']):
        y = g['y_true'].values
        pl = lambda q, tau: np.mean(np.maximum(tau * (y - q), (tau - 1) * (y - q)))
        pin.append({'Series': s, 'h': h, 'Pinball τ=0.1': pl(g['lo'].values, 0.1), 'Pinball τ=0.9': pl(g['hi'].values, 0.9),
                    'Quantile coverage 80% (%)': 100 * np.mean((y >= g['lo']) & (y <= g['hi']))})
    cft = cf[(cf.Series == 'Food')].pivot_table(index=['Model', 'Nominal (%)'], columns='h', values='Coverage (%)')
    cft.columns = [f'Cov. h={c}' for c in cft.columns]; cft = cft.reset_index()
    save_table(cft, 'T29_conformal_coverage', 'Empirical coverage of leave-one-origin-out conformal prediction intervals (food DPI).', float_fmt='{:.1f}',
               notes='Few origins make coverage estimates coarse; nominal 80% and 95%.')
    if pin: save_table(pd.DataFrame(pin), 'T30_quantile_scores', 'Pinball loss and coverage of LightGBM quantile forecasts.', float_fmt='{:.4f}')

    # ---------- reconstruction uncertainty (MI) ----------
    mi = load_ck('04_mi_draws'); Bw = wide_from_ck('06_dataset_B')
    mi_food, mi_all = [], []
    for dr in mi['draws_log']:
        Wd = pd.DataFrame(np.exp(dr), index=Bw.index, columns=Bw.columns)
        res = build_indices(Wd, meta, 0, formulas=('Jevons',), schemes=('sku',))
        mi_food.append(100 * res[('Food', 'Jevons', 'sku')]); mi_all.append(100 * res[('All items', 'Jevons', 'sku')])
    mi_food, mi_all = np.array(mi_food), np.array(mi_all)
    ann = lambda v: 100 * np.log(v[-1] / v[0]) / ((dates[-1] - dates[0]).days / 365.25)
    bt = []
    for sc, arr in [('Food', mi_food), ('All items', mi_all)]:
        s = piv[sc].values; ci = block_bootstrap_ann(np.log(s), dates)
        yoy_mi = 100 * np.log(arr[:, 365:] / arr[:, :-365])
        bt.append({'Scope': sc, 'Annualised (B) (%)': ann(s), 'Block-bootstrap 95% CI low': ci[0], 'Block-bootstrap 95% CI high': ci[1],
                   'MI range of annualised (pp)': float(np.ptp([ann(a) for a in arr])), 'MI mean |YoY spread| (pp)': float(np.mean(yoy_mi.max(0) - yoy_mi.min(0))),
                   'MI max |YoY spread| (pp)': float(np.max(yoy_mi.max(0) - yoy_mi.min(0)))})
    save_table(pd.DataFrame(bt), 'T31_inflation_uncertainty', 'Sampling and reconstruction uncertainty of digital inflation.', float_fmt='{:.3f}',
               notes=f'Moving-block bootstrap (block 30 days, 1,000 replications) of daily log changes. MI: {len(mi_food)} draws of the stochastic sticky bridge; spread = max−min across draws.')

    # ---------- figure ----------
    fig, axes = plt.subplots(1, 3, figsize=(W2, 2.6), gridspec_kw={'width_ratios': [1.5, 1, 1]})
    ax = axes[0]; shade_gaps(ax)
    ax.fill_between(dates, mi_food.min(0), mi_food.max(0), color=OKABE[1], alpha=0.35, lw=0, label='MI range (sticky draws)')
    ax.plot(dates, piv['Food'], color=OKABE[1], lw=1.0, label='Food DPI (B)')
    ax.set_ylabel('Food index'); date_axis(ax, interval=4); ax.legend(loc='lower right', fontsize=6.3); panel_label(ax, 'a', x=-0.13)
    ax = axes[1]
    w30 = cf[(cf.Series == 'Food') & (cf['Nominal (%)'] == 80)]
    w30 = w30[w30.h == w30.h.max()].groupby('Model')['Mean width'].mean().sort_values() if len(w30) else pd.Series(dtype=float)
    if len(w30): ax.barh(w30.index, w30.values, color=OKABE[0], height=0.6)
    else: ax.text(0.5, 0.5, 'too few origins\nwith observed targets', transform=ax.transAxes, ha='center', fontsize=7, color=GREY)
    ax.set_xlabel('Mean 80% interval width (index pts)'); ax.set_title(f'Conformal intervals, h={int(cf.h.max()) if len(cf) else 0}', fontweight='normal'); ax.tick_params(axis='y', labelsize=6)
    ax.grid(axis='y', visible=False); panel_label(ax, 'b', x=-0.55)
    ax = axes[2]
    tt = R[R.level == 'Category'].groupby(['method', 'h'])['err'].apply(lambda e: np.sqrt(np.mean(e ** 2))).unstack('method')
    for k, mname in enumerate(tt.columns):
        ax.plot(tt.index, tt[mname], marker='o', ms=2.5, lw=1.0, color=OKABE[k % 8], label=mname)
    ax.set_xscale('log'); ax.set_xticks(hz); ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:g}'))
    ax.set_xlabel('Horizon (days)'); ax.set_ylabel('Category RMSE (index pts)'); ax.legend(fontsize=5.5); panel_label(ax, 'c', x=-0.3)
    fig.tight_layout()
    savefig(fig, 'F16_uncertainty_hierarchy', 'Uncertainty and coherence. (a) Food DPI with the range across multiple-imputation draws of the reconstruction; (b) sharpness (mean width) of 80% leave-one-origin-out conformal intervals at the longest horizon (coverage in Table T29); (c) category-level RMSE of reconciled forecasts by horizon.')
    save_ck(R, '13_reconciliation'); save_ck(cf, '13_conformal'); save_ck({'mi_food': mi_food.tolist(), 'bootstrap': bt}, '13_uncertainty')
    show(ht); show(pd.DataFrame(bt))

run_module('m13', m13_hier_unc, ['13_reconciliation', '13_conformal', '13_uncertainty'])

## Module 14 · Robustness scenarios and ablation
The headline findings are recomputed under every scenario:
1. Observed data only (A) · 2. Preferred reconstruction (B) · 3. Alternative reconstruction (C) · 4. Long gaps excluded (D)
· 5. Multiple-imputation draws · 6. Alternative upper-level weights · 7. Alternative index formulas · 8. Leave-one-category-out.

Headline quantities: annualised food / all-items inflation, latest food YoY, frequency of price change and implied
duration, synchronisation (FK), and the 30-day forecast skill of damped ETS relative to naive.
**Ablation** of the learned bridge: retrain without each feature group and measure the deterioration in masked-gap
validation (why the model works, not just whether).

In [ ]:
def headline(W, meta, label, dates_obs_mask):
    res = build_indices(W, meta, 0, formulas=('Jevons',), schemes=('sku',))
    dates = W.columns; yrs = (dates[-1] - dates[0]).days / 365.25
    out = {'Scenario': label}
    for sc in ['Food', 'All items']:
        s = pd.Series(res[(sc, 'Jevons', 'sku')], index=dates)
        out[f'{sc} annualised (%)'] = 100 * np.log(s.iloc[-1] / s.iloc[0]) / yrs
    s = pd.Series(res[('Food', 'Jevons', 'sku')], index=dates)
    last = s.dropna().index[-1]; prev = s.reindex([last - pd.Timedelta(days=365)]).iloc[0]
    out['Food YoY latest (%)'] = 100 * np.log(s[last] / prev) if np.isfinite(prev) else np.nan
    yoy = 100 * np.log(s / s.shift(365))
    out['Mean food YoY (%)'] = float(yoy.mean())
    D = transitions(W); V = ~np.isnan(D); C = (np.abs(np.nan_to_num(D)) > EPS) & V
    f = C.sum() / V.sum(); out['Freq. of change (%/day)'] = 100 * f; out['Implied duration (days)'] = -1 / np.log(1 - f)
    S = C.sum(0) / V.sum(0).clip(min=1); S = S[V.sum(0) > 0]
    out['FK sync.'] = float(np.sqrt(np.var(S) / (f * (1 - f))))
    return out, s

def m14_robust():
    from statsmodels.tsa.exponential_smoothing.ets import ETSModel
    meta = load_ck('02_meta'); idx = load_ck('07_indices'); calendar = load_ck('01_calendar')
    obs_mask = calendar['observed'].astype(bool).values; fm = load_ck('12_forecast_meta'); dsm = load_ck('06_datasets_meta')
    rows, paths = [], {}
    for k, lab in [('A', 'A: observed only'), ('B', f'B: {dsm["preferred"]}'), ('C', f'C: {dsm["alternative"]}'), ('D', 'D: long gaps excluded')]:
        W = wide_from_ck(f'06_dataset_{k}'); r, s = headline(W, meta, lab, obs_mask); rows.append(r); paths[lab] = s
    mi = load_ck('04_mi_draws'); Bw = wide_from_ck('06_dataset_B')
    for m_, dr in enumerate(mi['draws_log']):
        r, s = headline(pd.DataFrame(np.exp(dr), index=Bw.index, columns=Bw.columns), meta, f'MI draw {m_ + 1}', obs_mask); rows.append(r); paths[f'MI draw {m_ + 1}'] = s
    for mname in ['Linear', 'LOCF', 'Kalman (local level)']:
        if mname in (dsm['preferred'], dsm['alternative']): continue
        W = wide_from_ck(imp_name(mname)); r, s = headline(W, meta, f'Imputation: {mname}', obs_mask); rows.append(r); paths[f'Imputation: {mname}'] = s
    # weights & formulas (from module 7)
    def get(ds='B', scope='Food', f='Jevons', w='sku'):
        s = idx[(idx.dataset == ds) & (idx.scope == scope) & (idx.formula == f) & (idx.weights == w)].set_index('date')['index']
        s.index = pd.to_datetime(s.index); return s
    dates = get().index; yrs = (dates[-1] - dates[0]).days / 365.25
    for w in ['equal', 'hies']:
        s = get(w=w)
        if len(s): rows.append({'Scenario': f'Weights: {w}', 'Food annualised (%)': 100 * np.log(s.iloc[-1] / s.iloc[0]) / yrs, 'Mean food YoY (%)': float((100 * np.log(s / s.shift(365))).mean()),
                                'All items annualised (%)': 100 * np.log(get(scope='All items', w=w).iloc[-1] / 100) / yrs})
    for f in ['Laspeyres', 'Paasche', 'Fisher', 'Dutot']:
        s = get(f=f); rows.append({'Scenario': f'Formula: {f}', 'Food annualised (%)': 100 * np.log(s.iloc[-1] / s.iloc[0]) / yrs, 'Mean food YoY (%)': float((100 * np.log(s / s.shift(365))).mean()),
                                   'All items annualised (%)': 100 * np.log(get(scope='All items', f=f).iloc[-1] / 100) / yrs})
    # forecast skill per scenario (ETS vs naive, h=30, Food) for gap-free scenarios
    origins = [dates.get_loc(pd.Timestamp(o)) for o in fm['origins']]
    for r in rows:
        s = paths.get(r['Scenario'])
        if s is None or s.isna().any(): continue
        y = 100 * np.log(s.values / 100); e_n, e_e = [], []
        for o in origins:
            if o + 30 >= len(y) or not obs_mask[o + 30]: continue
            try:
                fc = ETSModel(pd.Series(y[:o + 1]), error='add', trend='add', damped_trend=True).fit(disp=False).forecast(30)[-1]
            except Exception:
                continue
            e_n.append(y[o + 30] - y[o]); e_e.append(y[o + 30] - fc)
        if e_n: r['ETS/naive RMSE (h=30)'] = m_rmse(np.array(e_e), 0) / max(m_rmse(np.array(e_n), 0), 1e-12)
    bkey = [k for k in paths if k.startswith('B:')][0]; sb = paths[bkey]
    mb = 100 * np.log(sb.resample('MS').mean()).diff()
    for r in rows:
        sp = paths.get(r['Scenario'])
        if sp is None: continue
        r['Path RMSE vs B (pts)'] = float(np.sqrt(np.nanmean((100 * (sp - sb)) ** 2)))
        ms_ = 100 * np.log(sp.resample('MS').mean()).diff()
        j = pd.concat([mb, ms_], axis=1).dropna()
        r['Corr. monthly MoM with B'] = float(np.corrcoef(j.iloc[:, 0], j.iloc[:, 1])[0, 1]) if len(j) > 3 else np.nan
    rob = pd.DataFrame(rows)
    save_table(rob, 'T32_robustness_scenarios', 'Robustness of headline findings across data, reconstruction, weighting and formula scenarios.', float_fmt='{:.3f}',
               notes='Micro statistics use consecutive non-missing days of each scenario (reconstructed days included for B/C/MI/imputation scenarios), which shows how reconstruction affects measured stickiness. ETS/naive < 1: ETS beats naive. Fixed-base annualised rates coincide across reconstructions because sample end points are observed; mean YoY, path RMSE and monthly correlation capture differences inside gaps.')
    # leave-one-category-out (food)
    food_cats = list(meta.loc[meta.is_food == 1, 'category'].value_counts().index)
    W = wide_from_ck('06_dataset_B'); loo = []
    base = rob.loc[rob.Scenario.str.startswith('B:'), 'Food annualised (%)'].iloc[0]
    for c in food_cats:
        mm = meta.copy(); mm.loc[mm.category == c, 'is_food'] = 0
        res = build_indices(W, mm, 0, formulas=('Jevons',), schemes=('sku',))
        s = res[('Food', 'Jevons', 'sku')]
        loo.append({'Excluded category': c, 'Food annualised (%)': 100 * np.log(s[-1] / s[0]) / yrs, 'Δ vs full (pp)': 100 * np.log(s[-1] / s[0]) / yrs - base})
    loo = pd.DataFrame(loo).sort_values('Δ vs full (pp)')
    save_table(loo, 'T33_leave_one_category_out', 'Leave-one-category-out sensitivity of annualised food DPI inflation.', float_fmt='{:.3f}')

    # ablation of the learned bridge
    cal = wide_from_ck('01_cal_wide'); Y = np.log(cal.values); cat_codes = pd.Categorical(meta['category']).codes
    im = load_ck('04_imputation_meta'); train_rows, test_rows = np.array(im['train_rows']), np.array(im['test_rows'])
    observed = obs_mask; cdates = cal.columns
    rng = np.random.default_rng(CFG['seed'] + 7)
    wins = []
    for L in [14, 45]:
        cand = candidate_starts(observed, L)
        if len(cand): wins += [(int(s), L) for s in rng.choice(cand, min(3, len(cand)), replace=False)]
    ab = []
    variants = {'Full model': BRIDGE_FEATURES}
    for gname, feats in FEATURE_GROUPS.items():
        variants[f'− {gname.replace("_", " ")}'] = [f for f in BRIDGE_FEATURES if f not in feats]
    for vname, feats in variants.items():
        mdl = train_bridge(Y, observed, cdates, cat_codes, train_rows, max(10, CFG['bridge_train_windows'] // 3), CFG['seed'],
                           CFG['bridge_max_rows'] // 3, features=feats, verbose=False)
        sc_ = []
        for s, L in wins:
            Ym = Y.copy(); Ym[:, s:s + L] = np.nan
            ctx = make_ctx(Ym, cdates, cat_codes, seed=s, bridge_model=mdl)
            Z = fill_bridge(Ym, [(s, s + L - 1)], ctx, model=mdl, features=feats)
            sc_.append(score_window(Y, Z, s, s + L - 1, test_rows, cat_codes)[0])
        sc_ = pd.DataFrame(sc_)
        ab.append({'Variant': vname, 'Features': len(feats), 'WAPE (%)': sc_['WAPE (%)'].mean(), 'MAE log ×100 | changed': sc_['MAE log ×100 | changed'].mean(),
                   'Index RMSE (pts)': sc_['Index RMSE (pts)'].mean(), 'Change-freq. error (pp)': sc_['Change-freq. error (pp)'].mean()})
        print(f'  ablation {vname}: done')
    ab = pd.DataFrame(ab)
    full = ab.iloc[0]
    for c in ['WAPE (%)', 'Index RMSE (pts)']:
        ab[f'Δ {c.split(" (")[0]} (%)'] = 100 * (ab[c] / full[c] - 1)
    save_table(ab, 'T34_bridge_ablation', 'Ablation of the learned bridge: contribution of feature groups (masked 14- and 45-day gaps).', float_fmt='{:.3f}',
               notes='Each variant is retrained without the feature group and evaluated on the same masked windows and held-out SKUs. Positive Δ = deterioration relative to the full model.')

    # figure
    fig, axes = plt.subplots(1, 3, figsize=(W2, 3.0), gridspec_kw={'width_ratios': [1.2, 1, 1]})
    ax = axes[0]
    rr = rob.dropna(subset=['Mean food YoY (%)']).iloc[::-1]
    cols = [OKABE[1] if s.startswith('B:') else OKABE[0] if s[:2] in ('A:', 'C:', 'D:') else GREY for s in rr['Scenario']]
    ax.scatter(rr['Mean food YoY (%)'], range(len(rr)), color=cols, s=14, zorder=3)
    ax.axvline(rr.loc[rr.Scenario.str.startswith('B:'), 'Mean food YoY (%)'].iloc[0], color=OKABE[1], lw=0.6, ls='--')
    ax.set_yticks(range(len(rr))); ax.set_yticklabels(rr['Scenario'], fontsize=5.8); ax.set_xlabel('Mean food YoY inflation (%)')
    ax.grid(axis='y', visible=False); panel_label(ax, 'a', x=-0.75)
    ax = axes[1]
    ax.barh(loo['Excluded category'].str.split(' &').str[0], loo['Δ vs full (pp)'], color=[OKABE[0] if v < 0 else OKABE[1] for v in loo['Δ vs full (pp)']], height=0.65)
    ax.axvline(0, color=GREY, lw=0.6); ax.set_xlabel('Δ annualised food inflation (pp)'); ax.tick_params(axis='y', labelsize=6)
    ax.set_title('Leave one category out', fontweight='normal'); ax.grid(axis='y', visible=False); panel_label(ax, 'b', x=-0.5)
    ax = axes[2]
    a2 = ab.iloc[1:]
    ax.barh(a2['Variant'], a2['Δ WAPE (%)'], color=OKABE[2], height=0.35, label='WAPE', align='edge')
    ax.barh(a2['Variant'], a2['Δ Index RMSE (%)'], color=OKABE[4], height=-0.35, label='Index RMSE', align='edge')
    ax.axvline(0, color=GREY, lw=0.6); ax.set_xlabel('Deterioration vs full (%)'); ax.tick_params(axis='y', labelsize=6)
    ax.set_title('Bridge ablation', fontweight='normal'); ax.legend(fontsize=6, loc='lower right'); ax.grid(axis='y', visible=False); panel_label(ax, 'c', x=-0.75)
    fig.tight_layout()
    savefig(fig, 'F17_robustness', 'Robustness. (a) Mean year-on-year food DPI inflation across data, reconstruction, weighting and formula scenarios (orange/dashed: preferred); (b) change in annualised food inflation when one category is excluded; (c) deterioration of the learned bridge when a feature group is removed.')
    save_ck(rob, '14_robustness'); save_ck(ab, '14_ablation'); save_ck(loo, '14_loo')
    show(rob); show(ab)

run_module('m14', m14_robust, ['14_robustness', '14_ablation', '14_loo'])

## Module 15 · Export of journal-ready tables and figures
* `tables/*.tex` – booktabs + threeparttable LaTeX (requires `\usepackage{booktabs,threeparttable,array,rotating}`), plus `.csv`
* `figures/*.pdf` (vector, embedded TrueType fonts) and `figures/*.png` (600 dpi), Okabe–Ito colour-blind-safe palette
* `tables/_all_tables.tex` and `figures/_all_figures.tex` – drop-in `\input{}` files with captions and labels
* `key_results.json` – headline numbers for the abstract; `DPI_outputs.zip` – everything in one archive

In [ ]:
def m15_export():
    import zipfile
    tabs = sorted(DIRS['tables'].glob('T*.tex'))
    (DIRS['tables'] / '_all_tables.tex').write_text('% Requires: \\usepackage{booktabs,threeparttable,array,rotating}\n' + '\n'.join(f'\\input{{tables/{p.stem}}}\n\\clearpage' for p in tabs))
    caps = {}
    cp = DIRS['figures'] / '_captions.json'
    if cp.exists():
        for line in cp.read_text().splitlines():
            try: d = json.loads(line); caps[d['name']] = d['caption']
            except Exception: pass
    figs = sorted(p.stem for p in DIRS['figures'].glob('F*.pdf'))
    blocks = []
    for f in figs:
        cap = tex_escape(caps.get(f, f))
        blocks.append(f'\\begin{{figure}}[!htbp]\n\\centering\n\\includegraphics[width=\\textwidth]{{figures/{f}.pdf}}\n\\caption{{{cap}}}\n\\label{{fig:{f}}}\n\\end{{figure}}\n')
    (DIRS['figures'] / '_all_figures.tex').write_text('\n'.join(blocks))
    key = {}
    for name in ['01_xcheck', '03_missingness_summary', '05_selection', '08_dynamics_summary', '09_cluster_summary', '11_macro_summary']:
        if ck_exists(name):
            v = load_ck(name)
            key[name] = {k: x for k, x in v.items() if not isinstance(x, (list, dict)) or k in ('composite_rank',)} if isinstance(v, dict) else None
    for name in ['14_robustness']:
        if ck_exists(name): key[name] = load_ck(name).to_dict('records')
    json.dump(key, open(ROOT / 'key_results.json', 'w'), indent=1, default=_json_default)
    # ---- auto-generated key findings (numbers for the abstract / results section) ----
    L = ['# Key findings (auto-generated)', '']
    try:
        x = load_ck('01_xcheck'); ms = load_ck('03_missingness_summary'); sel = load_ck('05_selection')
        L.append(f"- Panel: {len(load_ck('02_meta')):,} SKUs; {len(x.get('stale_days', []))} stale snapshot days removed; "
                 f"{ms['n_gap_blocks']} gap blocks, {ms['gap_days']} missing days, longest {ms['longest_gap']} days (missingness is system-wide: {ms['system_wide']}).")
        L.append(f"- Reconstruction: preferred = {sel['preferred']}; alternative = {sel['alternative']} (masked-gap validation, Module 5).")
        rob = load_ck('14_robustness').set_index('Scenario')
        b = [i for i in rob.index if i.startswith('B:')][0]
        sc = rob['Mean food YoY (%)'].iloc[:4 + CFG['mi_draws']]
        L.append(f"- Food DPI (B): annualised {rob.loc[b, 'Food annualised (%)']:.2f}%; mean YoY {rob.loc[b, 'Mean food YoY (%)']:.2f}% "
                 f"(range across A–D/MI scenarios: {sc.min():.2f}–{sc.max():.2f}%).")
        d = load_ck('08_dynamics_summary'); pdn = load_ck('08_product_dynamics')
        L.append(f"- Stickiness: {100*d['fbar']:.2f}% of SKUs change price per day; median implied duration {pdn['duration'].median():.0f} days; "
                 f"FK synchronisation {d['FK']:.2f}; {100*d['share_changes_top5pct_days']:.1f}% of changes occur on the 5% most synchronised days.")
        c = load_ck('09_cluster_summary'); cv = c["Cramér's V (regime × category)"]
        L.append(f"- Regimes: k = {c['k']} (+ no-change regime); bootstrap ARI {c['Bootstrap ARI mean']:.2f}; Cramér's V with category {cv:.2f}.")
        dm = load_ck('12_dm_tests'); f30 = dm[(dm.Series == 'Food') & (dm.h == 30)].sort_values('RMSE ratio vs naive')
        if len(f30): L.append(f"- Forecasting (food DPI, h=30): best = {f30.iloc[0]['Model']} (RMSE ratio vs naive {f30.iloc[0]['RMSE ratio vs naive']:.2f}).")
        m11 = load_ck('11_macro_summary'); L.append(f"- Official CPI linkage: {m11.get('status')}.")
    except Exception as e:
        L.append(f'(some findings unavailable: {e})')
    (ROOT / 'key_findings.md').write_text('\n'.join(L)); print('\n'.join(L))
    zp = ROOT / 'DPI_outputs.zip'
    with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
        for d in ['tables', 'figures']:
            for p in DIRS[d].glob('*'): z.write(p, f'{d}/{p.name}')
        z.write(ROOT / 'key_results.json', 'key_results.json'); z.write(ROOT / 'config.json', 'config.json')
        z.write(ROOT / 'key_findings.md', 'key_findings.md')
    manifest = pd.DataFrame({'File': [p.name for p in tabs] + [f + '.pdf' for f in figs],
                             'Type': ['Table'] * len(tabs) + ['Figure'] * len(figs)})
    save_ck({'n_tables': len(tabs), 'n_figures': len(figs), 'zip': str(zp)}, '15_export')
    print(f'Exported {len(tabs)} tables and {len(figs)} figures → {zp}')
    show(manifest, 80)
    if IN_COLAB:
        from google.colab import files
        try: files.download(str(zp))
        except Exception: pass

FORCE_RERUN.add('m15')   # export is cheap – always refresh
run_module('m15', m15_export, ['15_export'])

### Browse figures saved on Drive (works even when modules were skipped)

In [ ]:
def show_figures(pattern='F*.png', width=900):
    from IPython.display import Image, display, Markdown
    caps = {}
    cp = DIRS['figures'] / '_captions.json'
    if cp.exists():
        for line in cp.read_text().splitlines():
            try: d = json.loads(line); caps[d['name']] = d['caption']
            except Exception: pass
    for p in sorted(DIRS['figures'].glob(pattern)):
        display(Markdown(f'**{p.stem}** – {caps.get(p.stem, "")}'))
        display(Image(filename=str(p), width=width))

if IN_COLAB:
    show_figures()